# Thesis: Cross-Cultural NLP Analysis of Ghost Story Gender Framing
## PTT (zh) · Theqoo (ko) · Reddit (en)
### Pipeline v2 — Raw Articles → Character Mention Extraction → Classification → Analysis

**Data sources (raw):**
- `new_ptt_experience_articles_all.csv`  — 15,759 PTT articles (Traditional Chinese)
- `theqoo_experience_articles_v2.csv`    —    967 Theqoo articles (Korean)
- `reddit_merged_stories.csv`            —  5,667 Reddit articles (English)

**Pipeline independence:** No pre-labelled entity CSV is used as analysis input.
The entity CSV (`entities_all_*.csv`) is loaded **only** as external gold-standard
for validation (Part 11), never as a pipeline input.

## Part 0 · Setup & Dependencies

In [ ]:
import os
import random
import warnings
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib
import matplotlib.pyplot as plt

from scipy import stats
from scipy.stats import chi2_contingency
from sklearn.metrics import (classification_report, confusion_matrix,
                              ConfusionMatrixDisplay, f1_score,
                              precision_score, recall_score)

warnings.filterwarnings('ignore')
plt.rcParams['figure.dpi'] = 150

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# ── CJK fonts ─────────────────────────────────────────────────────────────────
from matplotlib import font_manager as _fm
_FONT_CANDIDATES = [
    ('Noto Sans TC', os.path.expanduser('~/Downloads/Noto_Sans_TC/NotoSansTC-Regular.otf')),
    ('AppleGothic', None),
    ('Hiragino Sans', None),
    ('Hiragino Sans GB', None),
    ('Arial Unicode MS', None),
]
_loaded = []
for _fname, _fpath in _FONT_CANDIDATES:
    if _fpath and os.path.exists(_fpath):
        _fm.fontManager.addfont(_fpath)
        _loaded.append(_fname)
    elif any(f.name == _fname for f in _fm.fontManager.ttflist):
        _loaded.append(_fname)
if _loaded:
    matplotlib.rcParams['font.family'] = 'sans-serif'
    matplotlib.rcParams['font.sans-serif'] = _loaded + ['DejaVu Sans']
    matplotlib.rcParams['axes.unicode_minus'] = False
    print(f"CJK fonts loaded: {', '.join(_loaded)}")
else:
    matplotlib.rcParams['axes.unicode_minus'] = False
    print("CJK font not found — CJK characters may not render.")

print("All imports successful.")

In [ ]:
# ── Part 0.2: Data paths ─────────────────────────────────────────────────────
RAW_DIR = Path(os.path.expanduser(
    '~/Downloads/ghost-story-gender-age-analysis/data/raw'
))
ENTITY_DIR = Path(os.path.expanduser(
    '~/Downloads/ghost-story-gender-age-analysis/statistical_reports_chinese/Thesis_CT_Kelly'
))

# Primary inputs: RAW article files only
RAW_FILES = {
    'ptt':    RAW_DIR / 'new_ptt_experience_articles_all.csv',
    'theqoo': RAW_DIR / 'theqoo_experience_articles_v2.csv',
    'reddit': RAW_DIR / 'reddit_merged_stories.csv',
}

# Used ONLY for validation (Part 11), never as analysis input
ENTITY_FILES = {
    'ptt':    ENTITY_DIR / 'entities_all_ptt.csv',
    'theqoo': ENTITY_DIR / 'entities_all_theqoo.csv',
    'reddit': ENTITY_DIR / 'entities_all_reddit.csv',
}

# REPORT_DIR is an alias for ENTITY_DIR — used in Part 11
REPORT_DIR = ENTITY_DIR

# Annotation file for state validation
ANNOTATION_FILE = RAW_DIR / 'ghost_stories_analysis_annotation - sample_ghost_stories_analysis_annotate - 工作表1.csv'

OUT_DIR = RAW_DIR.parent.parent / 'results'
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("Data paths configured.")
for name, p in RAW_FILES.items():
    exists = "✓" if p.exists() else "✗ MISSING"
    print(f"  {name:8s}: {exists}  {p.name}")

## Part 1 · Load Raw Articles

In [ ]:
# ── Part 1.1: Load raw article CSVs ─────────────────────────────────────────
raw_dfs = {}

for name, path in RAW_FILES.items():
    df = pd.read_csv(path)
    df['platform'] = name
    # Assign language
    if name == 'ptt':
        df['lang'] = 'zh'
    elif name == 'theqoo':
        df['lang'] = 'ko'
    else:  # reddit
        # Reddit file may already have a lang column
        if 'lang' not in df.columns:
            df['lang'] = 'en'
    raw_dfs[name] = df
    print(f"{name.upper():8s}: {len(df):,} articles | columns: {df.columns.tolist()}")

# Standardise: every df has 'content' and 'lang' columns
for name, df in raw_dfs.items():
    if 'content' not in df.columns:
        # try common alternatives
        for alt in ['body', 'text', 'story']:
            if alt in df.columns:
                df.rename(columns={alt: 'content'}, inplace=True)
                break
    df['article_id'] = df.index
    raw_dfs[name] = df

print("\nRaw articles loaded successfully.")

In [ ]:
# ── Part 1.2: Data overview ──────────────────────────────────────────────────
for name, df in raw_dfs.items():
    n_content = df['content'].dropna().shape[0]
    avg_len   = df['content'].dropna().str.len().mean()
    print(f"{name.upper():8s}: {n_content:,} articles with content | avg {avg_len:.0f} chars/article")

# Total story corpus
total_articles = sum(len(df) for df in raw_dfs.values())
print(f"\nTotal: {total_articles:,} raw articles across 3 platforms")

## Part 2 · Character Mention Extraction

Each article is segmented into sentences. A *trigger-term detector* scans
every sentence for character-relevant terms (gendered nouns, kinship terms,
ghost keywords, gendered pronouns). When a trigger is found, the sentence
is packaged as a **character mention record** with its surrounding context window.

This produces a `mentions_df` that is the foundation for all downstream analyses.
No pre-existing entity labels are used.

In [ ]:
# ── Part 2.1: Multilingual lexicons ──────────────────────────────────────────

# ── 1. Gender lexicon (entity_canon lookup + subword matching) ─────────────────
GENDER_LEXICON = {
    'zh': {
        'Female': [
            '媽','母','媽媽','母親','阿母','老媽',
            '姊','姐','姐姐','姊姊','大姐','二姐',
            '妹','妹妹','小妹','妹仔',
            '女兒','阿嬤','奶奶','外婆','姑姑',
            '阿姨','舅媽','伯母','嫂嫂','嫂','嫂子',
            '婆婆','岳母','外祖母','祖母','曾祖母','太奶奶',
            '女生','女人','女孩','女童','女娃',
            '學姊','女同學','女學生','女老師','女網友','大媽'
            '女友','前女友','學妹','女乘客','女子',
            '嬸嬸','姨媽','太太','老婆','妻子','新娘','婦人','小姐','姑媽',
            '嬸婆','姑婆','姪女','外甥女','孫女',
            '妻','媳婦','前妻',
            '學姐','女士','女性','少女','婦女','老婦','少婦',
            '女醫生','女護士','女警','女主人',
            '女鬼','女靈','女魂','女妖','冤女','鬼女','女鬼仔','紅衣女子','白衣女子',
            '她',
        ],
        'Male': [
            '爸','父','爸爸','父親','阿爸','老爸',
            '哥','哥哥','大哥','二哥',
            '弟','弟弟','小弟',
            '兒子','阿公','爺爺','外公','阿伯',
            '叔叔','舅舅','伯伯','姑丈','姨丈',
            '公公','岳父','外祖父','祖父','曾祖父','太爺爺',
            '先生','老公','丈夫','新郎','前夫',
            '男孩','男生','男同學','學長','學弟','男士',
            '男子','男人','男性','老翁','老頭','壯漢',
            '男老師','男醫生','男警察','男主人',
            '男鬼','男靈','男魂','男妖','鬼男',
            '他',
            '男學生','男網友','男童','男娃',
            '男友','前男友','女婿','男乘客','少年',
            '大叔','大伯','大舅','叔公','舅公','伯公','小叔','姪子','外甥子','孫子'
        ],
    },
    'ko': {
        'Female': [
            '엄마','어머니','어미','모친','할머니','외할머니',
            '언니','누나','여동생','딸','며느리',
            '이모','고모','형수','올케','시어머니','장모','외숙모',
            '아내','부인','와이프','전와이프',
            '여자','소녀','여성','아가씨','여사',
            '여학생','여선생','여의사','여경','여주인',
            '여귀','여령','귀녀','여자귀신',
            '그녀','그 여자',
            '아주머니','여왕','공주','무당','무속인','여승','수녀',
            '큰엄마','작은엄마','증조할머니','여사님','계집','증조외할머니','증조할머니'
        ],
        'Male': [
            '아빠','아버지','아비','부친','할아버지','외할아버지',
            '형','오빠','남동생','아들',
            '삼촌','외삼촌','매형','처남','시아버지','장인',
            '남편','남자','소년','남성','신랑','전남편',
            '남학생','남선생','남의사','남경찰','남주인',
            '남귀','남령','귀남','남자귀신',
            '그 남자',
            '아저씨','총각','왕자','스님','법사','도사','박수무당',
            '큰아버지','작은아버지','증조할아버지','영감','아범','놈','증조외할아버지','증조할아버지'
        ],
    },
    'en': {
        'Female': [
            'woman','girl','lady','female','daughter','sister','mother','mom','mum',
            'wife','aunt','grandmother','grandma','niece','bride','widow','stepmom',
            'stepsister','mother-in-law','sister-in-law','ex-girlfriend',
            'girlfriend','fiancee','ex-wife','housewife',
            'waitress','actress','hostess','nun','witch',
            'female ghost','girl ghost','spirit woman','banshee',
            'mommy','granny','miss','mrs','ms','madam',
            'stepmother','female teacher','nurse',
            'maid','landlady','schoolgirl','little girl',
            'old woman','young woman','elderly woman','female figure'
        ],
        'Male': [
            'man','boy','guy','male','son','brother','father','dad',
            'husband','uncle','grandfather','grandpa','nephew','groom','widower',
            'stepfather','stepbrother','father-in-law','brother-in-law','ex-boyfriend',
            'boyfriend','fiance','ex-husband','waiter','actor','host','priest','monk',
            'male ghost','ghost man','spirit man','phantom man',
            'daddy','grampa','gentleman','sir','mister','mr',
            'stepfather','male teacher','landlord',
            'schoolboy','little boy','old man','young man','elderly man','male figure'
        ],
    },
}

# ── 2. Gendered pronouns ───────────────────────────────────────────────────────
PRONOUNS = {
    'zh': {
        'Female': ['她','她們'],
        'Male':   ['他','他們'],
    },
    'ko': {
        'Female': ['그녀','그녀가','그녀를','그녀의','그녀는','그녀도'],
        # '그' alone is gender-neutral — excluded
    },
    'en': {
        'Female': ['she','her','hers','herself'],
        'Male':   ['he','him','his','himself'],
    },
}

# ── 3. Ghost / supernatural keywords ─────────────────────────────────────────
GHOST_KWS = {
    'zh': [
        '鬼','幽靈','幽魂','亡靈','亡魂','陰靈','冤魂','孤魂','野鬼',
        '靈異','靈體','靈魂','魂魄','鬼魂','好兄弟','魔神仔','女鬼','男鬼',
        '水鬼','嬰靈','附身','超自然','鬼怪','妖怪','邪靈','厲鬼','惡鬼',
        '鬼壓床','鬼打牆','陰氣','陰風','白色身影','黑色身影',
        '詭異聲音','靈異事件','撞鬼','見鬼','遇鬼','鬼影','鬼聲',
        '陰間','孤魂野鬼',
        '腳步聲','敲門聲','冷風','黑影','白影','陰冷','陰森',
        '託夢','卡到陰','顯靈','現身','跟回家','纏上','纏著','索命',
        '抓交替','找替身','還魂','還陽','報復','報仇','冤魂不散',
        '超渡','超度',
        '法會','招魂','送煞','驅邪','驅魔',
        '被跟','被盯上','被牽走',
        '阿飄','飄仔','紅衣小女孩','紅衣女鬼','白衣女鬼',
        '紅衣女子','白衣女子','鬼差','黑白無常','無頭鬼','小鬼','鬼嬰',
        '送肉粽','床邊有人','那個東西','嬰靈',
        '突然不見','忽然不見','瞬間消失','轉頭就不見','一下子不見','憑空消失',
        '剛剛明明還在','回頭就沒人','突然消失',
        '沒有臉','看不清臉','臉看不清楚','五官模糊','沒有五官','整張臉霧霧的','臉是一片黑',
        '腳沒有著地','漂在空中','浮在半空中','身體扭曲','頭轉過來','倒著爬',
        '不是正常人','像飄著',
        '一直盯著我','站在那裡不動','直直看著我','死盯著我','一動也不動',
        '站在門口','站在床邊',
        '有人靠近','拖地聲','哭聲','笑聲','有人在耳邊說話','高跟鞋聲','鐵鍊聲','嬰兒哭聲','尖叫聲','尖叫'
    ],
    'ko': [
        '귀신','유령','혼령','망령','영혼','도깨비','원귀','악령',
        '귀물','음령','잡귀','저승사자','귀녀','처녀귀신',
        '빙의','홀린','신들림','원혼',
        '귀신 소리','귀신이 나타','귀신을 봤','섬뜩','소름 돋',
        '넋','사령','한복','신들린',
        '가위눌림','발소리','인기척','검은형체','으슥한','몽중몽',
        '냉기','문소리','두드리는 소리',
        '현몽','귀신들림','나타나다','현신','돌아오다',
        '원한','복수','목숨을 노리다','괴롭히다','혼불','꿈에 나타났다',
        '얼굴이 없','얼굴이 안 보','얼굴이 안보','얼굴이 보이지 않',
        '형체를 알 수 없','눈코입이 없','얼굴은 나무에 가려',
        '발이 닿지 않','공중에 떠','허공에 떠','다리가 없','목이 돌아가',
        '기어오','기어왔','허리가 꺾','사람이 아닌 것 같',
        '다가왔','스쳐지나갔','발소리만 났','속삭였','웃음소리가 났',
        '총각귀신','몽달귀신','물귀신','달걀귀신','손각시',
        '악귀','구미호','머슴귀신','요괴','자유로귀신',
        '빨간 마스크','화장실 귀신','여귀신','남귀신','여자귀신','남자귀신',
        '지박령','부유령','수귀','동자귀신'
    ],
    'en': [
        'ghost','spirit','apparition','specter','spectre','phantom',
        'wraith','revenant','poltergeist','banshee','shade','entity',
        'presence','shadow figure','dark figure','demon','djinn',
        'haunt','haunting','haunted','possessed','possession',
        'supernatural','paranormal','unexplained','eerie','unearthly',
        'sleep paralysis','cold spot','disembodied','cursed','omen',
        'ouija board','spirit box','EVP','EMF','sage smudging','smudging',
        'skinwalker','jinn','qareen','cryptid','fae',
        'disembodied voice','footsteps','knocking','banging',
        'cold touch','shadow like',
        'ghoul','devil','haunter','shadow','shadow person',
        'black figure','humanoid figure','faceless figure',
        'woman in white','lady in white','white lady','hat man',
        'creature','soul','restless spirit',
        'male spirit','female spirit','child spirit','ghost child',
        'phantom woman','phantom man',
        'appeared','showed up','manifested','manifestation',
        'vanished','disappeared','faded away','materialized','dematerialized',
        'whispered','watched me','watching me','stared at me','followed me',
        'stood at the foot of my bed','stood in the doorway',
        'hovering','floating','crawling on the ceiling',
        'lingering spirit','latched onto','knock the door','scratched the wall',
        'opened the door by itself','called my name','someone','somebody','something',
        'a figure','some figure','someone standing there',
        'something in the hallway','something behind me','something watching me',
        'a presence','some presence','something dark','something pale',
        'someone in white','a shadow','a dark shape','a silhouette',
        'pale','gaunt','faceless','hollow eyes','glowing eyes','sunken eyes',
        'black eyes','white dress','transparent','translucent','shadowy',
        'unnaturally tall','too tall','too thin','skeletal','distorted face',
        'smiling too wide','grinning','standing unnaturally still',
        'sudden cold','smell of death','rotten smell','static feeling',
        'pressure on my chest','goosebumps','hair standing up','felt watched','couldn\'t move',
        'sage','banishing','binding','cleansing'
    ],
}

# ── 4. Death / deceased keywords ──────────────────────────────────────────────
DECEASED_KWS = {
    'zh': [
        '亡故','過世','往生','死去','離世','殞命','身亡','死亡','已故',
        '遺體','喪生','命喪','枉死','橫死','去世','仙逝','罹難',
        '棺材','屍體','遺骸','死屍','亡者','先人','祖先',
        '亡母','亡父','自殺','他殺','病逝','溺死',
        '冤死','含冤','離開人世','命喪黃泉',
        '死掉','死了','喪命','燒炭','上吊','跳樓','墜樓','投河',
        '車禍身亡','被殺','命案','兇殺','靈堂','告別式','頭七','出殯',
        '火化','冥婚','忌日','喪禮','牌位','遺照','輕生','非自然死亡'
    ],
    'ko': [
        '죽은','사망한','고인','죽다','돌아가다','타계','시신',
        '사망','별세','유해','장례','묘지',
        '영정 사진','빈소','부고','순직','자살','타살',
        '익사','병사','급사','횡사','요절',
        '임종','죽음','살해','추락사','사고로 사망','목숨을 잃다',
        '세상을 떠나다','돌아가시다','영안실','장례식','입관','발인','목을 매다','투신',
        '스스로 목숨을 끊다','살해당하다','서거','영면','운명하다','명복을 빌다',
        '죽었다','죽었','사망했다','사망했','돌아가셨다','돌아가셨'
    ],
    'en': [
        'dead','deceased','died','death','passed away','perished',
        'killed','murdered','corpse','funeral',
        'buried','grave','tomb','coffin','obituary','rest in peace',
        'gone','slain','drowned','suicide','homicide',
        'DOA','RIP','IWTD',
        'trauma','passed on','taken away','no longer with us',
        'was killed','got killed','hung himself','hung herself',
        'committed suicide','killed herself','killed himself',
        'dead body','wake','graveyard','cemetery','tombstone','burial',
        'casket','morgue','ashes','cremated',
        'burned to death','stabbed to death','shot to death',
        'fatal accident','murder victim'
    ],
}

# ── 5. Stopwords ──────────────────────────────────────────────────────────────
_PROJ_ROOT = RAW_DIR.parent.parent  # project root (RAW_DIR defined in Part 0.2)

def _load_stopwords(rel_path, fallback=()):
    p = _PROJ_ROOT / rel_path
    if p.exists():
        return {ln.strip() for ln in p.read_text(encoding='utf-8').splitlines()
                if ln.strip() and not ln.startswith('#')}
    return set(fallback)

STOPWORDS_KO = (
    _load_stopwords('~/Downloads/ghost-story-gender-age-analysis/stopwords-ko.txt')       # Aug 2025 file (679 words)
    | _load_stopwords('~/Downloads/ghost-story-gender-age-analysis/stopwords-ko_v2.txt')  # 2017 file (595 words) — merge both
    | {
        # Discourse connectors
        '그래서','그런데','그러면','하지만','그리고','또한','즉','왜냐하면',
        '어떤','이런','저런','아무','정말','진짜','매우','너무','아주','좀','왜냐면'
        # Chosung abbreviations (from topic modeling notebook)
        'ㅎㅎ','ㅋㅋ','ㅠㅠ','ㅜㅜ',
        # Morpheme / particle noise
        '아','음','응','야','헉','어','뭐','것','좀','네','아니',
        '그래','그럼','왜','어떻게','어디','언제','님','수',
        '어요','나오','서',
    }
)

STOPWORDS_EN = _load_stopwords('~/Downloads/ghost-story-gender-age-analysis/data/raw/stopwords-en.txt') | {
    'like','just','really','actually','basically','literally','obviously',
    'maybe','probably','definitely','honestly','apparently','seriously',
}

def _load_stopwords_zh():
    words = set()
    # TCSP Traditional Chinese stopwords (installed in project root)
    try:
        import sys as _sys
        import os 
        _tcsp_path = os.path.expanduser('~/Downloads/ghost-story-gender-age-analysis/TCSP-0.0.9')
        if _tcsp_path not in _sys.path:
            _sys.path.insert(0, _tcsp_path)
        from TCSP import read_stopwords_list
        words.update(read_stopwords_list())
    except (ImportError, Exception):
        pass
    # PTT-specific noise words (from topic modeling notebook)
    words.update({
        '未經授','權者','文章','商業','用途','分享','轉載',
        '嗨','哈','哈哈','呵呵','嘻嘻','嘿嘿','哎呀','啊呀',
        '哦','嗯','啊','喔','唉','哼','哇','呀','咦','哎哟',
        '恕','原文','手機','排版','請見諒','經驗文','真的','住',
        '發文','文章代碼','時間','看板','標題','作者',
    })
    # Baseline fallback (guaranteed even if TCSP fails)
    words.update({
        '的','了','在','是','們',
        '和','與','或','但','而','也','都','很','就','到','從',
        '這','那','有','一','說','看','去','來',
        '什麼','怎麼','哪','誰','為什麼','因為','所以','雖然','如果',
    })
    return words

STOPWORDS_ZH = _load_stopwords_zh()

# ── 6. POS tag whitelists ─────────────────────────────────────────────────────
KEEP_TAGS_ZH = {'n','nr','ns','nt','nz','nw','vn','v','a','ad','an','d'}
KEEP_TAGS_KO = {'NNG','NNP','VV','VA','XR','MAG','NNB'}
KEEP_POS_EN  = {'NOUN','VERB','ADJ','ADV','PROPN'}
NOUN_FORCE_EN = {'ghost', 'haunt', 'shadow', 'spirit', 'figure', 'presence', 'possession'}

# ── 7. Build trigger term lists ───────────────────────────────────────────────
def build_triggers(lang_key):
    triggers = set()
    for terms in GENDER_LEXICON[lang_key].values():
        triggers.update(terms)
    triggers.update(GHOST_KWS[lang_key])
    triggers.update(DECEASED_KWS[lang_key])
    for terms in PRONOUNS.get(lang_key, {}).values():
        triggers.update(terms)
    return sorted(triggers, key=len, reverse=True)  # longest-first greedy

TRIGGERS = {lang: build_triggers(lang) for lang in ['zh','ko','en']}

print("Lexicons ready.")
for lang in ['zh','ko','en']:
    g = sum(len(v) for v in GENDER_LEXICON[lang].values())
    print(f"  {lang}: {len(TRIGGERS[lang])} trigger terms | gender_lex={g} | "
          f"ghost={len(GHOST_KWS[lang])} | deceased={len(DECEASED_KWS[lang])}")
print(f"  stopwords: ko={len(STOPWORDS_KO)} en={len(STOPWORDS_EN)} zh={len(STOPWORDS_ZH)}")

# ── 8. Custom tokenizer vocabulary ────────────────────────────────────────────
# Defined here; applied in Part 7.1 after jieba/spaCy are loaded.

JIEBA_CUSTOM_WORDS = [
    # Ghost types (must not be split)
    '女鬼','男鬼','紅衣女鬼','魔神仔','好兄弟','水鬼','嬰靈',
    '鬼壓床','鬼打牆','陰陽眼','撞鬼','見鬼','鬼影幢幢',
    # Compound phenomena
    '靈異事件','超自然現象','靈魂出竅','農曆七月',
    '中元節','中元普渡','冤親債主','附身事件',
    # Practitioners
    '陰陽師','風水師傅','通靈師','驅魔師','乩童',
    # Locations (multi-char place names)
    '廢棄醫院','廢棄學校','廢棄旅館','民雄鬼屋','鬼屋',
    # Bodily sensations
    '雞皮疙瘩','毛骨悚然','鬼壓身',
    # Ritual objects
    '符咒','八卦鏡','桃木劍',
    # Taiwanese folk religion / ritual (new)
    '觀落陰','地基主','拜地基主','孤魂野鬼','招魂術',
    '陰陽界','前世冤孽','靈媒師','通靈儀式','民間信仰',
    '清明節','普渡祭祀','祭祖祭拜','超渡儀式','法會','法事',
]

# Load external ghost-story custom words TXT (extends jieba protection to all compounds)
_custom_txt = _PROJ_ROOT / 'custom_words_ghost_analysis_traditional_chinese.txt'
if _custom_txt.exists():
    _ext_words = [
        ln.strip() for ln in _custom_txt.read_text(encoding='utf-8').splitlines()
        if ln.strip() and not ln.startswith('#')
    ]
    JIEBA_CUSTOM_WORDS = list(dict.fromkeys(JIEBA_CUSTOM_WORDS + _ext_words))
else:
    print(f'  WARNING: custom TXT not found at {_custom_txt}')

SPACY_PROTECTED_PHRASES = [
    # Ghost / supernatural phenomena (must stay as one token)
    'sleep paralysis', 'shadow person', 'shadow figure',
    'old hag syndrome', 'near death experience',
    'out of body experience', 'out-of-body experience',
    # Hypnagogic / liminal states
    'half awake', 'half asleep', 'half-awake', 'half-asleep',
    # Locations (multi-word)
    'living room', 'school building', 'abandoned building',
    'abandoned house', 'abandoned school',
    # Paranormal investigation tools / methods (new)
    'ouija board', 'spirit box', 'EVP session',
    'ghost hunting', 'ghost hunter', 'paranormal activity',
    'paranormal investigation', 'haunted location',
    'disembodied voice', 'cold touch', 'sage smudging',
    'urban exploration', 'abandoned mansion',
]

KIWI_CUSTOM_WORDS = [
    # Ghost compound nouns that Kiwi may mis-segment
    '처녀귀신','총각귀신','달걀귀신','손각시','지박령','부유령','동자귀신',
    '귀신들림','빨간마스크','화장실귀신','남자귀신','여자귀신','남귀신','여귀신',
    '귀신이야기','귀신소리','귀신공포','귀신출몰','몽달귀신','물귀신','머슴귀신',
]

print(f"Custom vocab defined: {len(JIEBA_CUSTOM_WORDS)} jieba words "
      f"(incl. TXT file), {len(SPACY_PROTECTED_PHRASES)} spaCy phrases, "
      f"{len(KIWI_CUSTOM_WORDS)} Kiwi words")

# ── 9. Korean initial-consonant (초성) abbreviations ─────────────────────────
# Used for text normalisation BEFORE kiwipiepy tokenisation.
# Maps raw 초성 string → expanded Korean form for lexicon matching.
KOREAN_CHOSUNG_MAP = {
    # Fear / bodily reaction (common in ghost-story posts)
    'ㄷㄷ':   '덜덜',       # trembling / shivering in fear
    'ㅎㄷㄷ': '후덜덜',     # greatly trembling (more intense)
    'ㅂㄷ':   '벌벌',       # shaking
    'ㄷㄱ':   '덜컹',       # sudden thud sound
    # Distress / expletive markers
    'ㅆㅂ':   '씨발',       # strong expletive — peak distress marker
    'ㅁㅊ':   '미쳤어',     # "this is insane" — disbelief / extreme surprise
    # Mental health / crisis
    'ㄱㅎ':   '공황',       # panic / panic attack
    'ㅈㅅ':   '죄송',       # sorry (most common); also 자살시도 in crisis context
    # General affect baseline
    'ㄱㅅ':   '감사',       # thanks
    'ㅇㅇ':   '응응',       # yes / affirmative
    'ㄴㄴ':   '노노',       # no no
}

# ── 10. Haunted location keywords ─────────────────────────────────────────────
# NOT added to build_triggers() — too generic alone.
# Available for location-type filtering / sub-corpus analysis.
HAUNTED_LOCATION_KWS = {
    'zh': [
        '廢棄醫院','廢棄學校','廢棄工廠','廢棄旅館','民雄鬼屋',
        '地下室','天橋','墓地','墳場','靈骨塔','納骨塔',
        '宿舍','公廁','山區','山谷','軍營',
    ],
    'ko': [
        '아파트','엘리베이터','지하주차장','기숙사','화장실',
        '공사장','횡단보도','사거리','산계곡','별장',
        '폐건물','폐병원','폐학교','묘지','군부대',
        '지하철역','골목','군대'
    ],
    'en': [
        'haunted house', 'abandoned mansion', 'abandoned hospital',
        'abandoned school', 'abandoned building',
        'graveyard', 'cemetery', 'military base', 'dormitory', 'dorm',
        'basement', 'attic', 'woods', 'forest', 'mountain',
        'elevator', 'bathroom', 'toilet', 'construction site',
        'underground parking',
    ],
}

# ── 11. Affect abbreviations (English Reddit + internet) ─────────────────────
# Emotional-signal abbreviations that appear in ghost-story posts.
# Source: Ashpex/Slang-Word + kaspercools/genz-dataset (GitHub).
# Used for affect analysis (Part 8+); NOT supernatural trigger terms.
AFFECT_ABBREVS = {
    # Fear / shock
    'WTF':   'what the f***',
    'OMG':   'oh my god',
    'OMFG':  'oh my f***ing god',
    'FFS':   "for f***'s sake",
    'LMAO':  'laughing my ass off',    # ironic coping
    # Epistemic stance / credibility markers
    'NGL':   'not gonna lie',
    'TBH':   'to be honest',
    'IDK':   "i don't know",
    'IYKYK': 'if you know you know',
    # Death / crisis
    'DOA':   'dead on arrival',
    'RIP':   'rest in peace',
    'IWTD':  'i want to die',
}

print(f"Korean 초성 map: {len(KOREAN_CHOSUNG_MAP)} entries")
print(f"Haunted locations: zh={len(HAUNTED_LOCATION_KWS['zh'])} "
      f"ko={len(HAUNTED_LOCATION_KWS['ko'])} en={len(HAUNTED_LOCATION_KWS['en'])}")
print(f"Affect abbreviations: {len(AFFECT_ABBREVS)} entries")


In [ ]:
# ── Part 2.1B: Split ghost lexicon into ENTITY vs INDIRECT cues ─────────────
# Goal:
# 1) extract_mentions should trigger on character-like entities only
# 2) indirect supernatural cues are used later for state inference, not entity extraction

GHOST_ENTITY_KWS = {
    'zh': [
        '鬼','幽靈','幽魂','亡靈','亡魂','陰靈','冤魂','孤魂','野鬼',
        '鬼魂','女鬼','男鬼','水鬼','嬰靈','厲鬼','惡鬼','邪靈',
        '阿飄','飄仔','鬼差','黑白無常','無頭鬼','小鬼','鬼嬰',
        '紅衣女鬼','白衣女鬼','孤魂野鬼',
        '靈體','靈魂','魂魄','鬼魂','好兄弟','魔神仔','鬼怪','妖怪',
        '白色身影','黑色身影','鬼影','紅衣小女孩',
        '紅衣女子','白衣女子',
    ],
    'ko': [
        '귀신','유령','혼령','망령','영혼','도깨비','원귀','악령','귀물','잡귀',
        '저승사자','귀녀','처녀귀신','원혼','총각귀신','몽달귀신','물귀신','달걀귀신',
        '손각시','악귀','요괴','여귀신','남귀신','여자귀신','남자귀신','지박령','부유령',
        '수귀','동자귀신','검은형체','구미호','머슴귀신','자유로귀신',
        '빨간 마스크','화장실 귀신','지박령','부유령','수귀','동자귀신'
    ],
    'en': [
        'ghost','spirit','apparition','specter','spectre','phantom','wraith','revenant',
        'poltergeist','banshee','shade','demon','djinn','jinn','ghoul','haunter',
        'shadow person','shadow figure','black figure','humanoid figure','faceless figure',
        'restless spirit','male spirit','female spirit','child spirit','ghost child',
        'phantom woman','phantom man','woman in white','lady in white','white lady','hat man',
        'entity','presence','dark figure','demon','djinn','skinwalker','jinn','qareen','fae',
        'devil','shadow person','soul','a dark shape','a silhouette','faceless',
    ],
}

# Everything else in GHOST_KWS becomes indirect cue (phenomena / atmosphere / symptoms)
GHOST_INDIRECT_KWS = {
    lang: sorted(set(GHOST_KWS[lang]) - set(GHOST_ENTITY_KWS[lang]), key=len, reverse=True)
    for lang in ['zh', 'ko', 'en']
}


def build_entity_triggers(lang_key):
    triggers = set()
    for terms in GENDER_LEXICON[lang_key].values():
        triggers.update(terms)
    triggers.update(GHOST_ENTITY_KWS[lang_key])
    for terms in PRONOUNS.get(lang_key, {}).values():
        triggers.update(terms)
    return sorted(triggers, key=len, reverse=True)


TRIGGERS_ENTITY = {lang: build_entity_triggers(lang) for lang in ['zh', 'ko', 'en']}

print('Split lexicons ready:')
for lang in ['zh', 'ko', 'en']:
    print(
        f"  {lang}: entity_ghost={len(GHOST_ENTITY_KWS[lang])} | "
        f"indirect_ghost={len(GHOST_INDIRECT_KWS[lang])} | "
        f"entity_triggers={len(TRIGGERS_ENTITY[lang])}"
    )

In [ ]:
import re
# ── Part 2.2: Sentence segmentation functions ────────────────────────────────

# Forum-style text often has: multiple blank lines, irregular spaces,
# separators like "---" / "====", and mixed punctuation.
def _normalize_forum_text(text):
    if pd.isna(text):
        return ''
    t = str(text).replace('\r\n', '\n').replace('\r', '\n')
    t = re.sub(r'\u3000', ' ', t)                  # full-width spaces
    t = re.sub(r'\t+', ' ', t)                     # tabs -> single space
    t = re.sub(r'\n\s*\n+', '\n\n', t)         # collapse many blank lines
    t = re.sub(r'\n[-_=*~]{3,}\n', '\n\n', t)   # separator line as paragraph break
    return t.strip()


def _split_forum_paragraphs(text):
    # Primary split by paragraph break (blank lines)
    return [p.strip() for p in re.split(r'\n\n+', text) if p.strip()]


def _apply_loose_space_split(paragraph, loose_mode=False):
    # In loose mode, 3+ spaces inside one paragraph are treated as weak sentence boundaries.
    if not loose_mode:
        return paragraph
    return re.sub(r' {3,}', '\n', paragraph)


def segment_zh(text, loose_mode=False):
    """Split Chinese forum text by paragraphs and sentence-final punctuation."""
    t = _normalize_forum_text(text)
    if not t:
        return []
    chunks = []
    for p in _split_forum_paragraphs(t):
        p = _apply_loose_space_split(p, loose_mode=loose_mode)
        parts = re.split(r'[。！？!？；;…\n]+', p)
        chunks.extend(parts)
    return [s.strip() for s in chunks if len(s.strip()) >= 4]


def segment_ko(text, loose_mode=False):
    """Split Korean forum text by paragraphs and sentence-final punctuation."""
    t = _normalize_forum_text(text)
    if not t:
        return []
    chunks = []
    for p in _split_forum_paragraphs(t):
        p = _apply_loose_space_split(p, loose_mode=loose_mode)
        parts = re.split(r'[.!?。！？；;…\n]+', p)
        chunks.extend(parts)
    return [s.strip() for s in chunks if len(s.strip()) >= 4]


def segment_en(text, loose_mode=False):
    """Split English forum text by paragraphs and punctuation/newline cues."""
    t = _normalize_forum_text(text)
    if not t:
        return []
    chunks = []
    for p in _split_forum_paragraphs(t):
        p = _apply_loose_space_split(p, loose_mode=loose_mode)
        # Normal sentence boundary + single line breaks as weak boundaries
        parts = re.split(r'(?<=[.!?])\s+|\n+', p)
        chunks.extend(parts)
    return [s.strip() for s in chunks if len(s.strip()) >= 8]


# Default behavior remains unchanged (strict mode)
SEG_FN = {
    'zh': lambda text: segment_zh(text, loose_mode=False),
    'ko': lambda text: segment_ko(text, loose_mode=False),
    'en': lambda text: segment_en(text, loose_mode=False),
}

# Optional loose-mode map (3+ spaces => weak boundaries)
SEG_FN_LOOSE = {
    'zh': lambda text: segment_zh(text, loose_mode=True),
    'ko': lambda text: segment_ko(text, loose_mode=True),
    'en': lambda text: segment_en(text, loose_mode=True),
}

# Test
test_cases = [
    ('zh', '她昨晚出現在門口。\n\n嚇死我了！   那個女鬼一直盯著我看。\n---\n真的很毛。'),
    ('ko', '처녀귀신이 나타났어요.\n\n그녀는 할머니처럼 생겼어요.\n====\n너무 무서웠어요'),
    ('en', 'She appeared at midnight.\n\n   The ghost was terrifying.\nI could not move.\n---\nThen it vanished.'),
    ('zh', '我看到她了   然後她突然不見   超可怕'),
    ('en', 'I heard footsteps   then silence   and the door moved'),
]

print('Strict mode:')
for lang, text in test_cases:
    sents = SEG_FN[lang](text)
    print(f"{lang}: {sents}")

print('\nLoose mode (3+ spaces as weak boundaries):')
for lang, text in test_cases:
    sents = SEG_FN_LOOSE[lang](text)
    print(f"{lang}: {sents}")

In [ ]:
# ── Part 2.3: Character mention extraction ───────────────────────────────────

def _match_terms_in_text(text, terms, lang):
    """Return matched terms in text; en uses token/phrase exact matching."""
    if not text:
        return []
    if lang == 'en':
        t_lower = text.lower()
        toks = re.findall(r'\b\w+\b', t_lower)
        tok_set = set(toks)
        out = []
        for term in terms:
            term_l = str(term).lower()
            if not term_l:
                continue
            if ' ' in term_l or '-' in term_l:
                if re.search(rf'(?<!\w){re.escape(term_l)}(?!\w)', t_lower):
                    out.append(term)
            else:
                if term_l in tok_set:
                    out.append(term)
        return out
    return [term for term in terms if term in text]


def _filter_entity_candidates(candidates, sent, lang, sent_indirect_hits):
    """Remove obvious false-positive entity matches (forum words / phenomenon-only cues)."""
    pronoun_set = set()
    for plist in PRONOUNS.get(lang, {}).values():
        pronoun_set.update(plist)

    blocked_phrases = {
        'zh': ['媽佛板', '媽佛', '鬼壓床', '鬼打牆'],
        'ko': [],
        'en': ['nosleep', 'reddit'],
    }
    generic_ghost_terms = {
        'zh': {'鬼'},
        'ko': {'귀신'},
        'en': {'ghost', 'spirit', 'entity', 'presence', 'shadow'},
    }

    out = []
    for ent in candidates:
        ent_str = str(ent).strip()
        if not ent_str:
            continue

        # zh/ko: avoid one-char noun overmatching, keep pronouns
        if lang in {'zh', 'ko'} and len(ent_str) == 1 and ent_str not in pronoun_set:
            continue

        # Block known forum/phenomenon phrases that should not be treated as characters
        if any(bp in sent and ent_str in bp for bp in blocked_phrases.get(lang, [])):
            continue

        # Generic ghost token appears only as part of an indirect phenomenon phrase
        if ent_str in generic_ghost_terms.get(lang, set()):
            in_indirect_phrase = any((ent_str in cue and cue != ent_str) for cue in sent_indirect_hits)
            has_explicit_entity_phrase = any(
                (ek != ent_str and ent_str in ek and ek in sent)
                for ek in GHOST_ENTITY_KWS.get(lang, [])
            )
            if in_indirect_phrase and not has_explicit_entity_phrase:
                continue

        out.append(ent_str)

    return out


def extract_mentions(article_id, content, lang, platform, ctx_window=2):
    """
    Extract mention rows from entity-like triggers only.
    Indirect supernatural cues are recorded as features, not entity_canon.
    """
    seg_fn = SEG_FN_LOOSE.get(lang, segment_en)
    sents = seg_fn(content)

    # Entity-only triggers (gender lexicon + ghost entities + pronouns)
    trig_entity = TRIGGERS_ENTITY.get(lang, TRIGGERS_ENTITY['en'])

    # Indirect cues and death cues for later state classification features
    indirect_kws = GHOST_INDIRECT_KWS.get(lang, [])
    death_kws = DECEASED_KWS.get(lang, [])

    mentions = []
    seen_pairs = set()

    for i, sent in enumerate(sents):
        matched_entities = _match_terms_in_text(sent, trig_entity, lang)
        if not matched_entities:
            continue

        sent_indirect_hits = _match_terms_in_text(sent, indirect_kws, lang)
        matched_entities = _filter_entity_candidates(
            matched_entities,
            sent,
            lang,
            sent_indirect_hits,
        )
        if not matched_entities:
            continue

        ctx_sents = sents[max(0, i - ctx_window): i + ctx_window + 1]
        context = ' '.join(ctx_sents)

        ctx_indirect_hits = _match_terms_in_text(context, indirect_kws, lang)
        sent_death_hits = _match_terms_in_text(sent, death_kws, lang)
        ctx_death_hits = _match_terms_in_text(context, death_kws, lang)

        for entity_canon in matched_entities:
            dup_key = (entity_canon, sent)
            if dup_key in seen_pairs:
                continue
            seen_pairs.add(dup_key)

            mentions.append({
                'article_id': article_id,
                'platform': platform,
                'lang': lang,
                'entity_canon': entity_canon,
                'sentence': sent,
                'context': context,
                'indirect_hits_sent_n': len(sent_indirect_hits),
                'indirect_hits_ctx_n': len(ctx_indirect_hits),
                'death_hits_sent_n': len(sent_death_hits),
                'death_hits_ctx_n': len(ctx_death_hits),
            })

    return mentions


# Apply to all articles
print("Extracting character mentions from raw articles...")
all_mentions = []

for name, df in raw_dfs.items():
    lang = 'zh' if name == 'ptt' else ('ko' if name == 'theqoo' else 'en')
    count = 0
    for _, row in df.iterrows():
        content = row.get('content', '')
        if pd.isna(content) or str(content).strip() == '':
            continue
        m = extract_mentions(row['article_id'], str(content), lang, name)
        all_mentions.extend(m)
        count += len(m)
    print(f"  {name.upper():8s}: {count:,} character mentions extracted")

mentions_df = pd.DataFrame(all_mentions)
print(f"\nTotal mentions_df: {len(mentions_df):,} rows × {len(mentions_df.columns)} columns")
print(mentions_df.head(3).to_string())

In [ ]:
# ── Part 2.4: Extraction overview ────────────────────────────────────────────
print("=" * 60)
print("CHARACTER MENTION EXTRACTION OVERVIEW")
print("=" * 60)
print(f"Total mentions: {len(mentions_df):,}")
print()
print("Per platform:")
print(mentions_df.groupby('platform').size().rename('n_mentions'))
print()
print("Top-10 most common entity_canon (all platforms):")
print(mentions_df['entity_canon'].value_counts().head(10))
print()
print("Top-5 entity_canon per platform:")
for plat, grp in mentions_df.groupby('platform'):
    top5 = grp['entity_canon'].value_counts().head(5)
    print(f"  {plat.upper()}: {top5.to_dict()}")

## Part 3 · Gender Classification

**Conservative rule-based classifier** (3 layers):
1. Pronoun in sentence → Female / Male
2. entity_canon in gender lexicon → Female / Male
3. Subword character cues in entity_canon → Female / Male
4. None triggered → **Unknown** (preserved, not forced)

Korean-specific: bare `그` is excluded (gender-neutral third-person).

In [ ]:
# ── Part 3.1: Gender classifier ─────────────────────────────────────────────

def get_lang_key(lang):
    s = str(lang).strip().lower()
    if s in ('ko','kr'): return 'ko'
    if s in ('en',):     return 'en'
    return 'zh'  # default for zh / ch / tw


def _match_lexicon_en(entity, lex):
    """English lexicon match: exact token/phrase only (no substring)."""
    entity_l = entity.lower()
    etoks = re.findall(r'\b\w+\b', entity_l)
    etok_set = set(etoks)

    for gender, terms in lex.items():
        for t in terms:
            t_norm = str(t).strip().lower()
            if not t_norm:
                continue
            if ' ' in t_norm or '-' in t_norm:
                if re.search(rf'(?<!\w){re.escape(t_norm)}(?!\w)', entity_l):
                    return gender
            else:
                if t_norm in etok_set:
                    return gender
    return None


def _match_lexicon_zhko_hybrid(entity, lex):
    """
    zh/ko hybrid matching:
    1) exact term match first (high precision)
    2) conservative substring fallback for terms with len >= 2 (better recall)
    """
    e = str(entity).strip()
    if not e:
        return None

    # Pass 1: exact match first
    for gender, terms in lex.items():
        norm_terms = [str(t).strip() for t in terms if str(t).strip()]
        if e in norm_terms:
            return gender

    # Pass 2: conservative fallback
    for gender, terms in lex.items():
        norm_terms = sorted(
            [str(t).strip() for t in terms if str(t).strip()],
            key=len,
            reverse=True,
        )
        for t in norm_terms:
            # Skip single-char fallback to reduce over-matching like 弟/哥 inside compounds.
            if len(t) < 2:
                continue
            if t in e:
                return gender

    return None


def classify_gender(entity_canon, sentence, lang):
    """
    Conservative gender classifier. C1: entity-first layer ordering.
    Layers: entity-pronoun → neutral-supernatural override → lexicon(hybrid)
            → subword → sentence-pronoun (last resort) → Unknown
    """
    entity   = str(entity_canon).strip() if pd.notna(entity_canon) else ''
    sent     = str(sentence).strip()     if pd.notna(sentence)     else ''
    lang_key = get_lang_key(lang)

    # Domain-specific neutral supernatural terms (do not force binary gender)
    ZH_NEUTRAL_SUPERNATURAL = {'好兄弟', '魔神仔', '阿飄', '孤魂野鬼', '鬼魂'}

    lex   = GENDER_LEXICON.get(lang_key, {})
    prons = PRONOUNS.get(lang_key, {})

    # ── Layer 1: entity IS a pronoun (entity-first) ───────────────────────────
    if lang_key == 'en':
        etoks = re.findall(r'\b\w+\b', entity.lower())
        for gender, plist in prons.items():
            if any(p in etoks for p in plist):
                return gender
    else:
        for gender, plist in prons.items():
            if entity in plist:
                return gender

    # ── Layer 1.5: Neutral supernatural override ──────────────────────────────
    if lang_key == 'zh' and any(x in entity for x in ZH_NEUTRAL_SUPERNATURAL):
        return 'Unknown'

    # ── Layer 2: entity_canon in lexicon (language-specific matching) ────────
    if lang_key == 'en':
        lex_hit = _match_lexicon_en(entity, lex)
    else:
        lex_hit = _match_lexicon_zhko_hybrid(entity, lex)
    if lex_hit is not None:
        return lex_hit

    # ── Layer 3: subword cues ────────────────────────────────────────────────
    if lang_key == 'zh':
        if any(c in entity for c in ['女','妹','姐','嬤','媽','娘','姑','姨','婆','嬸']):
            return 'Female'
        if any(c in entity for c in ['男','弟','哥','爸','叔','公','伯','爹','舅']):
            return 'Male'
    elif lang_key == 'ko':
        if any(x in entity for x in ['여자','여성','소녀','여귀','여']):
            return 'Female'
        if any(x in entity for x in ['남자','남성','소년','남귀','남']):
            return 'Male'
    elif lang_key == 'en':
        etoks = re.findall(r'\b\w+\b', entity.lower())
        if any(t in {'woman','girl','female','lady','widow','she','her','hers','herself'} for t in etoks):
            return 'Female'
        if any(t in {'man','boy','male','guy','him','his','he','himself'} for t in etoks):
            return 'Male'

    # ── Layer 4: sentence pronouns (last resort — low precision) ─────────────
    if lang_key == 'en':
        toks = re.findall(r'\b\w+\b', sent.lower())
        for gender, plist in prons.items():
            if any(p in toks for p in plist):
                return gender
    else:
        for gender, plist in prons.items():
            if any(p in sent for p in plist):
                return gender

    return 'Unknown'


# Smoke tests — basic + edge cases
tests = [
    ('女鬼',        '那個女鬼出現了',                'zh', 'Female'),
    ('男鬼',        '男鬼的樣子很可怕',              'zh', 'Male'),
    ('阿嬤',        '阿嬤的鬼魂回來了',              'zh', 'Female'),
    ('ghost',       'She was a terrifying ghost',    'en', 'Female'),
    ('귀신',        '그녀가 귀신처럼 나타났다',       'ko', 'Female'),
    ('할머니',      '할머니 귀신이 나타났어요',        'ko', 'Female'),
    ('她',          '她突然出現在走廊盡頭',           'zh', 'Female'),
    ('他',          '他的聲音從牆壁裡傳出來',         'zh', 'Male'),
    ('she',         'She floated through the wall',  'en', 'Female'),
    ('he',          'He disappeared into the mist',  'en', 'Male'),
    ('그녀',        '그녀가 나를 향해 걸어왔다',      'ko', 'Female'),
    ('好兄弟',      '好兄弟來收驚了',                'zh', 'Unknown'),
    ('魔神仔',      '魔神仔把他帶走了',              'zh', 'Unknown'),
    ('woman',       'The woman in white kept staring at me', 'en', 'Female'),
    ('man',         'An old man stood at the end of the hall', 'en', 'Male'),
    ('여자',        '그 여자가 저를 따라왔어요',     'ko', 'Female'),
    ('남자',        '남자 귀신이 보였어요',          'ko', 'Male'),
    ('unknown_entity', 'It appeared suddenly',       'en', 'Unknown'),
    ('그것',        '그것이 나타났다',               'ko', 'Unknown'),
    ('那個東西',    '那個東西一直看著我',             'zh', 'Unknown'),
]
print("Gender classifier smoke tests:")
all_pass = True
for entity, sent, lang, expected in tests:
    pred = classify_gender(entity, sent, lang)
    ok = '✓' if pred == expected else '✗'
    if pred != expected:
        all_pass = False
    print(f"  {ok} [{lang}] classify_gender('{entity}') → {pred}  (expected: {expected})")
print(f"\n{'All tests PASSED ✓' if all_pass else 'SOME TESTS FAILED ✗'}")

In [ ]:
# ── Part 3.2: Apply to mentions_df ───────────────────────────────────────────

# Pronoun resolution with nearest-distance heuristic + cross-sentence fallback:
# - Within sentence: resolve pronoun to nearest non-pronoun entity
# - If no same-sentence anchor: try previous sentence (last entity), then next sentence (first entity)
def _all_pronouns_for_lang(lang_key):
    p = set()
    for plist in PRONOUNS.get(lang_key, {}).values():
        p.update(plist)
    return p


def _resolve_article_for_gender(article_df):
    """
    Resolve pronouns to nearest anchors, with cross-sentence fallback.
    
    Args:
        article_df: DataFrame for single (article_id, platform, lang) group
    
    Returns:
        Series of resolved entity values (indexed by original positions)
    """
    lk = get_lang_key(article_df['lang'].iloc[0])
    pron_set = _all_pronouns_for_lang(lk)
    
    # Create a working copy with sentence order
    work_df = article_df.copy()
    work_df['_sent_order'] = work_df.groupby('sentence', sort=False).ngroup()
    
    resolved = {}
    
    for idx, row in work_df.iterrows():
        entity = str(row['entity_canon']).strip()
        
        # If not a pronoun, keep as-is
        if entity not in pron_set:
            resolved[idx] = entity
            continue
        
        # It's a pronoun, find nearest anchor
        current_sent_order = row['_sent_order']
        anchor = None
        
        # 1. Search within current sentence (by index proximity)
        same_sent_mask = work_df['_sent_order'] == current_sent_order
        same_sent_indices = work_df[same_sent_mask].index.tolist()
        
        non_pron_in_sent = [
            i for i in same_sent_indices
            if str(work_df.loc[i, 'entity_canon']).strip() not in pron_set
        ]
        
        if non_pron_in_sent:
            # Find nearest non-pronoun by index distance
            nearest_idx = min(non_pron_in_sent, key=lambda i: abs(i - idx))
            anchor = str(work_df.loc[nearest_idx, 'entity_canon']).strip()
            resolved[idx] = anchor
            continue
        
        # 2. Try previous sentence (use last entity in prev sentence)
        prev_sent_order = current_sent_order - 1
        prev_sent_mask = work_df['_sent_order'] == prev_sent_order
        prev_sent_indices = work_df[prev_sent_mask].index.tolist()
        
        non_pron_in_prev = [
            i for i in prev_sent_indices
            if str(work_df.loc[i, 'entity_canon']).strip() not in pron_set
        ]
        
        if non_pron_in_prev:
            # Use last non-pronoun in previous sentence
            anchor = str(work_df.loc[non_pron_in_prev[-1], 'entity_canon']).strip()
            resolved[idx] = anchor
            continue
        
        # 3. Try next sentence (use first entity in next sentence)
        next_sent_order = current_sent_order + 1
        next_sent_mask = work_df['_sent_order'] == next_sent_order
        next_sent_indices = work_df[next_sent_mask].index.tolist()
        
        non_pron_in_next = [
            i for i in next_sent_indices
            if str(work_df.loc[i, 'entity_canon']).strip() not in pron_set
        ]
        
        if non_pron_in_next:
            # Use first non-pronoun in next sentence
            anchor = str(work_df.loc[non_pron_in_next[0], 'entity_canon']).strip()
            resolved[idx] = anchor
            continue
        
        # 4. No anchor found, keep original pronoun
        resolved[idx] = entity
    
    return pd.Series(resolved)


# Apply to all articles
mentions_df['entity_for_gender'] = (
    mentions_df
    .groupby(['article_id', 'platform', 'lang'], group_keys=False)
    .apply(_resolve_article_for_gender)
)

mentions_df['gender_rb'] = mentions_df.apply(
    lambda r: classify_gender(r['entity_for_gender'], r['sentence'], r['lang']),
    axis=1
)

print("Gender distribution (all platforms, new pipeline with distance + cross-sentence heuristic):")
print(mentions_df['gender_rb'].value_counts())
print()
print("Per platform:")
print(mentions_df.groupby(['platform', 'gender_rb']).size().unstack(fill_value=0))

In [ ]:
mentions_df.to_csv(OUT_DIR / 'mentions_with_gender(all platforms).csv', index=False, encoding='utf-8-sig')
mentions_df.head()

## Part 3B · LLM Gender Classification (Zero-Shot & Cascade)

Three classifiers compared:
1. **Rule-Based** (Part 3): fast, free, deterministic
2. **LLM Zero-Shot**: sends entity + sentence to Claude/Qwen for inference
3. **Cascade (Rule → LLM fallback)**: rule-based first; LLM resolves `Unknown` only

**Validation**: compared against entity CSV gold labels — **no new manual annotation needed**.
Entity CSV serves as the external gold standard (it was produced by an independent pipeline).

In [ ]:
import os
# ── Part 3B.1: LLM backend setup ─────────────────────────────────
# Option A (recommended): Anthropic Claude API (claude-haiku-4-5)
#   1. Go to console.anthropic.com → sign up / log in
#   2. Billing → Add $10 credit (enough for the entire thesis)
#   3. API Keys → Create key → copy the sk-ant-... string
#   4. Paste below:  CLAUDE_API_KEY = os.environ.get('ANTHROPIC_API_KEY', '')  # set it in your shell or a .env file; never commit a key
# Option B: Local Ollama (Qwen2.5) — free, no API key, slower

CLAUDE_API_KEY = os.environ.get('ANTHROPIC_API_KEY', '')  # set it in your shell or a .env file; never commit a key
LLM_BACKEND    = 'claude'   # 'claude' | 'ollama' | 'none'
OLLAMA_MODEL   = 'qwen2.5'  # alternatives: 'llama3.2', 'gemma2'

CLAUDE_OK = False
OLLAMA_OK = False
_client   = None

HAIKU_MODEL  = 'claude-haiku-4-5'
SONNET_MODEL = 'claude-sonnet-4-5'   # adjust if your account uses a different model name

if LLM_BACKEND == 'claude' and CLAUDE_API_KEY:
    try:
        import anthropic
        _client = anthropic.Anthropic(api_key=CLAUDE_API_KEY)
        _client.messages.create(
            model=HAIKU_MODEL,
            max_tokens=1,
            messages=[{'role': 'user', 'content': 'hi'}]
        )
        CLAUDE_OK = True
        print(f'✓ Anthropic Claude API: ready OK  (model: {HAIKU_MODEL})')
    except Exception as e:
        print(f'✗ Claude API error: {e}')
        print('  Check your API key or billing at console.anthropic.com')

elif LLM_BACKEND == 'ollama':
    try:
        import requests
        r = requests.get('http://localhost:11434/api/tags', timeout=3)
        OLLAMA_OK = (r.status_code == 200)
        if OLLAMA_OK:
            print(f'✓ Ollama: ready OK  (model: {OLLAMA_MODEL})')
        else:
            print('✗ Ollama reachable but not ready.')
    except Exception:
        print('✗ Ollama not reachable. Start with: ollama serve')

else:
    print('⚠ LLM disabled. Set LLM_BACKEND="claude" or "ollama" to enable.')


In [ ]:
# ── Column naming convention for gender/state ─────────────────────────────────
# gender_rb:      rule-based only result
# gender_new:     FINAL column for all analysis
# gender_cascade: stored separately for annotation export / validation comparison
# gender_method:  rule | haiku | sonnet_fallback | ollama | error | none
# gender_confidence: High | Medium | Low
# gender_evidence: short phrase from LLM or rule-based note


# ── Part 3B.2: Cost estimation before LLM calls ──────────────────────────────
if 'gender_rb' not in mentions_df.columns:
    mentions_df['gender_rb'] = mentions_df.apply(
        lambda r: classify_gender(r['entity_canon'], r['sentence'], r['lang']),
        axis=1
    )

total_mentions = len(mentions_df)
unknown_gender = (mentions_df['gender_rb'] == 'Unknown').sum()
unknown_pct = (unknown_gender / total_mentions * 100) if total_mentions else 0

# rough cost estimate for Haiku
COST_PER_CALL_USD = 0.00022
NTD_PER_USD = 33

cascade_calls = unknown_gender
all_llm_calls = total_mentions

cascade_usd = cascade_calls * COST_PER_CALL_USD
all_llm_usd = all_llm_calls * COST_PER_CALL_USD
cascade_ntd = cascade_usd * NTD_PER_USD
all_llm_ntd = all_llm_usd * NTD_PER_USD

RATE = 200  # req/min
cascade_min = cascade_calls / RATE
all_llm_min = all_llm_calls / RATE

print('=' * 62)
print('COST ESTIMATION  —  Rule-Based Pre-scan Complete')
print('=' * 62)
print(f'  Total mentions in corpus : {total_mentions:>10,}')
print(f'  Unknown after rule-based : {unknown_gender:>10,}  ({unknown_pct:.1f}%)')
print()
print('  Scenario          | API calls | USD    | NTD    | Time')
print('  ' + '-' * 58)
print(f'  Cascade (Unknown) | {cascade_calls:>9,} | ${cascade_usd:>5.2f}  | {cascade_ntd:>5.0f}  | {cascade_min:.0f} min')
print(f'  LLM for ALL       | {all_llm_calls:>9,} | ${all_llm_usd:>5.2f}  | {all_llm_ntd:>5.0f}  | {all_llm_min/60:.1f} hrs')
print()
print('RECOMMENDED:  Cascade  (✔ fast, ✔ cheap, ✔ preserves conservative Unknown)')
print()
print('Set your choice in the config below, then run the LLM cells:')
print('  CHOSEN_GENDER = "cascade"  # rule-based + Haiku/Sonnet for Unknown only')
print('  CHOSEN_GENDER = "llm"      # LLM for ALL mentions')
print('  CHOSEN_GENDER = "rule"     # rule-based only')

CHOSEN_GENDER = 'cascade'
CHOSEN_STATE  = 'cascade'
print(f'\nChosen method: GENDER={CHOSEN_GENDER}, STATE={CHOSEN_STATE}')


In [ ]:
# ── Part 3B.3: LLM gender classifier (define functions) ──────────────────────
import re
import time

USE_SONNET_FALLBACK = True
FALLBACK_ON_UNKNOWN = True
FALLBACK_ON_LOW_CONF = True

GENDER_SYS = (
    "You are a multilingual NLP classifier for academic research on online ghost stories.\n"
    "Your task is to classify the gender of ONE character mention based only on the provided sentence and local context.\n\n"

    "Important rules:\n"
    "1. Classify the gender of THIS mention only, not the whole story.\n"
    "2. Use only explicit textual evidence from the sentence and context.\n"
    "3. If gender is not clearly supported, return Unknown.\n"
    "4. Do NOT infer gender from stereotypes, occupations, emotions, or actions.\n"
    "5. Ghosts, spirits, and dead characters can still be Female or Male if the text explicitly indicates gender.\n"
    "6. Kinship terms, honorifics, pronouns, and explicitly gendered nouns are valid evidence.\n"
    "7. If the mention refers to a group, ambiguous person, omitted subject, or unclear antecedent, return Unknown.\n\n"

    "Valid labels:\n"
    "- Female = clearly female from explicit textual evidence\n"
    "- Male = clearly male from explicit textual evidence\n"
    "- Unknown = not enough evidence to determine gender\n\n"

    "Return EXACTLY in this format:\n"
    "Gender: <Female|Male|Unknown>\n"
    "Confidence: <High|Medium|Low>\n"
    "Evidence: <short phrase>"
)

def parse_gender_response(text):
    gender = 'Unknown'
    confidence = 'Low'
    evidence = ''

    m1 = re.search(r'Gender:\s*(Female|Male|Unknown)', text, re.I)
    if m1:
        gender = m1.group(1).capitalize()

    m2 = re.search(r'Confidence:\s*(High|Medium|Low)', text, re.I)
    if m2:
        confidence = m2.group(1).capitalize()

    m3 = re.search(r'Evidence:\s*(.*)', text, re.I)
    if m3:
        evidence = m3.group(1).strip()

    if gender not in ('Female', 'Male', 'Unknown'):
        gender = 'Unknown'
    if confidence not in ('High', 'Medium', 'Low'):
        confidence = 'Low'

    return gender, confidence, evidence

def _build_gender_msg(entity, sentence, context, lang):
    lang_names = {'zh': 'Traditional Chinese', 'ko': 'Korean', 'en': 'English'}
    return (
        f"Language: {lang_names.get(lang, lang)}\n"
        f"Character mention: {entity}\n"
        f"Sentence: {sentence}\n"
        f"Local context: {str(context)[:500]}\n\n"
        f"Classify the gender of this mention."
    )

def _claude_gender_with_model(entity, sentence, context, lang, model_name):
    msg = _build_gender_msg(entity, sentence, context, lang)
    resp = _client.messages.create(
        model=model_name,
        max_tokens=40,
        system=GENDER_SYS,
        messages=[{'role': 'user', 'content': msg}]
    )
    raw = resp.content[0].text.strip()
    gender, confidence, evidence = parse_gender_response(raw)
    return {
        'gender': gender,
        'confidence': confidence,
        'evidence': evidence,
        'raw_response': raw
    }

def _ollama_gender(entity, sentence, context, lang):
    import requests as _req
    lang_names = {'zh': 'Traditional Chinese', 'ko': 'Korean', 'en': 'English'}
    prompt = (
        f"{GENDER_SYS}\n\n"
        f"Language: {lang_names.get(lang, lang)}\n"
        f"Character mention: {entity}\n"
        f"Sentence: {sentence}\n"
        f"Local context: {str(context)[:400]}\n"
    )
    payload = {'model': OLLAMA_MODEL, 'prompt': prompt, 'stream': False}
    resp = _req.post('http://localhost:11434/api/generate', json=payload, timeout=60)
    raw = resp.json().get('response', '').strip()
    gender, confidence, evidence = parse_gender_response(raw)
    return {
        'gender': gender,
        'confidence': confidence,
        'evidence': evidence,
        'raw_response': raw
    }

def classify_gender_llm(entity, sentence, context, lang):
    try:
        if CLAUDE_OK:
            r1 = _claude_gender_with_model(entity, sentence, context, lang, HAIKU_MODEL)
            g1, c1 = r1['gender'], r1['confidence']

            needs_fallback = (
                USE_SONNET_FALLBACK and (
                    (FALLBACK_ON_LOW_CONF and c1 == 'Low') or
                    (FALLBACK_ON_UNKNOWN and g1 == 'Unknown')
                )
            )

            if needs_fallback:
                r2 = _claude_gender_with_model(entity, sentence, context, lang, SONNET_MODEL)
                r2['model_used'] = 'sonnet_fallback'
                return r2

            r1['model_used'] = 'haiku'
            return r1

        if OLLAMA_OK:
            r = _ollama_gender(entity, sentence, context, lang)
            r['model_used'] = 'ollama'
            return r

    except Exception as e:
        return {
            'gender': 'Unknown',
            'confidence': 'Low',
            'evidence': f'ERROR: {e}',
            'raw_response': '',
            'model_used': 'error'
        }

    return {
        'gender': 'Unknown',
        'confidence': 'Low',
        'evidence': '',
        'raw_response': '',
        'model_used': 'none'
    }

def classify_gender_cascade(entity, sentence, context, lang):
    rb = classify_gender(entity, sentence, lang)

    if rb != 'Unknown':
        return {
            'gender': rb,
            'confidence': 'High',
            'evidence': 'rule-based explicit cue',
            'raw_response': '',
            'method': 'rule'
        }

    llm_res = classify_gender_llm(entity, sentence, context, lang)
    return {
        'gender': llm_res['gender'],
        'confidence': llm_res['confidence'],
        'evidence': llm_res['evidence'],
        'raw_response': llm_res['raw_response'],
        'method': llm_res['model_used']
    }

print('LLM gender classifier + cascade defined.')
n_unk = (mentions_df['gender_rb'] == 'Unknown').sum()
print(f'Rule-based Unknown: {n_unk:,}/{len(mentions_df):,} ({n_unk/len(mentions_df)*100:.1f}%)')

if CLAUDE_OK:
    est = n_unk * COST_PER_CALL_USD
    print(f'Estimated Claude cost for full cascade: ~${est:.2f} USD ({HAIKU_MODEL}, before Sonnet fallback)')

In [ ]:
import numpy as np
import pandas as pd
import time
# ── Part 3B.4: Exploratory sample inspection ─────────────────────────────────
# Quick spot-check only. Formal evaluation should use human annotation in Part 11.

LLM_N = 10

if (CLAUDE_OK or OLLAMA_OK):
    parts = []
    for plat in ['ptt', 'reddit', 'theqoo']:
        sub = mentions_df[
            (mentions_df['platform'] == plat) &
            (mentions_df['gender_rb'] == 'Unknown')
        ]
        if len(sub) > 0:
            parts.append(sub.sample(min(LLM_N, len(sub)), random_state=42))

    if parts:
        inspect_sample = pd.concat(parts, ignore_index=True).copy()

        inspect_sample['g_rule'] = inspect_sample.apply(
            lambda r: classify_gender(r['entity_canon'], r['sentence'], r['lang']),
            axis=1
        )

        out_gender, out_conf, out_evi, out_method = [], [], [], []
        for _, row in inspect_sample.iterrows():
            res = classify_gender_cascade(
                row['entity_canon'], row['sentence'], row['context'], row['lang']
            )
            out_gender.append(res['gender'])
            out_conf.append(res['confidence'])
            out_evi.append(res['evidence'])
            out_method.append(res['method'])
            time.sleep(0.05)

        inspect_sample['g_cascade'] = out_gender
        inspect_sample['g_conf'] = out_conf
        inspect_sample['g_evidence'] = out_evi
        inspect_sample['g_method'] = out_method

        inspect_sample.to_csv(OUT_DIR / 'gender_llm_spotcheck_sample.csv', index=False)
        print(f"Saved: {OUT_DIR / 'gender_llm_spotcheck_sample.csv'}")
    else:
        print("No Unknown cases available for spot-check.")
else:
    print("LLM not available. Skipping spot-check.")


In [ ]:
# ── Part 3B.5: Apply chosen method to full mentions_df ───────────────────────
print(f'Applying gender method: {CHOSEN_GENDER}')

if CHOSEN_GENDER == 'rule' or not (CLAUDE_OK or OLLAMA_OK):
    mentions_df['gender_new'] = mentions_df['gender_rb']
    mentions_df['gender_cascade'] = mentions_df['gender_rb']
    mentions_df['gender_method'] = 'rule'
    mentions_df['gender_confidence'] = np.where(
        mentions_df['gender_rb'] == 'Unknown', 'Low', 'High'
    )
    mentions_df['gender_evidence'] = np.where(
        mentions_df['gender_rb'] == 'Unknown',
        'rule-based no explicit cue',
        'rule-based explicit cue'
    )

    print('Using rule-based gender results.')
    if CHOSEN_GENDER != 'rule':
        print(f"  Requested {CHOSEN_GENDER!r} but no LLM available — fell back to rule-based.")

elif CHOSEN_GENDER == 'cascade':
    from tqdm.auto import tqdm

    CKPT_GENDER = OUT_DIR / 'ckpt_gender.csv'

    if CKPT_GENDER.exists():
        ckpt = pd.read_csv(CKPT_GENDER)

        required_ckpt_cols = ['gender_cascade', 'gender_method', 'gender_confidence', 'gender_evidence']
        missing_ckpt = [c for c in required_ckpt_cols if c not in ckpt.columns]
        if missing_ckpt:
            raise ValueError(
                f"Old checkpoint format detected. Missing columns: {missing_ckpt}. "
                f"Delete {CKPT_GENDER} and re-run."
            )

        g_new  = list(ckpt['gender_cascade'])
        g_meth = list(ckpt['gender_method'])
        g_conf = list(ckpt['gender_confidence'])
        g_evi  = list(ckpt['gender_evidence'])
        start = len(g_new)
        print(f'  Resuming from checkpoint ({start:,} done)...')
    else:
        g_new, g_meth, g_conf, g_evi, start = [], [], [], [], 0

    rows = list(mentions_df.iterrows())

    for _, r in tqdm(rows[start:], total=len(rows)-start, desc='Gender cascade'):
        res = classify_gender_cascade(
            r['entity_canon'], r['sentence'], r['context'], r['lang']
        )
        g_new.append(res['gender'])
        g_meth.append(res['method'])
        g_conf.append(res['confidence'])
        g_evi.append(res['evidence'])

        if len(g_new) % 500 == 0:
            pd.DataFrame({
                'gender_cascade': g_new,
                'gender_method': g_meth,
                'gender_confidence': g_conf,
                'gender_evidence': g_evi
            }).to_csv(CKPT_GENDER, index=False)

    # save final checkpoint too
    pd.DataFrame({
        'gender_cascade': g_new,
        'gender_method': g_meth,
        'gender_confidence': g_conf,
        'gender_evidence': g_evi
    }).to_csv(CKPT_GENDER, index=False)

    all_idx = [idx for idx, _ in rows]
    if len(g_new) != len(all_idx):
        raise RuntimeError(
            f"Cascade length mismatch: {len(g_new)} results vs {len(all_idx)} rows. "
            "Delete checkpoint and re-run."
        )

    mentions_df.loc[all_idx, 'gender_cascade'] = g_new
    mentions_df.loc[all_idx, 'gender_method'] = g_meth
    mentions_df.loc[all_idx, 'gender_confidence'] = g_conf
    mentions_df.loc[all_idx, 'gender_evidence'] = g_evi
    mentions_df['gender_new'] = mentions_df['gender_cascade']

    rule_n = (mentions_df['gender_method'] == 'rule').sum()
    llm_n = mentions_df['gender_method'].isin(['haiku', 'sonnet_fallback', 'ollama']).sum()
    print(f'  Rule-based: {rule_n:,} | LLM-resolved: {llm_n:,}')

elif CHOSEN_GENDER == 'llm':
    from tqdm.auto import tqdm

    g_new, g_meth, g_conf, g_evi = [], [], [], []

    for _, r in tqdm(mentions_df.iterrows(), total=len(mentions_df), desc='Gender LLM'):
        res = classify_gender_llm(
            r['entity_canon'], r['sentence'], r['context'], r['lang']
        )
        g_new.append(res['gender'])
        g_meth.append(res['model_used'])
        g_conf.append(res['confidence'])
        g_evi.append(res['evidence'])

    mentions_df['gender_new'] = g_new
    mentions_df['gender_cascade'] = g_new
    mentions_df['gender_method'] = g_meth
    mentions_df['gender_confidence'] = g_conf
    mentions_df['gender_evidence'] = g_evi

else:
    raise ValueError(f"Unsupported CHOSEN_GENDER: {CHOSEN_GENDER}")

print(f'  Final: {mentions_df["gender_new"].value_counts().to_dict()}')


# ── Method transparency summary ───────────────────────────────────────────────
_actual_gender_method = 'rule' if (CHOSEN_GENDER == 'rule' or not (CLAUDE_OK or OLLAMA_OK)) else CHOSEN_GENDER

print(f"\n{'─'*60}")
print("GENDER CLASSIFICATION SUMMARY")
print(f"  Requested method : {CHOSEN_GENDER}")
print(f"  Actual method    : {_actual_gender_method}")

if 'gender_method' in mentions_df.columns:
    _mc = mentions_df['gender_method'].value_counts()
    for _m, _n in _mc.items():
        print(f"  {_m:16s}: {_n:,} rows")

print(f"  Final distribution: {mentions_df['gender_new'].value_counts().to_dict()}")


In [ ]:
mentions_df.to_csv(OUT_DIR / 'mentions_final(gender).csv', index=False)
mentions_df.head()

## Part 4 · State Classification

**Hierarchical rule-based classifier** — No pre-existing labels used.

State categories:
- **Supernatural**: entity that died AND returned to interact with the living
- **Deceased**: entity that died but has NOT returned
- **Alive**: entity is a living person
- **Unknown**: insufficient context to determine

Classification proceeds through 4 context levels (increasing breadth):
1. entity_canon itself contains ghost keyword → Supernatural
2. Sentence contains ghost keyword → Supernatural
3. ±2-sentence context contains ghost keyword → Supernatural
4. Sentence/context contains death keyword → Deceased
5. Default → Alive; if context too short → Unknown

In [ ]:
# ── Part 4.1: Hierarchical state classifier ──────────────────────────────────
def _count_hits(text, terms, lang):
    if pd.isna(text) or str(text).strip() == '':
        return 0
    t = str(text)
    if lang == 'en':
        t_lower = t.lower()
        toks = re.findall(r'\b\w+\b', t_lower)
        tok_set = set(toks)
        c = 0
        for term in terms:
            term_l = str(term).lower().strip()
            if not term_l:
                continue
            if ' ' in term_l or '-' in term_l:
                if re.search(rf'(?<!\w){re.escape(term_l)}(?!\w)', t_lower):
                    c += 1
            else:
                if term_l in tok_set:
                    c += 1
        return c
    return sum(1 for term in terms if term in t)


def classify_state(entity_canon, sentence, context, lang):
    """
    State classifier with split cues:
    - entity ghost terms => strong supernatural signal
    - indirect cues => weak supernatural signal
    - death keywords => deceased signal
    - Unknown only when entity is completely missing and no evidence exists
    - otherwise default Alive 
    """
    lk = get_lang_key(lang)
    g_entity = GHOST_ENTITY_KWS.get(lk, [])
    g_indirect = GHOST_INDIRECT_KWS.get(lk, [])
    d_kws = DECEASED_KWS.get(lk, [])

    entity = str(entity_canon).strip() if pd.notna(entity_canon) else ''
    sent = str(sentence).strip() if pd.notna(sentence) else ''
    ctx = str(context).strip() if pd.notna(context) else ''

    g_ent_entity = _count_hits(entity, g_entity, lk)
    g_ent_sent   = _count_hits(sent, g_entity, lk)
    g_ent_ctx    = _count_hits(ctx, g_entity, lk)
    g_ind_sent   = _count_hits(sent, g_indirect, lk)
    g_ind_ctx    = _count_hits(ctx, g_indirect, lk)
    d_sent       = _count_hits(sent, d_kws, lk)
    d_ctx        = _count_hits(ctx, d_kws, lk)

    if g_ent_entity > 0:
        return {'state': 'Supernatural', 'confidence': 'High', 'evidence': 'entity ghost term'}

    supernatural_score = (
        2.0 * g_ent_sent +
        1.5 * g_ent_ctx +
        1.5 * g_ind_sent +
        0.5 * g_ind_ctx
    )
    deceased_score = (
        1.5 * d_sent +
        0.5 * d_ctx
    )

    if supernatural_score >= 2.0 and supernatural_score >= (deceased_score):
        return {'state': 'Supernatural', 'confidence': 'High', 'evidence': 'supernatural cues'}

    if entity != '' and d_sent >= 1.0 and deceased_score > supernatural_score:
        return {'state': 'Deceased', 'confidence': 'High', 'evidence': 'death cues'}

    # ambiguous / weak signal: let cascade decide
    if supernatural_score > 0 or deceased_score > 0:
        return {'state': 'Unknown', 'confidence': 'Low', 'evidence': 'weak or mixed cues'}

    if entity == '':
        return {'state': 'Unknown', 'confidence': 'Low', 'evidence': 'missing entity'}

    return {'state': 'Alive', 'confidence': 'Low', 'evidence': 'default fallback'}


# Smoke tests — basic + edge cases
state_tests = [
    # ── Supernatural (ghost actively appearing) ───────────────────────────────
    ('女鬼',  '那個女鬼昨晚出現了',           '',                        'zh', 'Supernatural'),
    ('阿嬤',  '阿嬤的鬼魂回來看我',           '',                        'zh', 'Supernatural'),
    ('ghost', 'A ghost appeared in the doorway', '',                     'en', 'Supernatural'),
    ('귀신',  '귀신이 나타났어요',            '',                        'ko', 'Supernatural'),
    ('유령',  '유령이 저를 따라다녔어요',      '',                        'ko', 'Supernatural'),
    # ── PTT-specific supernatural ─────────────────────────────────────────────
    ('好兄弟','好兄弟一直在門口徘徊',          '',                        'zh', 'Supernatural'),
    ('魔神仔','魔神仔把他帶到山裡去了',        '',                        'zh', 'Supernatural'),
    # ── Reddit-style supernatural ─────────────────────────────────────────────
    ('spirit','The spirit of my grandmother visited me', '',             'en', 'Supernatural'),
    ('shadow','A shadow figure stood at the foot of my bed', '',         'en', 'Supernatural'),
    # ── Deceased (dead but not appearing as ghost) ───────────────────────────
    ('阿公',  '阿公過世了，我很難過',         '他離開人世已經三年',       'zh', 'Deceased'),
    ('father','My father passed away last year', 'He died in hospital',  'en', 'Deceased'),
    ('할머니','할머니가 작년에 돌아가셨어요',  '장례식이 끝났어요',        'ko', 'Deceased'),
    # ── Alive (living person default path) ───────────────────────────────────
    ('同學',  '我跟同學去爬山',               '',                        'zh', 'Alive'),
    ('friend','My friend dared me to go in',   '',                       'en', 'Alive'),
    ('친구',  '친구가 같이 가자고 했어요',    '',                        'ko', 'Alive'),
    # ── Boundary: Deceased vs Supernatural ───────────────────────────────────
    ('woman', 'The woman had died years ago, but now her ghost haunts the halls',
              'She died in 1942',                                         'en', 'Supernatural'),
]
print("State classifier smoke tests:")
all_ok = True
for entity, sent, ctx, lang, exp in state_tests:
    pred = classify_state(entity, sent, ctx, lang)['state']
    ok = '✓' if pred == exp else '✗'
    if pred != exp:
        all_ok = False
    print(f"  {ok} [{lang}] classify_state('{entity}') → {pred}  (expected: {exp})")
print(f"\n{'All tests PASSED ✓' if all_ok else 'SOME TESTS FAILED ✗'}")

In [ ]:
# ── Part 4.2: Apply state classifier ─────────────────────────────────────────
state_rule_res = mentions_df.apply(
    lambda r: classify_state(
        r['entity_canon'], r['sentence'], r['context'], r['lang']
    ),
    axis=1
)

mentions_df['state_rb'] = state_rule_res.map(lambda x: x['state'])
mentions_df['state_rb_confidence'] = state_rule_res.map(lambda x: x['confidence'])
mentions_df['state_rb_evidence'] = state_rule_res.map(lambda x: x['evidence'])
mentions_df.to_csv(OUT_DIR / 'mentions_with_state(all platforms).csv', index=False, encoding='utf-8-sig')

print("State distribution (all platforms, new pipeline):")
print(mentions_df['state_rb'].value_counts())
print()
print("Per platform:")
print(mentions_df.groupby(['platform','state_rb']).size().unstack(fill_value=0))

In [ ]:
mentions_df.head()

## Part 4B · LLM State Classification (Zero-Shot & Cascade)

Same cascade architecture applied to state classification.

State definitions:
- **Supernatural**: ghost/spirit that died AND returned to interact with living
- **Deceased**: died but NOT appearing as ghost
- **Alive**: living person
- **Unknown**: cannot determine from text

In [ ]:
# ── Part 4B.1: LLM state classifier ──────────────────────────────────────────
STATE_SYS = (
    "You are a multilingual NLP classifier for academic research on online ghost stories.\n"
    "Classify the STATE of a character mention.\n\n"

    "Choose EXACTLY ONE label from:\n"
    "Alive, Deceased, Supernatural, Unknown\n\n"

    "Definitions:\n"
    "- Alive: a living human character.\n"
    "- Deceased: a person who has died but is NOT currently appearing as a ghost or interacting with the living.\n"
    "- Supernatural: a ghost, spirit, or dead entity that is actively appearing, interacting, or perceived as a supernatural presence.\n"
    "- Unknown: insufficient or ambiguous evidence.\n\n"

    "Important Rules:\n"
    "1. Classify THIS mention only, based on the sentence and local context.\n"
    "2. If a dead person is described as appearing, interacting, or being seen/heard, classify as Supernatural (NOT Deceased).\n"
    "3. If a dead person is only mentioned as having died (no appearance), classify as Deceased.\n"
    "4. Do NOT infer beyond explicit textual evidence.\n"
    "5. If unclear, choose Unknown.\n"
    "6. Seeing, hearing, or interacting with a dead person implies Supernatural.\n\n"

    "Choose EXACTLY ONE evidence label from:\n"
    "- entity_ghost_term\n"
    "- supernatural_context\n"
    "- death_context\n"
    "- indirect_alive\n"
    "- ambiguous\n"
    "- no_signal\n"
    "- other\n\n"

    "Evidence label guidance:\n"
    "- entity_ghost_term: the mention itself is explicitly a ghost/spirit term.\n"
    "- supernatural_context: the sentence/context shows haunting, appearing, being seen/heard, ghostly interaction, supernatural presence.\n"
    "- death_context: the sentence/context explicitly says the person died/passed away/is dead, without ghostly appearance.\n"
    "- indirect_alive: the context suggests the person is alive through ordinary human actions or disappearance without death/supernatural cues.\n"
    "- ambiguous: evidence is mixed, weak, or unclear.\n"
    "- no_signal: no useful cue is present.\n"
    "- other: some evidence exists but does not fit the above labels well.\n\n"

    "Return EXACTLY in this format:\n"
    "State: <Alive|Deceased|Supernatural|Unknown>\n"
    "Confidence: <High|Medium|Low>\n"
    "EvidenceLabel: <entity_ghost_term|supernatural_context|death_context|indirect_alive|ambiguous|no_signal|other>\n"
)
def parse_state_response(text):
    state = 'Unknown'
    confidence = 'Low'
    evidence = 'other'

    m1 = re.search(r'State:\s*(Alive|Deceased|Supernatural|Unknown)', text, re.I)
    if m1:
        state_map = {
            'alive': 'Alive',
            'deceased': 'Deceased',
            'supernatural': 'Supernatural',
            'unknown': 'Unknown'
        }
        state = state_map.get(m1.group(1).strip().lower(), 'Unknown')

    m2 = re.search(r'Confidence:\s*(High|Medium|Low)', text, re.I)
    if m2:
        conf_map = {
            'high': 'High',
            'medium': 'Medium',
            'low': 'Low'
        }
        confidence = conf_map.get(m2.group(1).strip().lower(), 'Low')

    m3 = re.search(
        r'EvidenceLabel:\s*(entity_ghost_term|supernatural_context|death_context|indirect_alive|ambiguous|no_signal|other)',
        text,
        re.I
    )
    if m3:
        evidence = m3.group(1).strip().lower()

    return state, confidence, evidence

def _claude_state(entity, sentence, context, lang, model_name):
    lang_names = {'zh':'Traditional Chinese','ko':'Korean','en':'English'}

    msg = (
        f"Language: {lang_names.get(lang, lang)}\n"
        f"Character mention: {entity}\n"
        f"Sentence: {sentence}\n"
        f"Local context: {str(context)[:200]}\n\n"
        f"Classify the state of this mention."
    )

    resp = _client.messages.create(
        model=model_name,
        max_tokens=50,
        system=STATE_SYS,
        messages=[{'role':'user','content':msg}]
    )

    raw = resp.content[0].text.strip()
    state, confidence, evidence = parse_state_response(raw)

    return {
        'state': state,
        'confidence': confidence,
        'evidence': evidence,
        'raw_response': raw
    }

def _ollama_state(entity, sentence, context, lang):
    import requests as _req
    lang_names = {'zh':'Traditional Chinese','ko':'Korean','en':'English'}

    prompt = (
        f"{STATE_SYS}\n\n"
        f"Language: {lang_names.get(lang, lang)}\n"
        f"Character mention: {entity}\n"
        f"Sentence: {sentence}\n"
        f"Local context: {str(context)[:200]}\n"
    )

    payload = {'model': OLLAMA_MODEL, 'prompt': prompt, 'stream': False}
    resp = _req.post('http://localhost:11434/api/generate', json=payload, timeout=60)
    raw = resp.json().get('response', '').strip()

    state, confidence, evidence = parse_state_response(raw)

    return {
        'state': state,
        'confidence': confidence,
        'evidence': evidence,
        'raw_response': raw
    }

def classify_state_llm(entity, sentence, context, lang):
    try:
        if CLAUDE_OK:
            r1 = _claude_state(entity, sentence, context, lang, HAIKU_MODEL)
            s1, c1 = r1['state'], r1['confidence']

            needs_fallback = (
                USE_SONNET_FALLBACK and (
                    (FALLBACK_ON_LOW_CONF and c1 == 'Low') or
                    (FALLBACK_ON_UNKNOWN and s1 == 'Unknown')
                )
            )

            if needs_fallback:
                r2 = _claude_state(entity, sentence, context, lang, SONNET_MODEL)
                r2['model_used'] = 'sonnet_fallback'
                return r2

            r1['model_used'] = 'haiku'
            return r1

        if OLLAMA_OK:
            r = _ollama_state(entity, sentence, context, lang)
            r['model_used'] = 'ollama'
            return r

    except Exception as e:
        return {
            'state': 'Unknown',
            'confidence': 'Low',
            'evidence': f'ERROR: {e}',
            'raw_response': '',
            'model_used': 'error'
        }

    return {
        'state': 'Unknown',
        'confidence': 'Low',
        'evidence': '',
        'raw_response': '',
        'model_used': 'none'
    }

def classify_state_cascade(entity, sentence, context, lang):
    rb = classify_state(entity, sentence, context, lang)

    # only escalate uncertain cases
    if rb['state'] != 'Unknown':
        return {
        'state': rb['state'],
        'confidence': rb['confidence'],
        'evidence': rb['evidence'],
        'raw_response': '',
        'method': 'rule'
        }

    llm_res = classify_state_llm(entity, sentence, context, lang)

    if llm_res.get('model_used') in ['error', 'none']:
        return {
            'state': rb['state'],
            'confidence': rb['confidence'],
            'evidence': f"{rb['evidence']} | rule fallback after LLM {llm_res.get('model_used')}",
            'raw_response': '',
            'method': 'rule_fallback'
        }

    return {
        'state': llm_res['state'],
        'confidence': llm_res['confidence'],
        'evidence': llm_res['evidence'],
        'raw_response': llm_res['raw_response'],
        'method': llm_res['model_used']
    }

print('LLM state classifier + cascade defined.')
n_unk_s = (mentions_df['state_rb']=='Unknown').sum()
print(f'Rule-based Unknown state: {n_unk_s:,}/{len(mentions_df):,} ({n_unk_s/len(mentions_df)*100:.1f}%)')


In [ ]:
# ── Part 4B.2: Exploratory sample inspection ────────────────────────────────
LLM_N_S = 3

if (CLAUDE_OK or OLLAMA_OK):
    parts = []
    for plat in ['ptt','reddit','theqoo']:
        sub = mentions_df[
            (mentions_df['platform'] == plat) &
            (
            (mentions_df['state_rb'] == 'Unknown')
            )
        ]
        if len(sub) > 0:
            parts.append(sub.sample(min(LLM_N_S, len(sub)), random_state=42))

    if parts:
        inspect_sample = pd.concat(parts, ignore_index=True).copy()

        inspect_sample['s_rule'] = inspect_sample['state_rb']

        out_state, out_conf, out_evi, out_method = [], [], [], []

        for _, row in inspect_sample.iterrows():
            res = classify_state_cascade(
                row['entity_canon'], row['sentence'], row['context'], row['lang']
            )
            out_state.append(res['state'])
            out_conf.append(res['confidence'])
            out_evi.append(res['evidence'])
            out_method.append(res['method'])
            time.sleep(0.1)

        inspect_sample['s_cascade'] = out_state
        inspect_sample['s_conf'] = out_conf
        inspect_sample['s_evidence'] = out_evi
        inspect_sample['s_method'] = out_method

        inspect_sample.to_csv(OUT_DIR / 'state_llm_spotcheck_sample.csv', index=False)
        print("Saved state spotcheck.")


In [ ]:
inspect_sample.head()

In [ ]:
# ── Part 4B.3: Apply chosen method to full mentions_df ───────────────────────
print(f'Applying state method: {CHOSEN_STATE}')

if CHOSEN_STATE == 'rule' or not (CLAUDE_OK or OLLAMA_OK):
    mentions_df['state_new'] = mentions_df['state_rb']
    mentions_df['state_rb'] = mentions_df['state_rb']
    mentions_df['state_method'] = 'rule'
    mentions_df['state_confidence'] = mentions_df['state_rb_confidence']
    mentions_df['state_evidence'] = mentions_df['state_rb_evidence']

    print('Using rule-based state results.')
    if CHOSEN_STATE != 'rule':
        print(f"  Requested {CHOSEN_STATE!r} but no LLM available — fell back to rule-based.")

elif CHOSEN_STATE == 'cascade':
    from tqdm.auto import tqdm

    # 固定 row_id，避免 checkpoint 對錯列
    if 'row_id' not in mentions_df.columns:
        mentions_df = mentions_df.reset_index(drop=True).copy()
        mentions_df['row_id'] = np.arange(len(mentions_df))

    CKPT_STATE = OUT_DIR / 'ckpt_state.csv'

    if CKPT_STATE.exists():
        ckpt = pd.read_csv(CKPT_STATE)

        required_ckpt_cols = [
            'row_id',
            'state_cascade',
            'state_method',
            'state_confidence',
            'state_evidence'
        ]
        missing_ckpt = [c for c in required_ckpt_cols if c not in ckpt.columns]
        if missing_ckpt:
            raise ValueError(
                f"Old checkpoint format detected. Missing columns: {missing_ckpt}. "
                f"Delete {CKPT_STATE} and re-run."
            )

        if len(ckpt) > len(mentions_df):
            raise ValueError(
                f"Checkpoint has {len(ckpt)} rows but mentions_df has only {len(mentions_df)} rows. "
                f"Delete {CKPT_STATE} and re-run."
            )

        expected_row_ids = mentions_df.loc[:len(ckpt)-1, 'row_id'].tolist()
        ckpt_row_ids = ckpt['row_id'].tolist()
        if ckpt_row_ids != expected_row_ids:
            raise ValueError(
                "Checkpoint row order does not match current mentions_df. "
                f"Delete {CKPT_STATE} and re-run."
            )

        s_row  = list(ckpt['row_id'])
        s_new  = list(ckpt['state_cascade'])
        s_meth = list(ckpt['state_method'])
        s_conf = list(ckpt['state_confidence'])
        s_evi  = list(ckpt['state_evidence'])
        start = len(s_new)
        print(f'  Resuming from checkpoint ({start:,} done)...')

    else:
        s_row, s_new, s_meth, s_conf, s_evi = [], [], [], [], []
        start = 0

    rows = mentions_df.iterrows()
    print("Total rows:", len(mentions_df))
    print("Unknown rows (expected LLM calls):", (mentions_df['state_rb'] == 'Unknown').sum())

    for i, (_, r) in tqdm(enumerate(rows), total=len(mentions_df)-start, desc='State cascade'):
        if i < start:
            continue
        
        res = classify_state_cascade(
            r['entity_canon'], r['sentence'], r['context'], r['lang']
        )
        s_row.append(r['row_id'])
        s_new.append(res['state'])
        s_meth.append(res['method'])
        s_conf.append(res['confidence'])
        s_evi.append(res['evidence'])
        
        if res['method'] not in ['rule', 'rule_fallback']:
            time.sleep(0.1)

        if len(s_new) % 500 == 0:
            pd.DataFrame({
                'row_id': s_row,
                'state_cascade': s_new,
                'state_method': s_meth,
                'state_confidence': s_conf,
                'state_evidence': s_evi
            }).to_csv(CKPT_STATE, index=False)

    # final checkpoint
    pd.DataFrame({
        'row_id': s_row,
        'state_cascade': s_new,
        'state_method': s_meth,
        'state_confidence': s_conf,
        'state_evidence': s_evi
    }).to_csv(CKPT_STATE, index=False)

    ckpt_final = pd.DataFrame({
        'row_id': s_row,
        'state_cascade': s_new,
        'state_method': s_meth,
        'state_confidence': s_conf,
        'state_evidence': s_evi
    })

    if len(ckpt_final) != len(mentions_df):
        raise RuntimeError(
            f"Cascade length mismatch: {len(ckpt_final)} results vs {len(mentions_df)} rows. "
            "Delete checkpoint and re-run."
        )

    mentions_df = mentions_df.drop(
        columns=['state_cascade', 'state_method', 'state_confidence', 'state_evidence'],
        errors='ignore'
    ).merge(
        ckpt_final,
        on='row_id',
        how='left',
        validate='one_to_one'
    )

    mentions_df['state_new'] = mentions_df['state_cascade']

    rule_n = (mentions_df['state_method'] == 'rule').sum()
    fallback_n = (mentions_df['state_method'] == 'rule_fallback').sum()
    llm_n = mentions_df['state_method'].isin(['haiku', 'sonnet_fallback', 'ollama']).sum()

    print(f'  Rule-based direct: {rule_n:,}')
    print(f'  Rule fallback: {fallback_n:,}')
    print(f'  LLM-resolved: {llm_n:,}')

elif CHOSEN_STATE == 'llm':
    from tqdm.auto import tqdm

    s_new, s_meth, s_conf, s_evi = [], [], [], []

    for _, r in tqdm(mentions_df.iterrows(), total=len(mentions_df), desc='State LLM'):
        res = classify_state_llm(
            r['entity_canon'], r['sentence'], r['context'], r['lang']
        )
        s_new.append(res['state'])
        s_meth.append(res['model_used'])
        s_conf.append(res['confidence'])
        s_evi.append(res['evidence'])

    mentions_df['state_new'] = s_new
    mentions_df['state_llm'] = s_new
    mentions_df['state_method'] = s_meth
    mentions_df['state_confidence'] = s_conf
    mentions_df['state_evidence'] = s_evi

else:
    raise ValueError(f"Unsupported CHOSEN_STATE: {CHOSEN_STATE}")

print(f'  Final: {mentions_df["state_new"].value_counts().to_dict()}')


# ── Method transparency summary ───────────────────────────────────────────────
_actual_state_method = 'rule' if (CHOSEN_STATE == 'rule' or not (CLAUDE_OK or OLLAMA_OK)) else CHOSEN_STATE

print(f"\n{'─'*60}")
print("STATE CLASSIFICATION SUMMARY")
print(f"  Requested method : {CHOSEN_STATE}")
print(f"  Actual method    : {_actual_state_method}")

if 'state_method' in mentions_df.columns:
    _mc = mentions_df['state_method'].value_counts()
    for _m, _n in _mc.items():
        print(f"  {_m:16s}: {_n:,} rows")

print(f"  Final distribution: {mentions_df['state_new'].value_counts().to_dict()}")


Fix rule_fallback with LLM

In [ ]:
import time

mentions_df = pd.read_csv(OUT_DIR / 'mentions_final(state_updated+gender).csv', encoding='utf-8-sig')
mentions_df.head()

target_mask = (mentions_df['state_method'] == 'rule_fallback')

target_df = mentions_df.loc[target_mask, [
    'row_id', 'entity_canon', 'sentence', 'context', 'lang'
]].copy()

target_df['row_id'] = target_df['row_id'].astype(int)

print("Total fallback rows:", len(target_df))

# ── Part 4B.1: LLM state classifier ──────────────────────────────────────────
STATE_SYS = (
    "You are a multilingual NLP classifier for academic research on online ghost stories.\n"
    "Classify the STATE of a character mention.\n\n"

    "Choose EXACTLY ONE label from:\n"
    "Alive, Deceased, Supernatural, Unknown\n\n"

    "Definitions:\n"
    "- Alive: a living human character.\n"
    "- Deceased: a person who has died but is NOT currently appearing as a ghost or interacting with the living.\n"
    "- Supernatural: a ghost, spirit, or dead entity that is actively appearing, interacting, or perceived as a supernatural presence.\n"
    "- Unknown: insufficient or ambiguous evidence.\n\n"

    "Important Rules:\n"
    "1. Classify THIS mention only, based on the sentence and local context.\n"
    "2. If a dead person is described as appearing, interacting, or being seen/heard, classify as Supernatural (NOT Deceased).\n"
    "3. If a dead person is only mentioned as having died (no appearance), classify as Deceased.\n"
    "4. Do NOT infer beyond explicit textual evidence.\n"
    "5. If unclear, choose Unknown.\n"
    "6. Seeing, hearing, or interacting with a dead person implies Supernatural.\n\n"

    "Choose EXACTLY ONE evidence label from:\n"
    "- entity_ghost_term\n"
    "- supernatural_context\n"
    "- death_context\n"
    "- indirect_alive\n"
    "- ambiguous\n"
    "- no_signal\n"
    "- other\n\n"

    "Evidence label guidance:\n"
    "- entity_ghost_term: the mention itself is explicitly a ghost/spirit term.\n"
    "- supernatural_context: the sentence/context shows haunting, appearing, being seen/heard, ghostly interaction, supernatural presence.\n"
    "- death_context: the sentence/context explicitly says the person died/passed away/is dead, without ghostly appearance.\n"
    "- indirect_alive: the context suggests the person is alive through ordinary human actions or disappearance without death/supernatural cues.\n"
    "- ambiguous: evidence is mixed, weak, or unclear.\n"
    "- no_signal: no useful cue is present.\n"
    "- other: some evidence exists but does not fit the above labels well.\n\n"

    "Return EXACTLY in this format:\n"
    "State: <Alive|Deceased|Supernatural|Unknown>\n"
    "Confidence: <High|Medium|Low>\n"
    "EvidenceLabel: <entity_ghost_term|supernatural_context|death_context|indirect_alive|ambiguous|no_signal|other>\n"
)
def parse_state_response(text):
    state = 'Unknown'
    confidence = 'Low'
    evidence = 'other'

    m1 = re.search(r'State:\s*(Alive|Deceased|Supernatural|Unknown)', text, re.I)
    if m1:
        state_map = {
            'alive': 'Alive',
            'deceased': 'Deceased',
            'supernatural': 'Supernatural',
            'unknown': 'Unknown'
        }
        state = state_map.get(m1.group(1).strip().lower(), 'Unknown')

    m2 = re.search(r'Confidence:\s*(High|Medium|Low)', text, re.I)
    if m2:
        conf_map = {
            'high': 'High',
            'medium': 'Medium',
            'low': 'Low'
        }
        confidence = conf_map.get(m2.group(1).strip().lower(), 'Low')

    m3 = re.search(
        r'EvidenceLabel:\s*(entity_ghost_term|supernatural_context|death_context|indirect_alive|ambiguous|no_signal|other)',
        text,
        re.I
    )
    if m3:
        evidence = m3.group(1).strip().lower()

    return state, confidence, evidence

def _claude_state(entity, sentence, context, lang, model_name):
    lang_names = {'zh':'Traditional Chinese','ko':'Korean','en':'English'}

    msg = (
        f"Language: {lang_names.get(lang, lang)}\n"
        f"Character mention: {entity}\n"
        f"Sentence: {sentence}\n"
        f"Local context: {str(context)[:200]}\n\n"
        f"Classify the state of this mention."
    )

    resp = _client.messages.create(
        model=model_name,
        max_tokens=50,
        system=STATE_SYS,
        messages=[{'role':'user','content':msg}]
    )

    raw = resp.content[0].text.strip()
    state, confidence, evidence = parse_state_response(raw)

    return {
        'state': state,
        'confidence': confidence,
        'evidence': evidence,
        'raw_response': raw
    }

def _ollama_state(entity, sentence, context, lang):
    import requests as _req
    lang_names = {'zh':'Traditional Chinese','ko':'Korean','en':'English'}

    prompt = (
        f"{STATE_SYS}\n\n"
        f"Language: {lang_names.get(lang, lang)}\n"
        f"Character mention: {entity}\n"
        f"Sentence: {sentence}\n"
        f"Local context: {str(context)[:200]}\n"
    )

    payload = {'model': OLLAMA_MODEL, 'prompt': prompt, 'stream': False}
    resp = _req.post('http://localhost:11434/api/generate', json=payload, timeout=60)
    raw = resp.json().get('response', '').strip()

    state, confidence, evidence = parse_state_response(raw)

    return {
        'state': state,
        'confidence': confidence,
        'evidence': evidence,
        'raw_response': raw
    }

def classify_state_llm(entity, sentence, context, lang):
    try:
        if CLAUDE_OK:
            r1 = _claude_state(entity, sentence, context, lang, HAIKU_MODEL)
            s1, c1 = r1['state'], r1['confidence']

            needs_fallback = (
                USE_SONNET_FALLBACK and (
                    (FALLBACK_ON_LOW_CONF and c1 == 'Low') or
                    (FALLBACK_ON_UNKNOWN and s1 == 'Unknown')
                )
            )

            if needs_fallback:
                r2 = _claude_state(entity, sentence, context, lang, SONNET_MODEL)
                r2['model_used'] = 'sonnet_fallback'
                return r2

            r1['model_used'] = 'haiku'
            return r1

        if OLLAMA_OK:
            r = _ollama_state(entity, sentence, context, lang)
            r['model_used'] = 'ollama'
            return r

    except Exception as e:
        return {
            'state': 'Unknown',
            'confidence': 'Low',
            'evidence': f'ERROR: {e}',
            'raw_response': '',
            'model_used': 'error'
        }

    return {
        'state': 'Unknown',
        'confidence': 'Low',
        'evidence': '',
        'raw_response': '',
        'model_used': 'none'
    }

In [ ]:
import pandas as pd
from tqdm import tqdm
# 2️ checkpoint 設定
CKPT_FIX = OUT_DIR / 'ckpt_state_fix_fallback_v3.csv'

if CKPT_FIX.exists():
    ckpt = pd.read_csv(CKPT_FIX)

    required_cols = [
        'row_id',
        'state_cascade',
        'state_method',
        'state_confidence',
        'state_evidence'
    ]
    missing = [c for c in required_cols if c not in ckpt.columns]
    if missing:
        raise ValueError(f"Checkpoint missing columns: {missing}")

    ckpt['row_id'] = ckpt['row_id'].astype(int)
    done_ids = set(ckpt['row_id'].tolist())
    results = ckpt.to_dict('records')
    

    print(f"Resuming from checkpoint: {len(done_ids)} rows done")

else:
    done_ids = set()
    results = []

# 3️ 只跑還沒做過的
pending = target_df.loc[~target_df['row_id'].isin(done_ids)]

print("Need LLM calls:", len(pending))

for i, (_, r) in enumerate(
    tqdm(pending.iterrows(), total=len(pending), desc='Fix fallback')
):
    try:
        res = classify_state_llm(
        r['entity_canon'],
        r['sentence'],
        r['context'],
        r['lang']
        )

        results.append({
        'row_id': r['row_id'],
        'state_cascade': res['state'],
        'state_method': res.get('model_used', 'llm'),
        'state_confidence': res.get('confidence', 'Medium'),
        'state_evidence': res.get('evidence', '')
        })
    except Exception as e:
        results.append({
        'row_id': int(r['row_id']),
        'state_cascade': mentions_df.loc[mentions_df['row_id'] == r['row_id'], 'state_cascade'].iloc[0],
        'state_method': 'llm_error',
        'state_confidence': 'Low',
        'state_evidence': f'LLM error: {e}'
    })

    # LLM rate limit
    time.sleep(0.1)

    # ─── checkpoint 每 500 筆 ───
    if (i + 1) % 500 == 0:
        pd.DataFrame(results).drop_duplicates(
            subset=['row_id'], keep='last'
        ).to_csv(CKPT_FIX, index=False)

# final checkpoint
fix_df = pd.DataFrame(results).drop_duplicates(
    subset=['row_id'], keep='last'
)
fix_df.to_csv(CKPT_FIX, index=False)

# 4️ 回填（不動 row 順序）
state_map = dict(zip(fix_df['row_id'], fix_df['state_cascade']))
method_map = dict(zip(fix_df['row_id'], fix_df['state_method']))
conf_map = dict(zip(fix_df['row_id'], fix_df['state_confidence']))
evi_map = dict(zip(fix_df['row_id'], fix_df['state_evidence']))

mask = mentions_df['row_id'].isin(fix_df['row_id'])

mentions_df.loc[mask, 'state_cascade'] = mentions_df.loc[mask, 'row_id'].map(state_map)
mentions_df.loc[mask, 'state_method'] = mentions_df.loc[mask, 'row_id'].map(method_map)
mentions_df.loc[mask, 'state_confidence'] = mentions_df.loc[mask, 'row_id'].map(conf_map)
mentions_df.loc[mask, 'state_evidence'] = mentions_df.loc[mask, 'row_id'].map(evi_map)

# final output
mentions_df['state_new'] = mentions_df['state_cascade']

print("Done fixing fallback.")

print("\nUpdated distribution:")
print(mentions_df['state_new'].value_counts(dropna=False))

print("\nMethod distribution:")
print(mentions_df['state_method'].value_counts(dropna=False))

In [ ]:
mentions_df.to_csv(OUT_DIR / 'mentions_final_checked.csv', index=False, encoding='utf-8-sig')
mentions_df.head()

In [ ]:
check_df = pd.read_csv(
    OUT_DIR / 'mentions_final_checked.csv',
    encoding='utf-8-sig'
)

print("Gender distribution:")
print(check_df['gender_new'].value_counts(dropna=False))
print()

print("Gender method distribution:")
print(check_df['gender_method'].value_counts(dropna=False))
print()

print("State distribution:")
print(check_df['state_new'].value_counts(dropna=False))
print()

print("State method distribution:")
print(check_df['state_method'].value_counts(dropna=False))

## Part 5 · Affect Score Computation

Multilingual positive, negative, and fear lexicons are used to compute
sentence-level affect scores for each character mention. Scores are
normalised by context length (words per sentence).

These scores are used in Part 8 (Sentiment Analysis).

In [ ]:
mentions_df = pd.read_csv(OUT_DIR / 'mentions_final_checked.csv', encoding='utf-8-sig')
mentions_df.head()

# ── Part 5.1: Multilingual affect lexicons ───────────────────────────────────
# Three dimensions: positive · negative · revenge
# Domain-specific for supernatural / ghost-story narratives.

AFFECT_LEXICON = {
    'zh': {
        'positive': [
            '溫柔','善良','慈祥','保護','保佑','關心','愛','溫暖','感謝',
            '和平','美麗','幸福','歡喜','平靜','慈愛','祥和','感動',
            '守護','思念','牽掛','不捨','眷戀','憐愛','呵護','陪伴',
            '安祥','寧靜','安定','撫慰','治癒','希望','幫助','幫忙',
            # NTUSD/domain: spiritual resolution vocabulary
            '平安','解脫','超度','安息','懷念','告別','寬恕','救贖','庇佑','安慰',
            '保庇','超渡','超生','冥福','善終','安詳','安穩','安樂','祈福',
            # Ghost-story closure and ancestral blessing terms
            '化解','釋懷','圓滿','成全','安心','安魂','安然','無憾','慈悲','庇護',
            '庇蔭','引路','指引','歸途','團圓','悼念','追思','靈驗','護佑',
            '顯靈','託夢','顯現','庇護靈','守靈','靈魂安息','功德圓滿','渡化',
            '引渡','轉世','再見一面','陰德','積德','善報'
        ],
        'negative': [
            '怨恨','憤怒','恐怖','兇惡','邪惡','詭異','可怕','恐懼',
            '驚嚇','詛咒','惡毒','憎恨','痛苦','哭泣','哀嚎','死亡',
            '腐爛','血腥','恐慌','絕望','悲痛','凶殘','殘忍','折磨',
            '陰森','陰暗','驚悚','詭譎','駭人',
            # NTUSD ntusd-negative.txt (ghost-relevant subset)
            '凶險','血淋淋','惶恐','慘死','慘叫','嚇人','心寒','毀滅',
            '令人發指','猙獰','詭秘','兇殘','自殺','跳樓','殺人','暴力','虐待','殺害',
            # Fear-heavy vocabulary intentionally merged into negative
            '害怕','恐懼','顫抖','嚇到','驚恐','汗毛','心跳',
            '冷汗','毛骨悚然','嚇壞','戰慄','膽顫','心驚','驚醒','驚魂未定',
            '腿軟','魂飛魄散','驚魂','膽戰','不寒而慄','皮皮剉','令人不安',
            # NTUSD + PTT ghost idioms
            '大驚失色','面如土色','嚇破膽','頭皮發麻','雞皮疙瘩',
            '全身發抖','汗如雨下','瑟瑟發抖','嚇得說不出話',
            '嚇得要死','嚇得半死','嚇得屁滾尿流','臉色發白','臉色蒼白','臉色發青',
            # Additional horror descriptors
            '陰冷','寒意','煞氣','陰風陣陣','淒厲','怵目驚心','死寂','驚駭','驚惶',
            '駭然','失魂落魄','魂不附體','驚惶失措','怵然','悚然',
            '陰煞','邪氣','煞氣沖天','怨氣','怨念深重','陰氣重','索魂聲','慘白',
            '面目猙獰','血肉模糊','氣氛詭譎','令人窒息',
        ],
        'revenge': [
            '報復','復仇','索命','索債','討債','仇恨','怨靈',
            '冤魂','報仇','冤枉','冤屈','申冤','伸冤','黑令旗',
            '詛咒','作祟','糾纏','不放過','找上門','討命',
            # Domain-specific ghost revenge
            '死不瞑目','冤魂索命','怨氣沖天','含恨而終','不得超生','含恨而死',
            '索命索債','討命討債','冤魂不散','陰魂不散',
            # Additional grudge and karmic retaliation terms
            '索魂','勾魂','追命','追魂','討血債','血債血還','冤有頭債有主','怨念','執念','報應不爽',
            '以牙還牙','清算','清償','討回公道','討回一命','含冤莫白','怨懟','怨毒','不肯罷休',
            '報應','天譴','來索命','血債','討命鬼','陰司索命','報仇雪恨','冤親債主','討報','死後報復',
            '回來算帳'
        ],
    },
    'ko': {
        'positive': [
            '따뜻','사랑','기쁨','행복','감사','자상','친절','보호',
            '평화','온화','인자','다정','포근','고마움','위로',
            '그리움','보고싶','지켜','함께','안심','기억','추억',
            # SenticNet #joy — ghost narrative resolution
            '평온','위안','구원','해방','희망','용서','치유',
            # Ritual/rest vocabulary in Korean ghost narratives
            '명복','천도','성불','안식','평안','포용','화해','다독임','안도','보살핌',
            '따스함','돌봄','배려','감싸다','편히 쉬','해원','한을 풀','떠나보내다','잘 가','평화롭다',
            # Additional Korean afterlife comfort terms
            '가호','보은','명복을 빌','편안히 잠들','편히 가','달래다','안정을 찾','극락왕생','여한이 없','성스럽다',
            '성불하다','좋은 곳으로 가다','편히 눈을 감다','극락으로 가다','영혼이 쉬다','해탈하다','좋은 인연'
        ],
        'negative': [
            '두려움','공포','무서움','불안','원한','복수','증오','악몽',
            '비명','죽음','피','저주','섬뜩','끔찍','소름','무섭',
            '음산','기괴','혐오','분노','고통','절망','오싹','처참',
            # SenticNet #anger/#disgust — ghost horror context
            '공포스러운','불길한','무시무시한','기이한','기묘한','흉물스러운','끔찍스러운','소름끼치는',
            # Fear-heavy vocabulary intentionally merged into negative
            '무서워','겁나','떨림','심장','식은땀','놀람',
            '무섭','겁먹','전율','오싹',
            # SenticNet #fear + colloquial variants
            '벌벌','후덜덜','덜덜','진땀','손발이 떨','심장 두근',
            # Additional Korean horror descriptors
            '섬찟한','음침한','흉흉한','살벌한','살기','핏빛','괴기한','잔혹한','악랄한','소름돋는',
            '등골이 서늘한','한기가 도는','불안감','숨막히는','소름이 돋','기절초풍','넋이 나간','혼비백산','죽을 만큼 무서운','끔찍한 장면',
            # Additional haunted-house / apparition cues
            '귀곡성','흉가','귀신들린','원령','핏자국','살점','썩은내','울부짖는','광기','사늘한',
            '서늘해지다','핏빛 눈','비릿한 냄새','숨이 턱 막히는','악귀 같은','기척이 느껴지는','죽음의 그림자',
            '미친 듯한','정신이 아찔한','소름이 끼치는','귀기','음기','살기','핏기 없는 얼굴','창백한 얼굴',
            '섬뜩한 기운','이상한 기척','정체불명의 존재','숨죽인 공기'
        ],
        'revenge': [
            '복수','원한','원수','앙갚음','보복','저주','원귀',
            '한풀이','원혼','억울','누명','찾아온',
            '용서못','잊지않','되갚',
            # Additional revenge/grudge terms
            '앙심','원통','한','원통하다','한이 맺힌',
            '원한을 품','복수심','피의 복수','복수귀','되갚아주','저승까지 쫓','한을 풀어야','억울한 죽음','한 맺힌 혼','용납 못해',
            # Additional ghost-retaliation expressions
            '앙복','복수하러 오다','원귀가 되다','해코지','끝까지 쫓다','한을 갚다','피값을 치르다','복수의 화신','원령의 복수','대가를 치르게 하다',
            '한을 품다','한이 풀리지 않다',
            '죽어서도 따라다니다',
            '끝까지 괴롭히다',
            '원한을 갚다',
            '복수귀가 되다',
            '저주를 내리다'
        ],
    },
    'en': {
        'positive': [
            'gentle','kind','warm','peaceful','calm','protective','loving',
            'tender','caring','comforting','serene','graceful','sweet',
            'benevolent','helpful','guardian','longing','miss','remember',
            'cherish','protect','watch over',
            # Ghost narrative closure vocabulary
            'redemption','closure','at peace','blessed','forgiven','healed',
            # Additional afterlife-resolution vocabulary
            'rest in peace','requiem','deliverance','release','reconciliation','mercy','compassion','safe passage','lay to rest','ascend',
            'guardian spirit','watchful','solace','consoled','peace granted','absolved','atoned','farewell','goodbye','final rest',
            # Additional ghost-story reconciliation vocabulary
            'spirit at rest','eternal rest','guided home','cross over','passed on','light a candle',
            'prayer for the dead','blessing','last rites','restful','quiet grace','gentle farewell',
            'peaceful passing','soul released','memory lives on','in loving memory','laid to rest',
            'spiritual comfort','divine mercy','safe in the afterlife',
            'spirit visitation','came in a dream,peaceful apparition',
            'guardian presence','protective spirit','final goodbye','closure through apparition'
        ],
        'negative': [
            'evil','dark','vengeful','angry','terrifying','malevolent',
            'wicked','sinister','threatening','cursed','violent','deadly',
            'horrifying','screaming','blood','crying','weeping','menacing',
            'hate','rage','cruel','sadistic','ominous','disturbing',
            'grotesque','ghastly','eerie','macabre','monstrous',
            # Horror-specific (not already present)
            'nightmarish','diabolical','demonic','hellish','unholy','abominable','vile','bloodthirsty','accursed','putrid',
            # Fear-heavy vocabulary intentionally merged into negative
            'scared','afraid','frightened','terrified','horrified',
            'trembling','shaking','panic','paranoid','anxious','startled',
            'chilled','frozen','dread','nightmare','apprehensive',
            'heart pounding','skin crawl','goosebumps',
            # Colloquial Reddit horror register
            'petrified','paralyzed','cold sweat','spine-tingling','bone-chilling','freaked out','creeped out','spooked',
            # Additional ghost-horror descriptors
            'haunted','possessed','malignant','grisly','ghoulish','rotting','howling','wailing','dreadful','charnel',
            'unnerving','hair-raising','stomach-turning','disfigured','cadaverous','spectral','phantasmal','deathly','terror-stricken','doom-laden',
            # Additional haunting/encounter descriptors
            'ominous silence','haunting whisper','sudden chill','foul stench','cursed place','evil presence',
            'restless dead','haunted house','grave terror','flesh-ripping','blood-soaked','soul-crushing',
            'spirit attack','possession-like','fear-stricken','death omen','lurking presence','unseen watcher',
            'whispering voice','cold breath','presence behind me','something was there','unexplainable fear',
            'staring from the dark'
        ],
        'revenge': [
            'revenge','vengeance','avenge','retaliate','payback','get back',
            'haunt','curse','torment','punish','unfinished business',
            'wronged','injustice','unfair','betrayed','no peace',
            'justice','retribution','wrath','fury','cannot rest',
            # Additional revenge resolution vocabulary
            'vindication','settle scores','score to settle','unresolved','restless soul','bound to this world',
            'grudge','vendetta','blood debt','oath of vengeance','hunt down','come back for','hex','damnation','pay in blood','even the score',
            'an eye for an eye','no forgiveness','wrong must be repaid','spirit of vengeance','sworn revenge','justice from beyond','return for vengeance','cannot move on','settle the debt','vengeful spirit',
            # Additional ghost-retaliation expressions
            'haunt for revenge','back from the grave','vengeance from beyond','pay for what you did',
            'make them pay','hounded by a spirit','marked for vengeance','cursed to suffer','debt of blood',
            'torment without end','restless for revenge','settle old wrongs','punishment from the dead',
            'ghostly retribution','relentless revenge','sworn to haunt','vengeance fulfilled','repaid in death',
            'retribution from beyond the grave','spirit returns for justice','came back for revenge',
            'seeking justice from beyond','haunting with purpose','unfinished revenge','bound by hatred',
            'returning to punish','trapped in vengeance','death did not end it'
        ],
    },
}

# I4: negation tokens — matches within a look-back window cancel affect
NEGATION_TOKENS = {
    'zh': ['不','沒有','並不','未','不是','沒','無','非','毫無',
           '別','從不','絕不','並無','全無','從來不','不曾','根本不','從未','哪有','絕對不'],
    'ko': ['안','아니','없','못','불','비','아니야','아닌','아니에요',
           '없다','못하다','아니다','절대','전혀','결코','아예','하나도'],
    'en': ['not', "n't", 'never', 'no', 'without', 'lack', 'hardly', 'barely',
           'neither', 'nor', 'nowhere', 'nobody', 'nothing', 'none', 'scarcely', 'free from'],
}

def _is_negated(text, pos, lang_key, window=4):
    """Return True if a negation token appears within `window` chars/tokens before pos."""
    neg_toks = NEGATION_TOKENS.get(lang_key, [])
    if lang_key == 'en':
        pre = text[:pos].split()
        return any(n in pre[-window:] for n in neg_toks)
    else:
        pre = text[max(0, pos-window):pos]
        return any(n in pre for n in neg_toks)

def compute_affect_score(context, lang, affect_type):
    if pd.isna(context) or str(context).strip() == '':
        return 0.0
    lk   = get_lang_key(lang)
    text = str(context).lower()
    words = AFFECT_LEXICON.get(lk, {}).get(affect_type, [])
    count = 0
    for w in words:
        wl = w.lower()
        start = 0
        while True:
            pos = text.find(wl, start)
            if pos == -1:
                break
            if not _is_negated(text, pos, lk):  # I4: skip negated hits
                count += 1
            start = pos + 1
    length = max(len(text), 1)
    return round(count / length * 100, 4)  # per 100 chars

# Apply all three dimensions
print("Computing affect scores for all mentions...")
for atype in ['positive', 'negative', 'revenge']:
    col = f'val_{atype[:3]}'
    mentions_df[col] = mentions_df.apply(
        lambda r, a=atype: compute_affect_score(r['context'], r['lang'], a), axis=1
    )
mentions_df.rename(columns={'val_rev': 'val_revenge'}, inplace=True, errors='ignore')

print("Affect score summary (per 100 chars):")
aff_cols = [c for c in ['val_pos','val_neg','val_revenge'] if c in mentions_df.columns]
print(mentions_df[aff_cols].describe().round(4))

In [ ]:
mentions_df.head()

## Part 6 · Ghost Entity Counts

**Integrity check**: downstream analyses use **only** `gender_new` (Part 3)
and `state_new` (Part 4) — both derived fresh from raw article text.
No pre-existing entity CSV columns enter the analysis.

In [ ]:
# ── Part 6.0: Pipeline integrity check ───────────────────────────────────────
assert 'gender_new' in mentions_df.columns, 'Run Part 3 first'
assert 'state_new'  in mentions_df.columns, 'Run Part 4 first'
assert 'val_pos'    in mentions_df.columns, 'Run Part 5 first'

# Confirm no pre-existing label columns are used for analysis
FORBIDDEN_COLS = ['gender','state','is_supernatural']
for col in FORBIDDEN_COLS:
    if col in mentions_df.columns:
        print(f"  WARNING: column '{col}' present — it is from entity CSV and must NOT be used for analysis")
    else:
        print(f"  ✓ '{col}' not in mentions_df — pipeline is independent")

print()
print("Pipeline columns confirmed:")
print("  gender_new → conservative rule-based classifier (Part 3)")
print("  state_new  → hierarchical keyword classifier (Part 4)")
print("  val_pos / val_neg / val_revenge → lexicon-computed affect scores (Part 5)")

In [ ]:
# ── Part 6.1: Filter supernatural entities (ghosts) ─────────────────────────
# 'ghosts' includes all supernatural entities (Female / Male / Unknown)
# 'ghosts_known' is Female + Male only (for gendered comparison)

ghosts       = mentions_df[mentions_df['state_new'] == 'Supernatural'].copy()
ghosts_known = ghosts[ghosts['gender_new'].isin(['Female','Male'])].copy()

print(f"Supernatural entity mentions (all genders):  {len(ghosts):,}")
print(f"Supernatural entity mentions (known gender): {len(ghosts_known):,}")
print()
print("Gender breakdown of ghost entities:")
print(ghosts['gender_new'].value_counts())
print()
print("Per platform:")
print(ghosts.groupby(['platform','gender_new']).size().unstack(fill_value=0))

In [ ]:
# ── Part 6.2: Ghost count visualisation — Mention-level & Article-level ────────

palette      = {'Female': '#E07B7B', 'Male': '#6B9EC7', 'Unknown': '#C0C0C0'}
gender_order = ['Female', 'Male', 'Unknown']
platform_labels = {'ptt': 'PTT (tw)', 'reddit': 'Reddit (en)', 'theqoo': 'Theqoo (ko)'}

def make_counts(df, dedup_cols=None):
    """Build platform × gender count + pct table. Optionally deduplicate first."""
    src = df.drop_duplicates(subset=dedup_cols) if dedup_cols else df
    gc = (src.groupby(['platform','gender_new']).size()
            .reset_index(name='count'))
    gc['total'] = gc.groupby('platform')['count'].transform('sum')
    gc['pct']   = gc['count'] / gc['total'] * 100
    return gc

ghost_counts_mention = make_counts(ghosts)
ghost_counts_article = make_counts(ghosts, dedup_cols=['article_id','gender_new'])

fig, axes = plt.subplots(2, 3, figsize=(13, 9), sharey=False)
titles = ['Mention-level', 'Article-level (robustness)']
for row_i, (gc, title_prefix) in enumerate(
        [(ghost_counts_mention, 'Mention-level'),
         (ghost_counts_article, 'Article-level')]):
    for ax, plat in zip(axes[row_i], ['ptt','reddit','theqoo']):
        sub = gc[gc['platform'] == plat].set_index('gender_new')                .reindex(gender_order, fill_value=0)
        sub['pct'] = sub['pct'].fillna(0)
        bars = ax.bar(sub.index, sub['pct'],
                      color=[palette[g] for g in sub.index],
                      edgecolor='white', linewidth=0.5)
        for bar, (_, row) in zip(bars, sub.iterrows()):
            if row['pct'] > 0:
                ax.text(bar.get_x() + bar.get_width()/2,
                        bar.get_height() + 1,
                        f"{row['pct']:.1f}%\n(n={int(row['count'])})",
                        ha='center', va='bottom', fontsize=7.5)
        ax.set_title(f"{platform_labels[plat]}\n({title_prefix})", fontsize=10, fontweight='bold')
        ax.set_ylabel('% of Ghost Entities')
        ax.set_ylim(0, max(sub['pct'].max() * 1.3, 60))
        ax.grid(axis='y', alpha=0.3, linestyle='--')

plt.suptitle('Ghost Entity Gender Distribution Across Platforms\n'
             'Top: mention-level  |  Bottom: article-level (each article counted once)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(OUT_DIR / 'ghost_counts_raw_pipeline.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: ghost_counts_raw_pipeline.png")
ghost_counts_mention.to_csv(OUT_DIR / 'ghost_counts_mention.csv', index=False)
ghost_counts_article.to_csv(OUT_DIR / 'ghost_counts_article.csv', index=False)
print(f"Saved: {OUT_DIR}/ghost_counts_mention.csv, ghost_counts_article.csv")


In [ ]:
# ── Part 6.3: Chi-square + Cramér's V  (mention-level AND article-level) ──────

def run_chisq(df, label, dedup_cols=None):
    src = df.drop_duplicates(subset=dedup_cols) if dedup_cols else df
    ct = (src.groupby(["platform","gender_new"])
            .size().unstack(fill_value=0)
            .reindex(columns=["Female","Male"], fill_value=0))
    chi2, p, dof, expected = chi2_contingency(ct.values)
    N = ct.values.sum()
    v = np.sqrt(chi2 / (N * (min(ct.shape) - 1)))
    effect = "small" if v < 0.3 else ("medium" if v < 0.5 else "large")
    print(f"=== {label} ===")
    print("Observed counts:")
    print(ct)
    exp_df = pd.DataFrame(expected, index=ct.index, columns=ct.columns)
    print(f"Expected counts:\n{exp_df.round(1)}")
    if np.any(expected < 5):
        print(f"    Expected frequency < 5 in {int(np.sum(expected < 5))} cell(s) — chi-square may be unreliable.")
    print(f"chi2={chi2:.2f}  p={p:.5f}  df={dof}  Cramér\'s V={v:.3f}  [{effect}]")
    print()
    return ct, chi2, p, v

ct_m, chi2_m, p_m, v_m = run_chisq(ghosts_known, "Mention-level  (all mentions)")
ct_a, chi2_a, p_a, v_a = run_chisq(ghosts_known, "Article-level  (each article counted once)",
                                    dedup_cols=["article_id","gender_new"])

# Robustness summary
agree = (p_m < 0.05) == (p_a < 0.05)
print("─" * 50)
print(f"Significance agreement (both sides of α=0.05): {'✓ CONSISTENT' if agree else '✗ DIVERGE — investigate'}")
if agree:
    print("→ Results not an artefact of repeated mentions within articles.")


## Part 7 · Collocational Semantic Analysis (LLR)

For each supernatural entity mention, a ±5-word sliding window of content words
is extracted from the containing sentence. Words are POS-filtered by language:
- **Chinese (PTT)**: `jieba` POS tagging; keep n/v/a/d (noun/verb/adj/adv)
- **English (Reddit)**: `spaCy` POS tagging; keep NOUN/VERB/ADJ/ADV
- **Korean (Theqoo)**: `kiwipiepy` morphological analysis; keep NNG/NNP/VA/VV/MAG

Association strength is measured by **Log-Likelihood Ratio** (Dunning, 1993).

In [ ]:
# ── Part 7.1: Load NLP tools ─────────────────────────────────────────────────
JIEBA_OK = False
SPACY_OK  = False
KIWI_OK   = False

try:
    import jieba
    import jieba.posseg as pseg
    JIEBA_OK = True
    print("jieba loaded ✓")
except ImportError:
    print("jieba not available — Chinese collocates will use regex fallback")

try:
    import spacy
    nlp_en = spacy.load('en_core_web_sm')
    SPACY_OK = True
    print("spaCy (en_core_web_sm) loaded ✓")
except Exception:
    print("spaCy not available — English collocates will use regex fallback")

try:
    from kiwipiepy import Kiwi
    kiwi_model = Kiwi()
    KIWI_OK = True
    print("kiwipiepy loaded ✓")
except ImportError:
    print("kiwipiepy not available — Korean collocates will use regex fallback")

# ── Apply custom vocabulary ───────────────────────────────────────────────────
if JIEBA_OK:
    zh_vocab = sorted(set(JIEBA_CUSTOM_WORDS + HAUNTED_LOCATION_KWS.get('zh', [])), key=len, reverse=True)
    for w in zh_vocab:
        jieba.add_word(w)
    print(f"  jieba: {len(zh_vocab)} custom words registered")

if SPACY_OK:
    en_phrases = sorted(set(SPACY_PROTECTED_PHRASES + HAUNTED_LOCATION_KWS.get('en', [])), key=len, reverse=True)
    for phrase in en_phrases:
        nlp_en.tokenizer.add_special_case(phrase, [{"ORTH": phrase}])
    print(f"  spaCy: {len(en_phrases)} protected phrases registered")

if KIWI_OK:
    ko_vocab = sorted(
        set(HAUNTED_LOCATION_KWS.get('ko', [])
            + list(KOREAN_CHOSUNG_MAP.values())
            + KIWI_CUSTOM_WORDS),
        key=len,
        reverse=True
    )
    for w in ko_vocab:
        kiwi_model.add_user_word(w)
    print(f"  kiwipiepy: {len(ko_vocab)} custom words registered")

# ── Text normalization hook for custom dictionaries ───────────────────────────
def normalize_custom_text(text, lang):
    """Normalize abbreviations/slang before POS tokenization."""
    import re

    t = str(text)
    lk = get_lang_key(lang)

    if lk == 'ko':
        for raw, expanded in KOREAN_CHOSUNG_MAP.items():
            t = t.replace(raw, expanded)

    if lk == 'en':
        for abbr, expanded in AFFECT_ABBREVS.items():
            # Whole-token replacement to avoid changing substrings inside words.
            t = re.sub(rf'(?<!\\w){re.escape(abbr)}(?!\\w)', expanded, t, flags=re.IGNORECASE)

    return t

In [ ]:
# ── Part 7.2b: Gendered reference / pronoun stoplist for LLR ────────────────
# Goal: remove gender labels, kinship terms, pronouns, and direct gendered references
# so LLR focuses more on semantic collocates rather than obvious referential words.

GENDER_STOPLIST = {
    'zh': {
        # direct gender labels / person refs
        '女', '女生', '女性', '女子', '女人', '女孩', '小姐', '太太', '老婆', '妻子', '新娘',
        '男', '男生', '男性', '男子', '男人', '男孩', '先生', '老公', '丈夫', '新郎','小女孩','小男孩',
        '小姊姊','小哥哥','小姐姐','小哥哥','大姊姊','大哥哥','少女','少年','女孩子','男孩子','女學生','男學生',
        '女同學','男同學','大姐','婦人','女士','女主人','男主人','女伴','男伴','老婦','小妹妹','小弟弟','小男生','小女生','兄弟',
        '老先生','老頭',

        # kinship / family role
        '媽媽', '母親', '阿媽', '阿嬤', '外婆', '奶奶', '姐姐', '姊姊', '妹妹', '阿姨', '姑媽', '姑姑','家母',
        '爸爸', '父親', '阿公', '爺爺', '外公', '哥哥', '弟弟', '叔叔', '伯伯', '舅舅','阿伯','老爸','老媽',
        '男友', '女友','小弟','小妹','男朋友','女朋友','男伴','女伴','學弟','學妹','同學','學長','學弟妹','學姊','學姐','學長姐','學長姊',
        '女兒', '兒子','婆婆','公公','岳母','岳父','老婆婆','老公公','姥姥','姥爺','外祖母','外祖父','表姐','表哥','堂姐','堂哥',
        '姑婆','姑公','舅婆','舅公','姨媽','姨公','叔婆','叔公','伯婆','伯公','曾祖母','曾祖父','玄祖母','玄祖父', '外曾祖母','外曾祖父',
        '小姨','小姨公','小叔公','小叔婆','小伯公','小伯婆','大嫂','小姑姑','大姑姑','舅媽','老伯伯','二老','老伯',
        '孫女','孫子','外孫女','外孫子','外孫', '曾孫女','曾孫子','玄孫女','玄孫子','祖母','祖父','妹','弟','姊','姐','兄','叔','伯','姨','姑','舅',
        '嬸','哥','嫂','婿','媳婦','兒媳','女婿','公婆','外公外婆','外祖父母','表哥表姐','堂哥堂姐','堂舅','堂姨媽','表舅','表姨媽','表姨','堂姨',
        '老母','老父','老爹','老娘','阿爸','阿母','阿爹','阿娘','爹地','媽咪','爸比','前夫','前妻','繼父','繼母','繼兄弟','繼姊妹',
        '前男友', '前女友','前男朋友','前女朋友','弟媳','妹婿','表姊','表兄弟','堂姊','堂兄','大哥',

        # pronouns / referential forms
        '她', '他', '她們', '他們', '妳', '你', '妳們', '你們',

        # name
        '小明', '小紅', '小華', '小美', '小麗', '小偉', '小芳', '小軍', '小英','小芬',
        '小剛','小杰','小婷','布萊恩','小橘','阿瑞','阿雪','小甫','阿賢', '阿諾', '艾克', '小潔', 
        '阿狗', '莎莎', '阿程', '毛仔', '老王','小王', '小李', '小張', '小陳', '小林', '小楊', '小吳', 
        '小趙', '小周', '小徐', '小孫', '小馬', '小朱', '小胡', '小郭', '小何', '小高', '小林', '雅雅',
        '阿耀', '阿民','張老師','花仔',

    },

    'en': {
        # direct gender labels
        'woman', 'female', 'girl', 'lady', 'ladies', 'wife', 'bride',
        'man', 'male', 'boy', 'guy', 'gentleman', 'husband', 'groom',
        

        # kinship / family role
        'mom', 'mum', 'mother', 'mommy', 'mummy','nani','auntie','grampa',
        'grandma', 'grandmother', 'sister', 'aunt', 'daughter','niece','cousin','auntie','ma','mama','mamma',
        'dad', 'father', 'daddy', 'grandpa', 'grandfather', 'brother', 'uncle', 'son','nan','stepmom','stepdad',
        'stepmother','stepfather', 'stepsister','stepbrother','ex-husband','ex-wife',

        # pronouns
        'she', 'her', 'hers', 'herself',
        'he', 'him', 'his', 'himself',

        # direct gendered ghost phrases if lemmatized into tokens
        'girlfriend', 'boyfriend',

        # name
        'elsa', 'anna', 'olaf', 'belle', 'cinderella', 'snow white', 'rapunzel', 'ariel',
        'zelda','claire','sarah','jessica','emily','lisa','linda','susan','karen','nancy','betty',
        'jennifer','laura','sarah','kimberly','molloy','ann', 'john','michael','david','james',
        'robert','charles','richard','steven','thomas','rachel','lisa','susan','kim','sally',
        'dennis','scott','jacob','tom','george','kevin','brian','edward','ronald','anthony','donald',
        'stephen','paul','andrew','ian','kyle','jimmy','larry','mike','tony','ryan',

    },

    'ko': {
        # direct gender labels
        '여자', '여성', '소녀', '숙녀',
        '남자', '남성', '소년','여자애','남자애','여학생','남학생',

        # kinship / family role
        '엄마', '어머니', '할머니', '외할머니', '언니', '누나', '여동생', '아내',
        '아빠', '아버지', '할아버지', '외할아버지', '오빠', '형', '남동생', '남편',
        '아주머니', '아줌마', '아저씨','외삼촌','삼촌','이모','고모','조카','사촌',
        '형님','누님','동생','선배','후배','할매','할배','아부지','어무니','할머님','할아버님',
        '형수','제수씨','처남','처형','외가',

        # pronouns / referential
        '그녀', '그', '그녀들', '그들',

        #fillers / common names
        '응응','어어','음음','아아','이이','오오','우우','으으','에에','저기요','여보세요',
    }
}
GENDER_STOPLIST_NORM = {
    lk: {str(x).strip().lower() for x in vals}
    for lk, vals in GENDER_STOPLIST.items()
}

In [ ]:
import re

# Regex for Chinese kinship variants:
# (optional pronoun prefix + optional 家) + (optional ordinal prefix) + kinship term + (optional 家)
_ZH_KINSHIP_PATTERN = re.compile(
    r'^(?:(?:我|你|他|她|我的|你的|他的|她的)(?:家)?)?'
    r'(?:大|二|三|四|五|六|七|八|九|十|十一|十二|十三|十四|十五)?'
    r'(?:媽媽|母親|爸爸|父親|老媽|老爸|阿媽|阿嬤|奶奶|爺爺|外婆|外公|祖母|祖父|姥姥|姥爺|'
    r'姐姐|姊姊|哥哥|弟弟|妹妹|姊|姐|兄|弟|妹|'
    r'阿姨|姨媽|姨|姑媽|姑姑|姑|叔叔|叔|伯伯|伯|舅舅|舅|嬸|嫂|嫂子|姐夫|妹夫|'
    r'婆婆|公公|岳母|岳父|妻子|丈夫|老婆|老公|兒子|女兒|孫子|孫女|外孫|外孫子|外孫女|'
    r'叔公|叔婆|伯公|伯婆|舅公|舅婆|姑公|姑婆|曾祖父|曾祖母|外曾祖父|外曾祖母)'
    r'(?:家)?$'
)

def is_gender_reference_word(word, lang):
    """
    Return True if a token is a gendered reference / pronoun / kinship term
    that should be excluded from LLR collocate analysis.
    """
    lk = get_lang_key(lang)
    if pd.isna(word):
        return True

    w = str(word).strip().lower()
    if not w:
        return True

    stopset = GENDER_STOPLIST_NORM.get(lk, set())
    if w in stopset:
        return True

    # Chinese kinship variant protection via regex
    if lk == 'zh' and _ZH_KINSHIP_PATTERN.match(str(word).strip()):
        return True

    return False

In [ ]:
# ── Part 7.2: POS-based content word extraction ──────────────────────────────
# Uses KEEP_TAGS_ZH, KEEP_TAGS_KO, KEEP_POS_EN, STOPWORDS_* from Part 2.1

def extract_content_words(text, lang, extra_stopwords=None):
    lk = get_lang_key(lang)
    if pd.isna(text) or not str(text).strip():
        return []

    text_norm = normalize_custom_text(text, lang)
    extra_stopwords = set() if extra_stopwords is None else {str(x).strip().lower() for x in extra_stopwords}

    if lk == 'zh':
        if JIEBA_OK:
            import jieba.posseg as pseg
            pairs = pseg.cut(str(text_norm))
            words = []
            for w, flag in pairs:
                w_clean = str(w).strip()
                if (
                    flag in KEEP_TAGS_ZH
                    and w_clean not in STOPWORDS_ZH
                    and w_clean.lower() not in extra_stopwords
                    and len(w_clean) > 1
                ):
                    words.append(w_clean)
        else:
            import re
            t = re.sub(r'[\s\W]', '', str(text_norm))
            words = [t[i:i+2] for i in range(len(t)-1)
                     if t[i:i+2].lower() not in extra_stopwords]
        return words

    if lk == 'ko':
        if KIWI_OK:
            result = kiwi_model.analyze(str(text_norm))[0][0]
            words = []
            for tok in result:
                form = str(tok.form).strip()
                if (
                    tok.tag in KEEP_TAGS_KO
                    and form not in STOPWORDS_KO
                    and form.lower() not in extra_stopwords
                    and len(form) > 1
                ):
                    words.append(form)
        else:
            words = [w for w in str(text_norm).split()
                     if w not in STOPWORDS_KO and w.lower() not in extra_stopwords]
        return words

    if lk == 'en':
        if SPACY_OK:
            doc = nlp_en(str(text_norm))
            words = []
            for tok in doc:
                lemma = tok.lemma_.lower().strip()
                if (
                    tok.pos_ in KEEP_POS_EN
                    and not tok.is_stop
                    and lemma not in STOPWORDS_EN
                    and lemma not in extra_stopwords
                    and len(lemma) > 2
                ):
                    words.append(lemma)
        else:
            import re
            words = [w.lower() for w in re.findall(r'[a-z]+', str(text_norm).lower())
                     if w not in STOPWORDS_EN and w not in extra_stopwords and len(w) > 2]
        return words

    return []

In [ ]:
# Build corpus of content words per mention
print("Extracting content words per mention...")

def extract_and_filter(r):
    words = extract_content_words(
        r.get('context', r.get('sentence', '')),
        r['lang'],
        extra_stopwords={str(x).strip().lower() for x in GENDER_STOPLIST.get(get_lang_key(r['lang']), set())}
    )
    return [w for w in words if not is_gender_reference_word(w, r['lang'])]

mentions_df['content_words'] = mentions_df.apply(extract_and_filter, axis=1)

avg_len = mentions_df['content_words'].apply(len).mean()
print(f"  avg content words per mention: {avg_len:.1f}")
print(f"  non-empty: {(mentions_df['content_words'].apply(len) > 0).sum()} / {len(mentions_df)}")

In [ ]:
mentions_df.head()
mentions_df.to_csv(OUT_DIR / 'mentions_with_content_words.csv', index=False)
print(f"Saved: {OUT_DIR}/mentions_with_content_words.csv")  

In [ ]:
from scipy.stats import chi2
from statsmodels.stats.multitest import multipletests
# ── Part 7.3: LLR computation ─────────────────────────────────────────────────

def compute_llr(n11, n12, n21, n22):
    """
    Dunning (1993) Log-Likelihood Ratio.
    n11: word in female ctx; n12: word in male ctx
    n21: other words in female ctx; n22: other words in male ctx
    Returns signed G²: positive = over-represented in female context.
    """
    N   = n11 + n12 + n21 + n22
    if N == 0: return np.nan
    E11 = (n11+n12)*(n11+n21)/N
    E12 = (n11+n12)*(n12+n22)/N
    E21 = (n21+n22)*(n11+n21)/N
    E22 = (n21+n22)*(n12+n22)/N
    
    def safe_ll(o, e):
        if o == 0 or e <= 0: return 0
        return o * np.log(o/e)
    
    G2 = 2*(safe_ll(n11,E11)+safe_ll(n12,E12)+
            safe_ll(n21,E21)+safe_ll(n22,E22))
    # Sign: positive if over-represented in female context
    female_rate = n11 / max(n11 + n21, 1)
    male_rate   = n12 / max(n12 + n22, 1)
    
    if female_rate > male_rate:
        sign = 1 
    elif female_rate < male_rate:
        sign = -1
    else:
        sign = 0
    return sign * G2

def compute_log_odds_ratio(ff, mf, total_f, total_m, alpha=0.5):
    """
    Smoothed log odds ratio with Haldane-Anscombe correction.
    """
    a = ff + alpha
    b = mf + alpha
    c = (total_f - ff) + alpha
    d = (total_m - mf) + alpha

    lor = np.log((a * d) / (b * c))
    se = np.sqrt(1/a + 1/b + 1/c + 1/d)
    ci_low = lor - 1.96 * se
    ci_high = lor + 1.96 * se

    return lor, se, ci_low, ci_high


def p_to_stars(p):
    if p < 0.001:
        return '***'
    elif p < 0.01:
        return '**'
    elif p < 0.05:
        return '*'
    else:
        return 'ns'

def build_llr_table(ghosts_df, lang, min_freq=3, source_col='context', exclude_gender_refs=True):
    """
    Build LLR collocate table for a given language's ghost subset.
    Returns DataFrame with [word, female_freq, male_freq, LLR, direction].

    Parameters
    ----------
    ghosts_df : DataFrame
        Ghost mentions subset for one platform/language.
    lang : str
        Language code ('zh', 'en', 'ko', etc.)
    min_freq : int
        Minimum combined frequency threshold.
    source_col : str
        Which text column to use for collocate extraction. Usually 'sentence' or 'context'.
    exclude_gender_refs : bool
        If True, remove pronouns / gender labels / kinship terms before LLR.
    """
    lk = get_lang_key(lang)

    gender_stop = GENDER_STOPLIST.get(lk, set()) if exclude_gender_refs else set()
    gender_stop_lower = {str(x).strip().lower() for x in gender_stop}

    def fn(text):
        words = extract_content_words(text, lang, extra_stopwords=gender_stop_lower)
        if exclude_gender_refs:
            words = [w for w in words if not is_gender_reference_word(w, lang)]
        return words

    female_counts = Counter()
    male_counts = Counter()

    f_rows = ghosts_df[ghosts_df['gender_new'] == 'Female'].copy()
    m_rows = ghosts_df[ghosts_df['gender_new'] == 'Male'].copy()

    for txt in f_rows[source_col].fillna(''):
        female_counts.update(fn(str(txt)))

    for txt in m_rows[source_col].fillna(''):
        male_counts.update(fn(str(txt)))

    all_words = set(female_counts) | set(male_counts)
    total_f = sum(female_counts.values())
    total_m = sum(male_counts.values())

    rows = []
    for word in all_words:
        if exclude_gender_refs and is_gender_reference_word(word, lang):
            continue

        ff = female_counts.get(word, 0)
        mf = male_counts.get(word, 0)

        if ff + mf < min_freq:
            continue

        llr = compute_llr(ff, mf, total_f - ff, total_m - mf)
        if pd.isna(llr):
            continue

        log_odds, log_odds_se, ci_low, ci_high = compute_log_odds_ratio(
            ff, mf, total_f, total_m
        )

        rows.append({
            'word': word,
            'female_freq': ff,
            'male_freq': mf,
            'total_freq': ff + mf,
            'total_f_tokens': total_f,
            'total_m_tokens': total_m,
            'LLR': llr,
            'direction': 'Female' if llr > 0 else ('Male' if llr < 0 else 'Tie'),
            'log_odds': log_odds,
            'log_odds_se': log_odds_se,
            'log_odds_ci_low': ci_low,
            'log_odds_ci_high': ci_high
        })
    if not rows:
        return pd.DataFrame(columns=['word', 'female_freq', 'male_freq',
            'total_f_tokens', 'total_m_tokens',
            'LLR', 'direction',
            'p_value', 'p_fdr', 'sig_fdr',
            'log_odds', 'log_odds_se', 'log_odds_ci_low', 'log_odds_ci_high'])

    df_llr = pd.DataFrame(rows)

    df_llr['p_value'] = df_llr['LLR'].abs().apply(lambda x: chi2.sf(x, df=1))
    # BH-FDR
    reject, p_fdr, _, _ = multipletests(df_llr['p_value'], alpha=0.05, method='fdr_bh')
    df_llr['p_fdr'] = p_fdr
    df_llr['significant_fdr'] = reject
    df_llr['sig_fdr'] = df_llr['p_fdr'].apply(p_to_stars)
    df_llr['sig_raw'] = df_llr['p_value'].apply(p_to_stars)

    if exclude_gender_refs:
        df_llr = df_llr[~df_llr['word'].apply(lambda w: is_gender_reference_word(w, lang))].copy()

    df_llr = df_llr.sort_values('LLR', ascending=False).reset_index(drop=True)
    return df_llr

# Custom post-hoc filtering for LLR tables

CUSTOM_STOPWORDS = {
    'reddit': {
        # female
    "tammy","hiroshi","tanot","rlly","thoose","fufaji","lieish","wanna","lydia",
    "spay","romae","tuk","mary","aswell","molly","marit","lily","luna","teddy",
    "sara","heya","i'm","mamaaa","delaney","want","understanding","dm’e","emma",
    "marissa","lucy","anne","tiffany","tifor","9yo","outta","grandmom","joanne",
    "lieed","mil","janie","o’ma","room(wich","ferotni","lilly","memory).i","jill",
    "theodore","hem","betsy","mind‘s","mar","no”","night(this","mauz","pristine",
    "johny","grammer","sleeping)so","preppe","zak","pre","arnold","fuccd","olivia",
    ".it","gui","suzy","tostayy","i‘m","o’pa","10minute","23yr","este","room(yk",
    "lizabeth","shockinot","saidit","amber","explanation-","girl(a",
    "girl(a)explaine","•apparently•","blinkede","cai","her(to","foot.of","yada",
    "10min","juju","23rd","back.where","5min","aunt(maasi","closer—25","pere",
    "istg","maisie","lorraine","me-","shu","mawmoo","banot","woahhhhh","dou",
    "bobby","esp","mariah","healer(aujha","fyi",

    # URLs
    "https://preview.redd.it/f3wv64x2mvff1.jpg",
    "https://preview.redd.it/yiy0tgx2mvff1.jpg",
    "of.me",

    # male
    "wyatt","adam","elijah","ghurapak","zozo","bob","rafita","josh","iblees","doo",
    "alex","baba","can","fred","bidin","tommy","cameron","rem","joe","up.i","cree",
    "cnt","trianot","kinda","julunga","sooooo","asking“when","mathis","jeune",
    "bak","bek","roy","context-","somthe",".at","exam(but","distorced","davinkof",
    "ali","eas","2th","mumu","balaji","thry","atchen","pop!—and","hazlitt","11yo",
    "mervin","sens","ava",".we","luca","lieu","chacha","jackson","psspsse","kwik",
    "guinea","katherine","joey","mima","umm","shit,,we","eve","thay",
    "grandfather‘s","football,,another","husband‘s","3:30–4:00","decades-",
    "*they","not\\","deffo","tvd","weirddddd-","think).it","ezekiel","7ish",
    "friend(i","dark/","john?(cousin","8am","noo","do?","side.i","qrf","lea",
    "india(shillong","5’6","muat","eveythe","yah","out.im","4yo","allan",
    "etc.—a","narra","5’9",".he","ect","tyler","gra","xoriyoh(mustard"
    },

    'ptt': {
        '咖咖', '打化','寧次','阿君仔','所有人','變得','阿嬷','女學',
        '阿花','姐說','巡家','糖糖','阿嬤才','道親','阿嬤會', '用罵',
        '不住','老奶奶','伯母','實有','小語','問學','二姊夫','姨婆','想用',
        '沒打','小杏','嬸嬸','妳是','也常','阿娥','時路','專時','爸人',
        '駕畫','時感','生被','具店','東東','節過','過放','舍妹','慧夢見','阿婆',
        '小雪','吳媽媽','條魚','技正','小替','添明','二哥','橘子','大好','阿喀',        
        '字字','所謂','王里長','林君','地頭','弟房','咖達','用望','應到','紛紛', '尤其'
        '平常','大概','口傑','死物','會伴','村伯','阿興','面找','信命','著門','艾克才','蹦蹦','德國'
    },

    'theqoo': {
        "자주","치이","어서","이전","절반","극소","제리","요즘","엄지","아들"
    }
}


def filter_llr_df(df, lang=None, platform=None):
    if df is None or df.empty:
        return df

    out = df.copy()
    out['word_str'] = out['word'].astype(str)
    out['word_lower'] = out['word_str'].str.lower().str.strip()

    stopset = set()
    if platform in CUSTOM_STOPWORDS:
        stopset = {w.lower() for w in CUSTOM_STOPWORDS[platform]}

    # 1) exact-match custom stopwords
    out = out[~out['word_lower'].isin(stopset)]

    # 2) remove URLs
    out = out[~out['word_lower'].str.contains(r'http|www|preview\.redd\.it', regex=True, na=False)]

    # 3) remove obvious malformed punctuation-heavy tokens
    out = out[~out['word_str'].str.contains(r'[\[\]\(\)\{\}]', regex=True, na=False)]

    # 4) remove tokens containing digits
    out = out[~out['word_str'].str.contains(r'\d', regex=True, na=False)]

    # 5) remove very broken tokens starting with punctuation
    out = out[~out['word_str'].str.contains(r'^[^\w가-힣一-龥]+', regex=True, na=False)]

    # cleanup
    out = out.drop(columns=['word_str', 'word_lower']).reset_index(drop=True)
    return out

print("LLR function ready.")

In [ ]:
# ── Part 7.4: Compute LLR for all 3 platforms (resumable) ───────────────────
import json

llr_tables = {}
llr_targets = [('ptt', 'zh'), ('reddit', 'en'), ('theqoo', 'ko')]
checkpoint_path = OUT_DIR / 'collocate_LLR_progress.json'

# Load checkpoint (if any)
if checkpoint_path.exists():
    try:
        with open(checkpoint_path, 'r', encoding='utf-8') as f:
            checkpoint = json.load(f)
    except Exception:
        checkpoint = {'done': {}, 'errors': {}}
else:
    checkpoint = {'done': {}, 'errors': {}}

print(f"Checkpoint file: {checkpoint_path}")
print(f"Completed so far: {list(checkpoint.get('done', {}).keys())}")

for plat, lang in llr_targets:
    csv_path = OUT_DIR / f'collocate_LLR_{plat}.csv'

    # Resume: if already done and csv exists, load and skip recompute
    if plat in checkpoint.get('done', {}) and csv_path.exists():
        llr_tables[plat] = pd.read_csv(csv_path)
        print(f"[SKIP] {plat.upper()} already computed -> {csv_path}")
        continue

    sub = ghosts_known[ghosts_known['platform'] == plat]
    print(f"Computing LLR for {plat.upper()} ({lang}) - {len(sub):,} known-gender ghost mentions...")

    try:
        llr_df = build_llr_table(
            sub,
            lang=lang,
            min_freq=3,
            source_col='context',      
            exclude_gender_refs=True
)
        llr_df = filter_llr_df(llr_df, lang=lang, platform=plat)
        
        llr_tables[plat] = llr_df

        # Atomic write: write tmp first, then replace
        tmp_path = OUT_DIR / f'collocate_LLR_{plat}.csv.tmp'
        llr_df.to_csv(tmp_path, index=False)
        tmp_path.replace(csv_path)

        top_f = llr_df[llr_df['direction'] == 'Female'].sort_values('LLR', ascending=False).head(10)['word'].tolist()
        top_m = llr_df[llr_df['direction'] == 'Male'].sort_values('LLR', ascending=True).head(10)['word'].tolist()
        print(f"  Top female-associated: {top_f}")
        print(f"  Top male-associated:   {top_m}")
        print(f"  Saved: {csv_path}")

        checkpoint['done'][plat] = {
            'lang': lang,
            'rows': int(len(llr_df)),
            'path': str(csv_path)
        }
        checkpoint['errors'].pop(plat, None)

        with open(checkpoint_path, 'w', encoding='utf-8') as f:
            json.dump(checkpoint, f, ensure_ascii=False, indent=2)
        print(f"  Checkpoint updated: {checkpoint_path}\\n")

    except KeyboardInterrupt:
        # Keep partial progress and stop cleanly
        with open(checkpoint_path, 'w', encoding='utf-8') as f:
            json.dump(checkpoint, f, ensure_ascii=False, indent=2)
        print("\\nInterrupted. Partial results are saved.")
        print(f"Re-run this cell to resume from: {checkpoint_path}")
        break

    except Exception as e:
        checkpoint['errors'][plat] = str(e)
        with open(checkpoint_path, 'w', encoding='utf-8') as f:
            json.dump(checkpoint, f, ensure_ascii=False, indent=2)
        print(f"[ERROR] {plat.upper()} failed: {e}")
        print("  Progress saved. You can fix and re-run to resume.\\n")

In [ ]:
# ── Part 7.5: Plot LLR with significance stars ──────────────────────────────
from matplotlib.patches import Patch
from matplotlib import font_manager as _fm

if 'llr_tables' not in globals() or not isinstance(llr_tables, dict):
    llr_tables = {}

plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = [
    'Noto Sans TC',
    'AppleGothic',
    'Hiragino Sans',
    'Hiragino Sans GB',
    'Arial Unicode MS',
    'DejaVu Sans',
]
plt.rcParams['axes.unicode_minus'] = False

fig, axes = plt.subplots(1, 3, figsize=(17, 7))
platform_labels = {
    'ptt': 'PTT (Traditional Chinese)',
    'reddit': 'Reddit (English)',
    'theqoo': 'Theqoo (Korean)'
}
colors = {'Female': '#E07B7B', 'Male': '#6B9EC7'}
N_SHOW = 15

for ax, plat in zip(axes, ['ptt', 'reddit', 'theqoo']):
    df_llr = llr_tables.get(plat, pd.DataFrame())

    # fallback after restart
    if df_llr.empty:
        csv_path = OUT_DIR / f'collocate_LLR_{plat}.csv'
        if csv_path.exists():
            df_llr = pd.read_csv(csv_path)
            llr_tables[plat] = df_llr
            print(f"Loaded cached LLR: {csv_path}")

    if df_llr.empty:
        ax.set_title(f'{plat.upper()} — no data')
        continue

    if 'direction' in df_llr.columns:
        direction_norm = df_llr['direction'].astype(str).str.strip().str.lower()
        female_mask = direction_norm.str.contains('female', na=False)
        male_mask = direction_norm.str.contains('male', na=False)
    else:
        female_mask = df_llr['LLR'] > 0
        male_mask = df_llr['LLR'] < 0

    top_f = df_llr[female_mask].sort_values('LLR', ascending=False).head(N_SHOW)
    top_m = df_llr[male_mask].sort_values('LLR', ascending=True).head(N_SHOW)

    if top_f.empty and top_m.empty:
        ax.set_title(f'{plat.upper()} — no valid LLR rows')
        continue

    plot_parts = []
    if not top_f.empty:
        plot_parts.append(top_f.assign(plot_llr=top_f['LLR'], side='Female'))
    if not top_m.empty:
        plot_parts.append(top_m.assign(plot_llr=top_m['LLR'], side='Male'))

    plot_df = pd.concat(plot_parts, ignore_index=True).sort_values('plot_llr')
    plot_df['label'] = plot_df['side'] + ' · ' + plot_df['word'].astype(str)
    bar_colors = [colors[side] for side in plot_df['side']]

    bars = ax.barh(
        plot_df['label'],
        plot_df['plot_llr'],
        color=bar_colors,
        edgecolor='white',
        linewidth=0.4
    )

    ax.axvline(0, color='black', linewidth=0.7)
    ax.set_title(platform_labels[plat], fontsize=10, fontweight='bold')
    ax.set_xlabel('Signed LLR (+ = Female-associated, − = Male-associated)', fontsize=8)

    max_abs = plot_df['plot_llr'].abs().max()
    if pd.notna(max_abs) and max_abs > 0:
        ax.set_xlim(-max_abs * 1.15, max_abs * 1.15)

    # significance stars: prefer FDR
    star_col = 'sig_fdr' if 'sig_fdr' in plot_df.columns else ('sig_raw' if 'sig_raw' in plot_df.columns else None)

    if star_col is not None and pd.notna(max_abs) and max_abs > 0:
        offset = max_abs * 0.02
        for bar, (_, row) in zip(bars, plot_df.iterrows()):
            star = row[star_col]
            if star == 'ns':
                continue

            x = bar.get_width()
            y = bar.get_y() + bar.get_height() / 2

            if x >= 0:
                ax.text(x + offset, y, star, va='center', ha='left', fontsize=8, fontweight='bold')
            else:
                ax.text(x - offset, y, star, va='center', ha='right', fontsize=8, fontweight='bold')

    ax.legend(
        handles=[
            Patch(fc=colors['Female'], label='Female ghost'),
            Patch(fc=colors['Male'], label='Male ghost')
        ],
        fontsize=8,
        loc='lower right'
    )

    if plat == 'theqoo':
        for tick_label in ax.get_yticklabels():
            tick_label.set_fontfamily('AppleGothic')

plt.suptitle(
    'Collocate Log-Likelihood Ratios: Words Associated with Female vs Male Ghosts',
    fontsize=12,
    fontweight='bold'
)
plt.tight_layout()
plt.savefig(OUT_DIR / 'collocate_LLR_divergence_3platforms_v5.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
mentions_df = pd.read_csv(OUT_DIR / 'mentions_final_checked_with_affect.csv', encoding='utf-8-sig')
mentions_df.head()

## Part 8 · Sentiment / Affect Analysis

Compare positive valence, negative valence, and revenge scores between
**female ghost** and **male ghost** entity mentions using:
- **Welch's two-sample t-test** (unequal variance)
- **Cohen's d** effect size

In [ ]:


# ── Part 8.1: Welch t-test + Cohen's d ───────────────────────────────────────

def cohens_d(a, b):
    # C3: pooled SD weighted by n (correct for unequal group sizes)
    n1, n2 = len(a), len(b)
    s1, s2 = a.std(ddof=1), b.std(ddof=1)
    pooled = np.sqrt(((n1-1)*s1**2 + (n2-1)*s2**2) / (n1+n2-2))
    return (a.mean() - b.mean()) / pooled if pooled > 0 else 0.0

affect_cols = ['val_pos','val_neg','val_revenge']
platforms   = ['ptt','reddit','theqoo']
sent_results = []

# Rationale for Welch's t-test without explicit normality test:
# Affect scores are zero-inflated and right-skewed, but with n >> 30 per group
# the Central Limit Theorem ensures the sampling distribution of means is
# approximately normal (Lumley et al. 2002, Statistics in Medicine).
# Welch's variant (equal_var=False) is robust to unequal group variances.
print("Sample balance per platform:")
for plat in platforms:
    sub = ghosts_known[ghosts_known['platform'] == plat]
    f_n = (sub['gender_new'] == 'Female').sum()
    m_n = (sub['gender_new'] == 'Male').sum()
    print(f"  {plat:8s}: Female={f_n:,}  Male={m_n:,}  ratio={f_n/max(m_n,1):.2f}")
print()

print("=" * 70)
print("AFFECT SCORE COMPARISON: Female vs Male Ghost Entities")
print("=" * 70)

for plat in platforms:
    sub = ghosts_known[ghosts_known['platform'] == plat]
    f_grp = sub[sub['gender_new'] == 'Female']
    m_grp = sub[sub['gender_new'] == 'Male']
    print(f"\n{plat.upper()} — Female n={len(f_grp):,} | Male n={len(m_grp):,}")
    print(f"  {'Metric':<12} {'Female μ':>9} {'Male μ':>9} {'t':>7} {'p':>8} {'d':>7}")
    print("  " + "-"*55)
    for col in affect_cols:
        fa = f_grp[col].dropna()
        ma = m_grp[col].dropna()
        if len(fa) < 5 or len(ma) < 5:
            continue
        t, p = stats.ttest_ind(fa, ma, equal_var=False)
        d    = cohens_d(fa, ma)
        sig  = '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else ''))
        print(f"  {col:<12} {fa.mean():>9.4f} {ma.mean():>9.4f} {t:>7.2f} {p:>8.4f}{sig:>4}  d={d:+.3f}")
        sent_results.append({
            'Platform': plat.upper(), 'Metric': col,
            'Female_mean': fa.mean(), 'Male_mean': ma.mean(),
            't': t, 'p': p, 'cohens_d': d,
        })

sent_df = pd.DataFrame(sent_results)
# I1: Bonferroni correction across all platform × affect dimension tests
from statsmodels.stats.multitest import multipletests
if len(sent_df) > 0:
    reject, p_adj, _, _ = multipletests(sent_df['p'].values, method='bonferroni')
    sent_df['p_adj']    = p_adj
    sent_df['sig_adj']  = ['***' if pa<0.001 else ('**' if pa<0.01 else ('*' if pa<0.05 else '')) for pa in p_adj]
    sent_df['reject_H0'] = reject
    n_sig = reject.sum()
    print(f'  Bonferroni-corrected: {n_sig}/{len(sent_df)} tests survive α=0.05')
sent_df.to_csv(OUT_DIR / 'sentiment_gender_comparison.csv', index=False)
print(f"\nSaved: {OUT_DIR / 'sentiment_gender_comparison.csv'}")

In [ ]:
# ── Part 8.2: Visualisation — violin plots ───────────────────────────────────
fig, axes = plt.subplots(len(affect_cols), 3,
                         figsize=(13, len(affect_cols) * 2.8), sharey='row')

for row_i, col in enumerate(affect_cols):
    for col_i, plat in enumerate(platforms):
        ax  = axes[row_i][col_i]
        sub = ghosts_known[ghosts_known['platform'] == plat]
        data = [sub[sub['gender_new'] == g][col].dropna().values
                for g in ['Female','Male']]
        vp = ax.violinplot(data, positions=[0,1], showmedians=True)
        for i, (pc, color) in enumerate(zip(vp['bodies'], ['#E07B7B','#6B9EC7'])):
            pc.set_facecolor(color)
            pc.set_alpha(0.7)
        ax.set_xticks([0,1])
        ax.set_xticklabels(['Female','Male'])
        if col_i == 0: ax.set_ylabel(col.replace('val_',''), fontsize=9)
        if row_i == 0:
            plat_label = {'ptt':'PTT (zh)','reddit':'Reddit (en)','theqoo':'Theqoo (ko)'}
            ax.set_title(plat_label[plat], fontsize=10, fontweight='bold')
        ax.grid(axis='y', alpha=0.3)

plt.suptitle('Affect Scores: Female vs Male Ghost Entity Mentions\n(Computed from multilingual affect lexicons)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(OUT_DIR / 'sentiment_violin_3platforms.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: sentiment_violin_3platforms.png")

In [ ]:
# ── Part 8.3: Cohen's d heatmap ──────────────────────────────────────────────
pivot = (sent_df.pivot(index='Metric', columns='Platform', values='cohens_d')
               .reindex(['val_pos','val_neg','val_revenge'])
               .fillna(0))

fig, ax = plt.subplots(figsize=(7, 4.5))
sns.heatmap(pivot, annot=True, fmt='.3f', center=0,
            cmap='RdBu_r', linewidths=0.5, ax=ax,
            cbar_kws={'label': "Cohen's d (Female − Male)"})
ax.set_title("Cohen's d: Female vs Male Ghost Affect Scores\n(positive = Female scores higher)",
             fontsize=11)
ax.set_xticklabels(pivot.columns, fontsize=9)
ax.set_yticklabels(['Positive','Negative','Revenge'], rotation=0, fontsize=9)
plt.tight_layout()
plt.savefig(OUT_DIR / 'cohens_d_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: {OUT_DIR / 'cohens_d_heatmap.png'}")

## Part 9 · Word2Vec Semantic Embedding

A separate **Word2Vec** model (Skip-gram, 200 dimensions, window=5, min_count=3)
is trained per platform on the full raw article corpus. Language-specific
tokenisation is applied. After training, gendered ghost anchor terms are queried
for nearest neighbours, and cosine similarity to sentiment poles is measured.

In [ ]:
# ── Part 9.1: Tokenisation per language ─────────────────────────────────────
# Uses STOPWORDS_EN / STOPWORDS_KO / STOPWORDS_ZH defined in Part 2.1

# ── Guard: re-register custom vocab for Part 9 tokenisers ───────────────────
# (Part 9 defines its own tokenize_zh/ko/en functions separate from Part 7;
#  re-registering ensures compound ghost terms survive segmentation)
if JIEBA_OK:
    import jieba as _jieba
    for _w in JIEBA_CUSTOM_WORDS:          # includes TXT file words (Cell 8)
        _jieba.add_word(_w)
    print(f"  [Part 9] jieba: {len(JIEBA_CUSTOM_WORDS)} custom words registered")

if SPACY_OK:
    for _phrase in SPACY_PROTECTED_PHRASES:
        try:
            nlp_en.tokenizer.add_special_case(_phrase, [{"ORTH": _phrase}])
        except Exception:
            pass
    print(f"  [Part 9] spaCy: {len(SPACY_PROTECTED_PHRASES)} phrases protected")

if KIWI_OK:
    for _w in KIWI_CUSTOM_WORDS:           # defined in Cell 8
        kiwi_model.add_user_word(_w)
    print(f"  [Part 9] Kiwi: {len(KIWI_CUSTOM_WORDS)} compound words registered")

import re
from gensim.models import Word2Vec

def tokenize_zh(text):
    if pd.isna(text): return []
    if JIEBA_OK:
        import jieba
        toks = list(jieba.cut(str(text), cut_all=False))
    else:
        t = re.sub(r'[\s\W]', '', str(text))
        toks = [t[i:i+2] for i in range(len(t)-1)]
    return [w for w in toks if w not in STOPWORDS_ZH and len(w) > 1]

def tokenize_ko(text):
    if pd.isna(text): return []
    if KIWI_OK:
        result = kiwi_model.analyze(str(text))[0][0]
        toks   = [tok.form for tok in result if tok.tag in KEEP_TAGS_KO]
    else:
        toks = str(text).split()
    return [w for w in toks if w not in STOPWORDS_KO and len(w) > 1]

def tokenize_en(text):
    if pd.isna(text): return []
    if SPACY_OK:
        doc  = nlp_en(str(text))
        toks = []
        for tok in doc:
            if tok.is_stop: continue
            if tok.lemma_.lower() in NOUN_FORCE_EN and tok.pos_ not in {'NOUN', 'PROPN'}:
                continue  # skip "to ghost someone" (VERB), "ghosted" (ADJ)
            if tok.pos_ in KEEP_POS_EN:
                toks.append(tok.lemma_.lower())
    else:
        toks = re.findall(r'[a-z]+', str(text).lower())
    return [w for w in toks if w not in STOPWORDS_EN and len(w) > 2]

TOKENIZERS = {'zh': tokenize_zh, 'ko': tokenize_ko, 'en': tokenize_en}
TOKENIZE_FN = {'ptt': tokenize_zh, 'theqoo': tokenize_ko, 'reddit': tokenize_en}

def get_lang_tokenizer(lang):
    lk = get_lang_key(lang)
    return TOKENIZERS.get(lk, lambda x: str(x).lower().split())

# Build tokenised corpus (for Word2Vec)
print("Tokenising mentions for word embeddings...")
tok_col = 'tokens_w2v'
mentions_df[tok_col] = mentions_df.apply(
    lambda r: get_lang_tokenizer(r['lang'])(r.get('context', r.get('sentence',''))),
    axis=1
)
corpus_size = mentions_df[tok_col].apply(len).sum()
print(f"  total tokens in corpus: {corpus_size:,}")
print(f"  avg tokens/mention: {mentions_df[tok_col].apply(len).mean():.1f}")

In [ ]:
# ── Part 9.2: Train Word2Vec per platform ────────────────────────────────────
# Checkpoint behaviour: if model file already exists, load it and skip retraining.
# Delete OUT_DIR/w2v_<platform>.model to force fresh training.
from gensim.models import Word2Vec
from gensim.models.callbacks import CallbackAny2Vec

class _EpochLogger(CallbackAny2Vec):
    """Print loss after each epoch; useful for tracking long training runs."""
    def __init__(self, plat):
        self.plat  = plat
        self.epoch = 0
    def on_epoch_end(self, model):
        self.epoch += 1
        try:
            loss = model.get_latest_training_loss()
        except Exception:
            loss = float('nan')
        print(f"    [{self.plat}] epoch {self.epoch}/10  loss={loss:.1f}")

w2v_models = {}

for plat, df in raw_dfs.items():
    model_path = OUT_DIR / f'w2v_{plat}.model'
    # ── Resume: load existing model if available ─────────────────────────
    if model_path.exists():
        try:
            w2v_models[plat] = Word2Vec.load(str(model_path))
            print(f"  [{plat.upper()}] loaded existing model "
                  f"({len(w2v_models[plat].wv):,} vocab) — skipping retraining")
            continue
        except Exception as e:
            bad_path = model_path.with_suffix(model_path.suffix + '.incompatible')
            print(f"  [{plat.upper()}] existing model load failed: {e}")
            print(f"  [{plat.upper()}] renaming incompatible checkpoint to: {bad_path.name}")
            model_path.rename(bad_path)

    # ── Train from scratch ───────────────────────────────────────────────
    tok_fn = TOKENIZE_FN[plat]
    print(f"Tokenising {plat.upper()} ({len(df):,} articles)...")
    sentences = [toks for content in df['content'].dropna()
                 if (toks := tok_fn(str(content)))]
    print(f"  Training Word2Vec on {len(sentences):,} tokenised documents...")
    model = Word2Vec(
        sentences,
        vector_size=200,
        window=5,
        min_count=3,
        sg=1,           # Skip-gram
        epochs=10,
        workers=4,
        seed=SEED,
        compute_loss=True,
        callbacks=[_EpochLogger(plat)],
    )
    w2v_models[plat] = model
    print(f"  Vocabulary size: {len(model.wv):,} terms")
    model.save(str(model_path))
    print(f"  Saved: {model_path}")
    print(f"  (Delete this file to force retraining on next run)")

In [ ]:
# ── Part 9.3: Nearest neighbours for gendered ghost anchors ─────────────────
# Anchors include both direct gendered terms AND indirect / euphemistic references.
ANCHOR_TERMS = {
    'ptt': {
        'female': [
            '女鬼','女性鬼魂',              # direct
            '白衣女子','紅衣女子','白衣女鬼','紅衣女鬼', # appearance-based
            '女性孤魂','女性靈體','女飄'
            
        ],
        'male': [
            '男鬼','男性鬼魂',               # direct
            '好兄弟',                  # euphemistic / indirect
            '男性孤魂','男性靈體','男飄'
        ],
    },
    'theqoo': {
        'female': [
            '여귀','여자귀신','여성귀신',                # direct
            '여자유령','처녀귀신',               # indirect female ghost
                              
        ],
        'male': [
            '남귀','남자귀신','남성귀신',                # direct
            '남자유령','총각귀신',
            
        ],
    },
    'reddit': {
        # English lacks gendered ghost nouns → vector arithmetic
        'female': ('arithmetic',
                   ['ghost','spirit','apparition','woman','girl','lady','she','her'],
                   ['man','boy','he','him']),
        'male':   ('arithmetic',
                   ['ghost','spirit','apparition','man','boy','he','him'],
                   ['woman','girl','she','her']),
    },
}

print("=" * 60)
print("WORD2VEC NEAREST NEIGHBOURS — GENDERED GHOST ANCHORS")
print("=" * 60)
nn_results = []

for plat, anchors in ANCHOR_TERMS.items():
    model = w2v_models.get(plat)
    if model is None: continue
    wv = model.wv
    print(f"\n{plat.upper()}:")

    for gender, spec in anchors.items():
        if isinstance(spec, tuple):
            # vector arithmetic (Reddit)
            _, pos_words, neg_words = spec
            pos_vecs = [wv[w] for w in pos_words if w in wv]
            neg_vecs = [wv[w] for w in neg_words if w in wv]
            if not pos_vecs: continue
            anchor_vec = (np.mean(pos_vecs, axis=0) -
                          (np.mean(neg_vecs, axis=0) if neg_vecs else 0))
            sims = [(w, np.dot(anchor_vec, wv[w]) /
                        (np.linalg.norm(anchor_vec) * np.linalg.norm(wv[w])))
                    for w in wv.key_to_index]
            sims.sort(key=lambda x: -x[1])
            nn = [(w, round(s,4)) for w,s in sims[:15]
                  if w not in pos_words + neg_words][:10]
        else:
            # direct/indirect lookup (Chinese / Korean): average anchor vectors
            found_vecs = [wv[t] for t in spec if t in wv]
            if not found_vecs:
                print(f"  {gender}: no anchor terms found in vocabulary")
                continue
            anchor_vec = np.mean(found_vecs, axis=0)
            candidates = wv.similar_by_vector(anchor_vec, topn=20 + len(spec))
            nn = [(w, round(s, 4)) for w, s in candidates if w not in spec][:10]

        print(f"  {gender.upper()} ghost nearest neighbours: {[w for w,_ in nn]}")
        for rank, (word, sim) in enumerate(nn, 1):
            nn_results.append({'platform': plat, 'gender': gender,
                                'rank': rank, 'word': word, 'cosine_sim': sim,
                                'anchor_type': 'hand_coded'})

nn_df = pd.DataFrame(nn_results)
nn_df.to_csv(OUT_DIR / 'w2v_nearest_neighbours.csv', index=False)
print(f"\nSaved: {OUT_DIR / 'w2v_nearest_neighbours.csv'}")

# ── Data-driven anchors: use cascade-classified entity names as anchors ───────
print("\n" + "─" * 60)
print("DATA-DRIVEN ANCHORS (from cascade-classified supernatural entities)")
print("─" * 60)

state_col = 'state_cascade' if 'state_cascade' in mentions_df.columns else None

for plat in ['ptt', 'reddit', 'theqoo']:
    wv  = w2v_models[plat].wv
    sub = mentions_df[mentions_df['platform'] == plat]
    if state_col:
        sub = sub[sub[state_col].isin(['Supernatural'])]

    for gender in ['Female', 'Male']:
        entity_counts = (
            sub[sub['gender_new'] == gender]['entity_canon']
            .dropna()
            .value_counts()
        )
        vecs = [wv[e] for e in entity_counts.index if e in wv][:20]
        if not vecs:
            print(f"  [{plat}/{gender}] no entity tokens in W2V vocab — skipping")
            continue
        anchor_vec = np.mean(vecs, axis=0)
        candidates = wv.similar_by_vector(anchor_vec, topn=20)
        nn = [(w, round(s, 4)) for w, s in candidates
              if w not in entity_counts.index][:10]
        print(f"  [{plat}/{gender:6s}] {[w for w, _ in nn]}")
        for rank, (word, sim) in enumerate(nn, 1):
            nn_results.append({
                'platform': plat, 'gender': gender,
                'rank': rank, 'word': word,
                'cosine_sim': sim, 'anchor_type': 'data_driven',
            })

# Re-save with data-driven rows included
pd.DataFrame(nn_results).to_csv(OUT_DIR / 'w2v_nearest_neighbours.csv', index=False)
print(f"\nUpdated CSV (hand-coded + data-driven): {OUT_DIR / 'w2v_nearest_neighbours.csv'}")

In [ ]:
# ── Part 9.4: Cosine similarity to sentiment poles ────────────────────────────
# SENTIMENT_POLES is derived directly from the expanded AFFECT_LEXICON (Cell 39)
# so it automatically includes all NTUSD / ksenticnet additions.
SENTIMENT_POLES = {
    plat: {
        'positive': list(AFFECT_LEXICON[lang]['positive']),
        'negative': list(AFFECT_LEXICON[lang]['negative']),
        'revenge':     list(AFFECT_LEXICON[lang]['revenge']),
    }
    for plat, lang in [('ptt', 'zh'), ('reddit', 'en'), ('theqoo', 'ko')]
}

# Appeasement / euphemism pole — tests 好兄弟 phenomenon cross-culturally
SENTIMENT_POLES['ptt']['euphemism'] = [
    '好兄弟','普渡','祭品','拜拜','保佑','超度','金紙','香','送走','謝謝','念佛','超渡','祭拜','祭祖',
    '有應公','萬善公','普度','中元','燒香','祭祀','供品','感謝','請走','念經',
]
SENTIMENT_POLES['theqoo']['euphemism'] = [
    '위령','달래','제사','공양','부탁','감사','무사히','위로','기도','천도',
    '고사','제물','위패','저승','부디','영혼','명복', '편히', '보살펴', '달래다',
    '49재','천도재','달래주다','내세','영계',
]
SENTIMENT_POLES['reddit']['euphemism'] = [
    'please','leave me alone','i mean no harm','rest in peace',
    'bless','appease','offering','pray','protect','go away','pass away',
    'thank you','sorry','i respect','i apologize','peace be with','forgive',
    'jesus christ','god', 'buddha', 'allah', 'saint', 'spirit world', 'afterlife',
    'respect','apologize','condolences','sympathies','heaven','hell','spirit realm',
]

print("=" * 60)
print("COSINE SIMILARITY: Ghost Gender Vectors → Sentiment Poles")
print("=" * 60)
cos_results = []

for plat in ['ptt','reddit','theqoo']:
    wv    = w2v_models[plat].wv
    poles = SENTIMENT_POLES[plat]
    print(f"\n{plat.upper()}:")

    for gender in ['female','male']:
        anchor_spec = ANCHOR_TERMS[plat][gender]
        if isinstance(anchor_spec, tuple):
            _, pos_words, neg_words = anchor_spec
            pvs = [wv[w] for w in pos_words if w in wv]
            nvs = [wv[w] for w in neg_words if w in wv]
            if not pvs: continue
            ghost_vec = np.mean(pvs, axis=0) - (np.mean(nvs, axis=0) if nvs else 0)
        else:
            vecs = [wv[t] for t in anchor_spec if t in wv]
            if not vecs: continue
            ghost_vec = np.mean(vecs, axis=0)

        ghost_vec_norm = ghost_vec / (np.linalg.norm(ghost_vec) + 1e-8)

        for pole_name, pole_words in poles.items():
            pole_vecs = [wv[w] for w in pole_words if w in wv]
            if not pole_vecs: continue
            pole_vec  = np.mean(pole_vecs, axis=0)
            pole_norm = pole_vec / (np.linalg.norm(pole_vec) + 1e-8)
            sim       = float(np.dot(ghost_vec_norm, pole_norm))
            print(f"  {gender.upper()} → {pole_name:14s}: {sim:.4f}")
            cos_results.append({'platform': plat, 'gender': gender,
                                 'pole': pole_name, 'cosine_sim': sim})

cos_df = pd.DataFrame(cos_results)
cos_df.to_csv(OUT_DIR / 'w2v_cosine_similarity_poles.csv', index=False)
print(f"\nSaved: {OUT_DIR / 'w2v_cosine_similarity_poles.csv'}")

## Part 10 · BERT Multilingual Contextual Embeddings

`bert-base-multilingual-cased` is used to extract mean-pooled contextual embeddings
(attention-mask-weighted average over all non-padding tokens) for a stratified
sample of ghost entity sentences. PCA (→50 dims) + t-SNE (→2 dims) visualise
whether female and male ghost contexts cluster separately.

In [ ]:
# ── Part 10.1: Load multilingual BERT ────────────────────────────────────────
BERT_OK = False
try:
    from transformers import AutoTokenizer, AutoModel
    import torch
    bert_name = 'bert-base-multilingual-cased'
    bert_tok  = AutoTokenizer.from_pretrained(bert_name)
    bert_model= AutoModel.from_pretrained(bert_name)
    bert_model.eval()
    BERT_OK = True
    print(f"BERT ({bert_name}) loaded ✓")
except Exception as e:
    print(f"BERT not available: {e}")
    print("Skipping BERT analysis (Parts 10.2–10.4).")

In [ ]:
# ── Part 10.2: Stratified sample + embeddings ────────────────────────────────
if BERT_OK:
    # Sample 300 per (platform × gender): Female, Male
    sample_parts = []
    for plat in ['ptt','reddit','theqoo']:
        for gender in ['Female','Male']:
            sub = ghosts_known[(ghosts_known['platform']==plat) &
                               (ghosts_known['gender_new']==gender)]
            n   = min(300, len(sub))
            if n > 0:
                sample_parts.append(sub.sample(n, random_state=SEED))
    sample = pd.concat(sample_parts, ignore_index=True)
    print(f"BERT sample: {len(sample):,} ghost mentions")
    print(sample.groupby(['platform','gender_new']).size().unstack(fill_value=0))
    
    # I5: mean-pool over non-padding tokens (more robust than [CLS] on mBERT)
    def get_mean_embedding(text, max_len=128):
        inputs = bert_tok(str(text)[:512], return_tensors='pt',
                          max_length=max_len, truncation=True, padding='max_length')
        with torch.no_grad():
            out = bert_model(**inputs)
        mask   = inputs['attention_mask'].unsqueeze(-1).float()
        summed = (out.last_hidden_state * mask).sum(dim=1)
        mean   = summed / mask.sum(dim=1).clamp(min=1e-9)
        return mean.squeeze().numpy()
    
    print("\nComputing BERT embeddings (mean pooling)...")
    from tqdm.auto import tqdm as _tqdm
    embeddings = []
    for _, row in _tqdm(sample.iterrows(), total=len(sample), desc='BERT embed'):
        emb = get_mean_embedding(row['sentence'])
        embeddings.append(emb)
    emb_matrix = np.stack(embeddings)  # (N, 768) float32
    np.save(OUT_DIR / 'bert_embeddings.npy', emb_matrix)
    sample[['platform','gender_new','sentence']].assign(
        emb_idx=range(len(sample))
    ).to_csv(OUT_DIR / 'bert_sample_meta.csv', index=False)
    print(f"Saved: {emb_matrix.shape} → {OUT_DIR / 'bert_embeddings.npy'}")
    # Keep in-session; free model from RAM
    sample = sample.copy()
    sample['bert_embedding'] = list(emb_matrix)
    del bert_model, bert_tok
    import gc; gc.collect()
    print("BERT model freed from memory.")

In [ ]:
# ── Part 10.3: PCA → t-SNE visualisation ─────────────────────────────────────
# I4: recover embeddings from disk if session was restarted
if BERT_OK and 'bert_embedding' not in sample.columns:
    _ep = OUT_DIR / 'bert_embeddings.npy'
    if _ep.exists():
        _mat = np.load(_ep)
        sample['bert_embedding'] = list(_mat)
        print(f"Recovered {_mat.shape} embeddings from {_ep}")
    else:
        print("bert_embedding missing — re-run Cell 54.")

if BERT_OK and 'bert_embedding' in sample.columns:
    from sklearn.decomposition import PCA
    from sklearn.manifold import TSNE
    
    valid = sample.dropna(subset=['bert_embedding'])
    X = np.stack(valid['bert_embedding'].values)
    
    n_pca = min(50, X.shape[0] - 1, X.shape[1])
    pca   = PCA(n_components=n_pca, random_state=SEED)
    X_pca = pca.fit_transform(X)
    print(f"PCA: using n_components={n_pca} (clamped to sample size)")
    print(f"PCA explained variance ({n_pca} components): {pca.explained_variance_ratio_.sum():.2%}")

    # sklearn compatibility: newer versions use max_iter, older use n_iter.
    try:
        tsne = TSNE(n_components=2, random_state=42, perplexity=30, max_iter=1000)
    except TypeError:
        tsne = TSNE(n_components=2, random_state=42, perplexity=30, n_iter=1000)
    X_tsne  = tsne.fit_transform(X_pca)
    
    valid   = valid.copy()
    valid['tsne_x'] = X_tsne[:, 0]
    valid['tsne_y'] = X_tsne[:, 1]
    
    # Plot by gender and platform
    palette_g = {'Female': '#E07B7B', 'Male': '#6B9EC7'}
    markers   = {'ptt': 'o', 'reddit': '^', 'theqoo': 's'}
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    for ax, colour_by in zip(axes, ['gender_new','platform']):
        for val in valid[colour_by].unique():
            sub = valid[valid[colour_by] == val]
            c   = palette_g.get(val, None)
            ax.scatter(sub['tsne_x'], sub['tsne_y'], c=c, label=val,
                       alpha=0.5, s=15, edgecolors='none')
        ax.set_title(f't-SNE coloured by {colour_by}', fontsize=11)
        ax.legend(fontsize=8, markerscale=2)
        ax.set_xlabel('t-SNE dim 1'); ax.set_ylabel('t-SNE dim 2')
    
    plt.suptitle('BERT mBERT — Ghost Entity Sentence Embeddings', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig(OUT_DIR / 'bert_tsne_ghost_embeddings.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Saved: {OUT_DIR / 'bert_tsne_ghost_embeddings.png'}")

In [ ]:
# ── Part 10.4: Centroid distance analysis ────────────────────────────────────
# I4: recover embeddings from disk if session was restarted
if BERT_OK and 'bert_embedding' not in sample.columns:
    _ep = OUT_DIR / 'bert_embeddings.npy'
    if _ep.exists():
        _mat = np.load(_ep)
        sample['bert_embedding'] = list(_mat)
        print(f"Recovered {_mat.shape} embeddings from {_ep}")
    else:
        print("bert_embedding missing — re-run Cell 54.")

if BERT_OK and 'bert_embedding' in sample.columns:
    from numpy.linalg import norm
    
    valid = sample.dropna(subset=['bert_embedding'])
    print("=" * 60)
    print("BERT CENTROID DISTANCES (Female vs Male ghost embeddings)")
    print("=" * 60)
    
    centroid_results = []
    for plat in ['ptt','reddit','theqoo']:
        sub = valid[valid['platform'] == plat]
        f_vecs = np.stack([r for r in sub[sub['gender_new']=='Female']['bert_embedding']])
        m_vecs = np.stack([r for r in sub[sub['gender_new']=='Male']['bert_embedding']])
        if len(f_vecs) < 10 or len(m_vecs) < 10:
            print(f"  {plat.upper()}: insufficient samples")
            continue
        f_centroid = f_vecs.mean(axis=0)
        m_centroid = m_vecs.mean(axis=0)
        cosine_dist = 1 - (np.dot(f_centroid, m_centroid) /
                           (norm(f_centroid) * norm(m_centroid) + 1e-8))
        euclidean   = norm(f_centroid - m_centroid)
        print(f"  {plat.upper():8s}  cosine_dist={cosine_dist:.4f}  euclidean={euclidean:.4f}")
        centroid_results.append({'platform': plat,
                                  'cosine_dist': cosine_dist,
                                  'euclidean_dist': euclidean,
                                  'n_female': len(f_vecs),
                                  'n_male': len(m_vecs)})
    
    pd.DataFrame(centroid_results).to_csv(OUT_DIR / 'bert_centroid_distances.csv', index=False)
    print(f"\nSaved: {OUT_DIR / 'bert_centroid_distances.csv'}")

In [ ]:
# ── Part 10.5: Euphemism & Appeasement Language — BERT Contextual Analysis ───
# Tests the 好兄弟 hypothesis: do writers use positive/euphemism language
# around female vs male ghost descriptions? Uses BERT sentence embeddings +
# keyword flagging from SENTIMENT_POLES['euphemism'] (defined in Part 9.4).
#
# Output: bert_appeasement_language_by_gender.csv

if BERT_OK and 'bert_embedding' in sample.columns:
    from scipy.stats import chi2_contingency
    from numpy.linalg import norm as _norm

    print('=' * 72)
    print('PART 10.5: Appeasement & Positive Language Around Ghost Descriptions')
    print('=' * 72)
    print('Hypothesis: writers use euphemism language (好兄弟 effect) around')
    print('ghost descriptions; pattern may differ by ghost gender & culture.')
    print()

    def _contains_any(text, wordset):
        t = str(text).lower()
        return int(any(w.lower() in t for w in wordset))

    results_10_5 = []

    for plat in ['ptt', 'reddit', 'theqoo']:
        sub = sample[sample['platform'] == plat].copy()

        app_words  = set(SENTIMENT_POLES[plat].get('euphemism', []))
        pos_words  = set(SENTIMENT_POLES[plat].get('positive',    []))

        sub = sub.copy()
        sub['has_appeasement'] = sub['sentence'].apply(lambda x: _contains_any(x, app_words))
        sub['has_positive']    = sub['sentence'].apply(lambda x: _contains_any(x, pos_words))
        
        print(f"{plat.upper()} — Language pattern by ghost gender:")
        print(f"  {'Gender':<10} {'N':>5}  {'Euphemism%':>9}  {'Positive%':>10} ")
        print(f"  {'─' * 50}")

        counts = {}
        for gender in ['Female', 'Male']:
            g = sub[sub['gender_new'] == gender]
            if len(g) == 0:
                continue
            app_pct  = g['has_appeasement'].mean() * 100
            pos_pct  = g['has_positive'].mean()    * 100
            print(f"  {gender:<10} {len(g):>5}  {app_pct:>8.1f}%  {pos_pct:>9.1f}%")
            counts[gender] = {
                'n': len(g), 'app': int(g['has_appeasement'].sum()),
            }
            results_10_5.append({
                'platform': plat, 'gender': gender, 'n': len(g),
                'euphemism_pct': round(app_pct, 2),
                'positive_pct':    round(pos_pct, 2),
                
            })

        # ── Chi-square: euphemism rate Female vs Male ─────────────────
        if 'Female' in counts and 'Male' in counts:
            f, m = counts['Female'], counts['Male']
            if (f['app'] + m['app']) > 0:
                ct   = [[f['app'], f['n']-f['app']], [m['app'], m['n']-m['app']]]
                chi2_stat, p, *_ = chi2_contingency(ct, correction=True)
                sig = '***' if p<0.001 else '**' if p<0.01 else '*' if p<0.05 else 'n.s.'
                print(f"  → Appeasement ♀ vs ♂: χ²={chi2_stat:.2f}  p={p:.4f} {sig}")
        print()

    # ── BERT centroid proximity to euphemism sentences ─────────────────
    print('─' * 72)
    print('BERT embedding: euphemism-flagged sentences vs gender centroids')
    print('─' * 72)
    for plat in ['ptt', 'reddit', 'theqoo']:
        sub = sample[
            (sample['platform'] == plat) & sample['bert_embedding'].notna()
        ].copy()
        sub['has_app'] = sub['sentence'].apply(
            lambda x: _contains_any(x, set(SENTIMENT_POLES[plat].get('euphemism', []))))
        app_sub = sub[sub['has_app'] == 1]
        if len(app_sub) < 5:
            print(f"  [{plat}] too few euphemism sentences ({len(app_sub)}) — skip")
            continue
        app_centroid = np.stack(app_sub['bert_embedding'].values).mean(axis=0)
        for gender in ['Female', 'Male']:
            g_sub = sub[sub['gender_new'] == gender]
            if len(g_sub) < 5:
                continue
            g_centroid = np.stack(g_sub['bert_embedding'].values).mean(axis=0)
            cos_sim = (np.dot(app_centroid, g_centroid)
                       / (_norm(app_centroid) * _norm(g_centroid) + 1e-8))
            print(f"  [{plat}] euphemism ↔ {gender} centroid: cosine={cos_sim:.4f}")
    print()

    # ── Save ─────────────────────────────────────────────────────────────
    pd.DataFrame(results_10_5).to_csv(
        OUT_DIR / 'bert_appeasement_language_by_gender.csv', index=False)
    print(f"Saved: {OUT_DIR / 'bert_appeasement_language_by_gender.csv'}")
    print()
    print("Interpretation: 好兄弟 (lit. 'good brothers') is a Taiwanese euphemism for")
    print("ghosts; writers use positive/euphemism terms to avoid antagonising spirits.")
    print("Higher euphemism% near Male ghost contexts → 好兄弟 effect is gendered.")
    print("Cross-cultural parallels: Korean 위령 (spirit consolation), English 'rest in peace'.")

else:
    print('Run Part 10.2 first to generate BERT embeddings.')

## Part 11 · Validation Strategy

### Why entity CSV labels cannot serve as gold standard
The entity CSV `gender` / `state` columns were generated by a prior automated pipeline,
not by human annotation. Using them to validate our new pipeline is **inter-system
agreement** — not true accuracy measurement.

### Three-tier validation approach (primary → supplement → reference)
1. **Human annotation** (Part 11A) — *PRIMARY GOLD STANDARD*: You label 500 examples in
   Excel (≈? hrs). P/R/F1 computed against your labels are the main thesis results.
2. **LLM-as-Judge** (Part 11B) — *Supplement*: Claude Sonnet annotates 500 additional
   examples (≈$0.90, 3 min). Cited as Zheng et al. 2023 scale supplement.


### Part 11A · Human Annotation (Primary Gold Standard)

**This is the main validation for your thesis.** You label 500 examples yourself in Excel.

Workflow:
1. Run **Part 11A.1** → generates `annotation_task.xlsx` (500 stratified examples)
2. Open the Excel file, read each sentence, fill in your labels (takes ? hours)
3. Save as `annotation_task_completed.csv` in the same folder
4. Run **Part 11A.2** → imports your labels and computes P/R/F1 for all three methods

**Columns to fill in Excel:**
- `human_gender`: Female / Male / Unknown
- `human_state`: Alive / Deceased / Supernatural / Unknown

Pre-filled columns (do not change):
- `entity_canon`, `sentence`, `context`, `platform`, `lang`
- `rb_gender`, `rb_state` (rule-based predictions — useful to see disagreements)


In [ ]:
# ── Part 11A.1: Export annotation_task.xlsx (500 stratified examples) ───────

ANNOTATION_N    = 500           # total examples to annotate
N_PER_PLATFORM  = ANNOTATION_N // 3   # ~67 per platform
EXPORT_PATH     = RAW_DIR / 'annotation_task.xlsx'

# Use the full mentions_df (rule-based run must be complete)
if 'gender_rb' not in mentions_df.columns:
    mentions_df['gender_rb'] = mentions_df.apply(
        lambda r: classify_gender(r['entity_canon'], r['sentence'], r['lang']), axis=1)
if 'state_rb' not in mentions_df.columns:
    mentions_df['state_rb'] = mentions_df.apply(
        lambda r: classify_state(r['entity_canon'], r['sentence'], r['context'], r['lang']), axis=1)

# Stratified sample: balanced across platforms
# Within each platform: aim for gender/state diversity
export_parts = []
for plat in ['ptt', 'reddit', 'theqoo']:
    sub = mentions_df[mentions_df['platform'] == plat].copy()
    if len(sub) == 0:
        print(f'  Warning: no mentions for {plat}')
        continue
    # Stratify: prefer female/male split and supernatural/alive mix
    female = sub[sub['gender_rb'] == 'Female']
    male   = sub[sub['gender_rb'] == 'Male']
    unk    = sub[sub['gender_rb'] == 'Unknown']
    n_f = min(len(female), N_PER_PLATFORM * 4 // 10)  # 40%
    n_m = min(len(male),   N_PER_PLATFORM * 4 // 10)  # 40%
    n_u = min(len(unk),    N_PER_PLATFORM - n_f - n_m)  # remainder
    parts = []
    if n_f > 0: parts.append(female.sample(n_f, random_state=42))
    if n_m > 0: parts.append(male.sample(n_m, random_state=42))
    if n_u > 0: parts.append(unk.sample(n_u, random_state=42))
    if parts:
        export_parts.append(pd.concat(parts, ignore_index=True))

if export_parts:
    export_df = pd.concat(export_parts, ignore_index=True)
    # Trim to ANNOTATION_N if over
    export_df = export_df.sample(min(ANNOTATION_N, len(export_df)), random_state=0).reset_index(drop=True)
    # Select and rename columns for annotator
    # Include cascade predictions if they were computed (run Part 3B.4 / 4B.3 first)
    # Ensure all 4 prediction columns exist — fill cascade with NA if not run yet
    _cascade_missing = []
    for _col in ['gender_cascade', 'state_cascade']:
        if _col not in export_df.columns:
            export_df[_col] = pd.NA
            _cascade_missing.append(_col)
    if _cascade_missing:
        print(f'  Note: {_cascade_missing} not found — exporting as empty columns.')
        print('  Run Part 3B.5 / 4B.3 before exporting if you want cascade predictions.')

    export_cols = ['platform', 'lang', 'entity_canon', 'sentence', 'context',
                   'gender_rb', 'state_rb', 'gender_cascade', 'state_cascade']
    out = export_df[export_cols].copy()
    rename_map = {
        'gender_rb':      'rb_gender',
        'state_rb':       'rb_state',
        'gender_cascade': 'cascade_gender',
        'state_cascade':  'cascade_state',
    }
    out = out.rename(columns=rename_map)
    # Empty human-label columns
    out['human_gender'] = ''
    out['human_state']  = ''
    out.insert(0, 'example_id', range(1, len(out)+1))
    # Save
    out.to_excel(EXPORT_PATH, index=False)
    print(f'✓ Exported {len(out)} annotation examples to:')
    print(f'  {EXPORT_PATH}')
    print()
    print('Platform breakdown:')
    print(out.groupby('platform').size().to_string())
    print()
    print('Rule-based gender distribution:')
    print(out['rb_gender'].value_counts().to_string())
    print()
    print('Rule-based state distribution:')
    print(out['rb_state'].value_counts().to_string())
    print()
    print('Prediction columns in export:')
    for _pc in ['rb_gender', 'rb_state', 'cascade_gender', 'cascade_state']:
        if _pc in out.columns:
            _n_filled = out[_pc].notna().sum()
            print(f'  {_pc:20s}: {_n_filled:,} / {len(out)} rows populated')
    print()
    print('NEXT: Open the Excel file, fill in human_gender and human_state columns,')
    print('      then save as annotation_task_completed.xlsx and run Part 11A.2.')
else:
    print('✗ No mentions available. Run Parts 2-4 first.')


In [ ]:
# ── Part 11A.2: Import completed annotations + compute P/R/F1 ───────────
# Run after filling annotation_task.xlsx and saving as annotation_task_completed.xlsx

from sklearn.metrics import classification_report, cohen_kappa_score

COMPLETED_PATH = RAW_DIR / 'annotation_task_completed.csv'

if not COMPLETED_PATH.exists():
    print(f'File not found: {COMPLETED_PATH}')
    print('Fill in the annotation_task.xlsx first, then re-run this cell.')
else:
    completed = pd.read_csv(COMPLETED_PATH)
    completed.columns = [c.strip() for c in completed.columns]

    # Merge with mentions_df predictions
    if 'gender_cascade' not in mentions_df.columns:
        print('Note: cascade predictions not found; comparing rule-based only')

    valid_gender = ['Female', 'Male', 'Unknown']
    valid_state  = ['Alive', 'Deceased', 'Supernatural', 'Unknown']

    print('=' * 70)
    print('VALIDATION REPORT — PRIMARY GOLD STANDARD: Human Annotation')
    print('=' * 70)

    # ─ GENDER ───────────────────────────────────────────────────────────
    g_df = completed[completed['human_gender'].isin(valid_gender)].copy()
    print(f'\n=== GENDER ({len(g_df)} labeled examples) ===')
    for col, label in [('rb_gender', 'Rule-Based'), ('cascade_gender', 'Cascade (Rule+LLM)')]:
        if col not in g_df.columns:
            print(f'  [{label}] column "{col}" not found in completed file, skipping.')
            continue
        yhat = g_df[col].fillna('Unknown')
        ytrue = g_df['human_gender']
        # F1 on Female vs Male (ignore Unknown)
        binary = g_df[g_df['human_gender'].isin(['Female','Male'])].copy()
        if len(binary) > 0:
            print(f'\n  [{label}]')
            print(classification_report(
                binary['human_gender'], binary[col].fillna('Unknown'),
                labels=['Female','Male'], zero_division=0, digits=3))
            unk_rate = (yhat == 'Unknown').mean()
            print(f'  Unknown rate (of labeled set): {unk_rate:.1%}')
            kappa = cohen_kappa_score(ytrue, yhat, labels=valid_gender)
            print(f'  Cohen’s κ (all 3 classes):     {kappa:.3f}')

    # ─ STATE ────────────────────────────────────────────────────────────
    s_df = completed[completed['human_state'].isin(valid_state)].copy()
    print(f'\n=== STATE ({len(s_df)} labeled examples) ===')
    for col, label in [('rb_state', 'Rule-Based'), ('cascade_state', 'Cascade (Rule+LLM)')]:
        if col not in s_df.columns:
            print(f'  [{label}] column "{col}" not found in completed file, skipping.')
            continue
        yhat_s = s_df[col].fillna('Unknown')
        ytrue_s = s_df['human_state']
        excl_unk = s_df[s_df['human_state'] != 'Unknown'].copy()
        if len(excl_unk) > 0:
            print(f'\n  [{label}]')
            print(classification_report(
                excl_unk['human_state'], excl_unk[col].fillna('Unknown'),
                labels=['Alive','Deceased','Supernatural'], zero_division=0, digits=3))
            kappa_s = cohen_kappa_score(ytrue_s, yhat_s, labels=valid_state)
            print(f'  Cohen’s κ (all 4 classes): {kappa_s:.3f}')

    print()
    print('These F1 / κ values are the PRIMARY validation results for your thesis.')
    s_df.to_csv(OUT_DIR / 'annotation_validation_results.csv', index=False)


In [ ]:
# ── Part 11A.3: McNemar's Test + Bootstrap 95% CI + Progressive Table ────────
from statsmodels.stats.contingency_tables import mcnemar
from sklearn.metrics import f1_score
import numpy as np

if not COMPLETED_PATH.exists():
    print('Run Part 11A.2 first (file not found).')
else:
    completed = pd.read_csv(COMPLETED_PATH)
    completed.columns = [c.strip() for c in completed.columns]

    print('=' * 72)
    print("PART 11A.3: McNemar's Test + Bootstrap 95% CI")
    print('=' * 72)

    results = {}
    for task, valid_labels, rb_col, cas_col, human_col in [
        ('Gender', ['Female','Male'], 'rb_gender', 'cascade_gender', 'human_gender'),
        ('State',  ['Alive','Deceased','Supernatural'], 'rb_state', 'cascade_state', 'human_state'),
    ]:
        df_t = completed[completed[human_col].isin(valid_labels)].copy()
        ytrue = df_t[human_col].values

        if rb_col not in df_t.columns or cas_col not in df_t.columns:
            print(f'\n  [{task}] columns not found — skipping.')
            continue

        yrb  = df_t[rb_col].fillna('Unknown').values
        ycs  = df_t[cas_col].fillna('Unknown').values

        # ── McNemar's test ───────────────────────────────────────────────
        rb_c  = (yrb == ytrue)
        cas_c = (ycs == ytrue)
        b = int(((~rb_c) & cas_c).sum())   # cascade fixed RB errors
        c = int((rb_c & (~cas_c)).sum())    # cascade broke RB correct
        try:
            mn   = mcnemar([[0, b], [c, 0]], exact=True)
            sig  = ('***' if mn.pvalue < 0.001 else
                    '**'  if mn.pvalue < 0.01  else
                    '*'   if mn.pvalue < 0.05  else 'n.s.')
            print(f"\n  [{task}] McNemar's test (Rule-Based vs Cascade):")
            print(f"    b={b} (cascade fixed RB errors), c={c} (cascade broke RB correct)")
            print(f"    p = {mn.pvalue:.4f} {sig}")
            mcn_p = mn.pvalue
        except Exception as e:
            print(f"  [{task}] McNemar failed: {e}")
            mcn_p = float('nan')
            sig   = 'n.a.'

        # ── Bootstrap 95% CI for macro-F1 ───────────────────────────────
        rng = np.random.default_rng(SEED)
        n   = len(ytrue)
        row = {'task': task, 'mcnemar_p': mcn_p, 'mcnemar_sig': sig}
        for ypred, method, key in [
            (yrb, 'Rule-Based', 'rb'),
            (ycs, 'Cascade',    'cas'),
        ]:
            boots = []
            for _ in range(1000):
                idx = rng.integers(0, n, size=n)
                try:
                    boots.append(f1_score(
                        ytrue[idx], ypred[idx],
                        labels=valid_labels, average='macro', zero_division=0))
                except Exception:
                    pass
            pt = f1_score(ytrue, ypred, labels=valid_labels, average='macro', zero_division=0)
            if boots:
                lo, hi = np.percentile(boots, [2.5, 97.5])
            else:
                lo, hi = float('nan'), float('nan')
            kap = cohen_kappa_score(ytrue, ypred)
            row[f'{key}_f1'] = pt
            row[f'{key}_lo'] = lo
            row[f'{key}_hi'] = hi
            row[f'{key}_k']  = kap
            print(f"  [{task}] {method:<20}  macro-F1={pt:.3f}  95% CI [{lo:.3f}, {hi:.3f}]  κ={kap:.3f}")
        results[task] = row

    # ── Progressive Improvement Table (thesis-ready) ─────────────────────
    print()
    print('=' * 72)
    print('PROGRESSIVE IMPROVEMENT TABLE (for thesis Table X)')
    print('=' * 72)
    header = (f"{'Task':<10} {'Rule-Based F1':>15} {'Cascade F1':>15} "
              f"{'Δ F1':>8} {'McNemar p':>12} {'Sig':>5}")
    print(header)
    print('-' * 72)
    for task, row in results.items():
        rb_str  = f"{row['rb_f1']:.3f} [{row['rb_lo']:.3f},{row['rb_hi']:.3f}]"
        cas_str = f"{row['cas_f1']:.3f} [{row['cas_lo']:.3f},{row['cas_hi']:.3f}]"
        delta   = row['cas_f1'] - row['rb_f1']
        print(f"{task:<10} {rb_str:>15} {cas_str:>15} "
              f"{delta:>+8.3f} {row['mcnemar_p']:>12.4f} {row['mcnemar_sig']:>5}")
    print()
    print("Citation: McNemar (1947); Dietterich (1998) for classifier comparison.")
    print("Citation: Efron & Tibshirani (1993) for Bootstrap CI (B=1000, seed=42).")

### Part 11B · LLM-as-Judge Validation (Claude Sonnet — Scale Supplement)

Uses a stronger model (Claude Sonnet) as an automatic annotator for 500 non-overlapping
examples. This supplements the 500 human labels at minimal extra cost ($1.50 USD).

**Role in thesis**: Zheng et al. (2023) established LLM-as-Judge as a standard
evaluation approach. Reporting Sonnet's agreement with your cascade classifier
strengthens the validity argument, especially where human annotation was limited
to 500 examples.

**Cost**: 500 calls × $0.003/call (Sonnet) = **$1.50 USD total**, ~5–8 minutes.

In [ ]:
# ── Part 11B.1: LLM-as-Judge setup (Claude Sonnet as gold annotator) ────────

JUDGE_N_PER_PLATFORM = 167  # 167 per platform ≈ 500 total
JUDGE_MODEL          = 'claude-sonnet-4-5'

JUDGE_GENDER_SYS = (
    "You are an expert multilingual linguist annotating ghost story texts for NLP research.\n"
    "Task: classify the GENDER of the character named in <entity> based on the provided sentence and local context.\n\n"

    "Important rules:\n"
    "1. Classify the gender of THIS mention only, not the whole story.\n"
    "2. Use only explicit textual evidence from the sentence and context.\n"
    "3. If gender is not clearly supported, return Unknown.\n"
    "4. Do NOT infer gender from stereotypes, occupations, emotions, or actions.\n"
    "5. Ghosts, spirits, and dead characters can still be Female or Male if the text explicitly indicates gender.\n"
    "6. If the mention refers to a group, ambiguous person, omitted subject, or unclear antecedent, return Unknown.\n\n"

    "Valid labels:\n"
    "- Female = clearly female from explicit textual evidence\n"
    "- Male = clearly male from explicit textual evidence\n"
    "- Unknown = not enough evidence to determine gender\n\n"

    "Return exactly one word: Female | Male | Unknown\n"
    "No explanation. No punctuation. One word only."
)

JUDGE_STATE_SYS = (
    "You are an expert multilingual linguist annotating ghost story texts for NLP research.\n"
    "Task: classify the STATE of the character named in <entity> based on the provided sentence and local context.\n\n"

    "Definitions:\n"
    "- Alive: a living human character.\n"
    "- Deceased: a person who has died but is NOT currently appearing as a ghost or interacting with the living.\n"
    "- Supernatural: a ghost, spirit, or dead entity that is actively appearing, interacting, or perceived as a supernatural presence.\n"
    "- Unknown: insufficient or ambiguous evidence.\n\n"

    "Important rules:\n"
    "1. Classify THIS mention only, based on the sentence and local context.\n"
    "2. If a dead person is described as appearing, interacting, or being seen/heard, classify as Supernatural (NOT Deceased).\n"
    "3. If a dead person is only mentioned as having died (no appearance), classify as Deceased.\n"
    "4. Do NOT infer beyond explicit textual evidence.\n"
    "5. If unclear, choose Unknown.\n"
    "6. Seeing, hearing, or interacting with a dead person implies Supernatural.\n\n"

    "Return exactly one word: Alive | Deceased | Supernatural | Unknown\n"
    "No explanation. No punctuation. One word only."
)

def judge_entity(entity, sentence, context, lang, task='gender'):
    sys_msg = JUDGE_GENDER_SYS if task == 'gender' else JUDGE_STATE_SYS
    valid   = ['Female','Male','Unknown'] if task == 'gender' else ['Supernatural','Alive','Deceased','Unknown']
    msg = f'Language: {lang}\n<entity>{entity}</entity>\n<sentence>{sentence}</sentence>\n<context>{context}</context>'
    try:
        resp = _client.messages.create(
            model=JUDGE_MODEL, max_tokens=5,
            system=sys_msg,
            messages=[{'role':'user','content':msg}]
        )
        r = resp.content[0].text.strip()
        return r if r in valid else valid[-1]
    except Exception as e:
        return valid[-1]

print(f'LLM-as-Judge model: {JUDGE_MODEL}')
print(f'Planned calls: {JUDGE_N_PER_PLATFORM * 3} gender + {JUDGE_N_PER_PLATFORM * 3} state = {JUDGE_N_PER_PLATFORM * 6} total')
est_cost = JUDGE_N_PER_PLATFORM * 6 * 0.003
print(f'Estimated cost: ${est_cost:.2f} USD ({est_cost * 33:.0f} NTD)')

In [ ]:
# ── Part 11B.2: Run LLM-as-Judge and compute agreement with cascade ───────
import time

if not CLAUDE_OK:
    print('Claude API not configured. Set CLAUDE_API_KEY in Part 3B.1 first.')
else:
    #sample balanced set (exclude examples already in human annotation if possible)
    judge_parts = []
    for plat in ['ptt', 'reddit', 'theqoo']:
        sub = mentions_df[mentions_df['platform'] == plat]
        n   = min(JUDGE_N_PER_PLATFORM, len(sub))
        if n > 0:
            judge_parts.append(sub.sample(n, random_state=7))
    judge_df = pd.concat(judge_parts, ignore_index=True).copy()
    print(f'LLM-as-Judge set: {len(judge_df):,} mentions')

    gold_g, gold_s = [], []
    for _, row in judge_df.iterrows():
        gold_g.append(judge_entity(row['entity_canon'], row['sentence'], row['context'], row['lang'], 'gender'))
        gold_s.append(judge_entity(row['entity_canon'], row['sentence'], row['context'], row['lang'], 'state'))
        time.sleep(0.3)  # ~200 req/min safe rate
    judge_df['gold_gender_sonnet'] = gold_g
    judge_df['gold_state_sonnet']  = gold_s

    judge_df.to_csv(OUT_DIR / 'llm_as_judge_results.csv', index=False)
    print(f'Saved LLM-as-Judge results to {OUT_DIR / "llm_as_judge_results.csv"}')


    from sklearn.metrics import classification_report, cohen_kappa_score
    print('\n' + '=' * 70)
    print(f'LLM-AS-JUDGE REPORT  —  Gold: {JUDGE_MODEL} (Supplement to human annotation)')
    print('=' * 70)



    # Gender
    gj = judge_df[judge_df['gold_gender_sonnet'].isin(['Female','Male'])]
    if 'gender_cascade' in gj.columns and len(gj) > 0:
        print('\n=== GENDER (Cascade vs Sonnet-as-Judge) ===')
        print(classification_report(gj['gold_gender_sonnet'], gj['gender_cascade'].fillna('Unknown'),
                                    labels=['Female','Male'], zero_division=0, digits=3))

    # State
    sj = judge_df[judge_df['gold_state_sonnet'].isin(['Alive','Deceased','Supernatural'])]
    if 'state_cascade' in sj.columns and len(sj) > 0:
        print('\n=== STATE (Cascade vs Sonnet-as-Judge) ===')
        print(classification_report(sj['gold_state_sonnet'], sj['state_cascade'].fillna('Unknown'),
                                    labels=['Alive','Deceased','Supernatural'], zero_division=0, digits=3))

    print()
    print('Note: These results SUPPLEMENT (not replace) the human annotation in Part 11A.')
    print('Citation: Zheng et al. (2023) Judging LLM-as-a-Judge with MT-Bench and Chatbot Arena.')


### Part 11C · Validation Summary for Thesis

After running both tiers, your thesis methodology chapter can report:

| Validation Tier | Gold Standard | N | Metric | Purpose |
|----------------|--------------|---|--------|---------|
| **Primary** | **Human (you)** | **500** | **P/R/F1, κ** | **Main validation** |
| Supplement | LLM-as-Judge (Sonnet) | 500 | F1, agreement | Scale validity |

**Recommended thesis sentence**: "Classifier validity was evaluated in two tiers:
(1) 500 human-annotated examples as the primary gold standard, yielding F1 scores
and 95% bootstrap CIs reported in Table X; (2) 500 LLM-as-Judge annotations
(Claude Sonnet; Zheng et al. 2023) as a supplementary validity check at scale.
Cascade improvement over rule-based was assessed with McNemar's test (Dietterich, 1998)."

In [ ]:
# ── Part 11 Summary: Print validation summary table ────────────────────
print('=' * 72)
print('VALIDATION SUMMARY')
print('=' * 72)
print()
print(f'{"Tier":<12} {"Gold Standard":<28} {"N":>5}  {"Metric":<22} Role')
print('-' * 72)
print(f'{"PRIMARY":<12} {"Human annotation (you)":<28} {500:>5}  {"P/R/F1 + Cohens k":<22} Main')
print(f'{"Supplement":<12} {"LLM-as-Judge (Sonnet)":<28} {500:>5}  {"F1 + agreement":<22} Scale')
print()
print('Order to run:')
print('  11A.1  Export annotation_task.xlsx (500 examples)')
print('  [YOU]  Fill in human_gender / human_state columns (2-3 hrs)')
print('  11A.2  Import completed file -> primary P/R/F1 results')
print('  11A.3  McNemar\'s test + Bootstrap 95% CI + Progressive Table')
print('  11B.1  (Optional) LLM-as-Judge setup')
print('  11B.2  (Optional) Run Sonnet annotations -> supplement F1')
print()
print('Recommended: complete 11A first. 11B is an optional supplement.')

In [ ]:
# Part 11A.x: append gender_method/state_method from mentions file into completed annotations
from pathlib import Path
import pandas as pd

completed_path = RAW_DIR / 'annotation_task_completed.csv'
mentions_path = OUT_DIR / 'mentions_final_checked_with_affect.csv'
output_path = OUT_DIR / 'annotation_task_completed_with_methods.csv'

completed = pd.read_csv(completed_path)
mentions = pd.read_csv(mentions_path)
completed.columns = [c.strip() for c in completed.columns]
mentions.columns = [c.strip() for c in mentions.columns]

# Shared keys between completed and mentions
key_cols = ['platform', 'lang', 'entity_canon', 'sentence', 'context']
method_cols = ['gender_method', 'state_method']

for c in key_cols:
    completed[c] = completed[c].astype(str).str.strip()
    mentions[c] = mentions[c].astype(str).str.strip()

for c in method_cols:
    mentions[c] = mentions[c].astype(str).str.strip().replace({'': pd.NA, 'nan': pd.NA})

# Diagnose key quality
print('completed rows:', len(completed))
print('mentions rows:', len(mentions))
print('completed duplicated keys:', int(completed.duplicated(key_cols).sum()))
print('mentions duplicated keys:', int(mentions.duplicated(key_cols).sum()))

# Aggregate methods by key and detect ambiguity
agg = mentions.groupby(key_cols, dropna=False).agg(
    gender_method_n=('gender_method', lambda s: s.dropna().nunique()),
    state_method_n=('state_method', lambda s: s.dropna().nunique()),
    gender_method=('gender_method', lambda s: s.dropna().iloc[0] if len(s.dropna()) else pd.NA),
    state_method=('state_method', lambda s: s.dropna().iloc[0] if len(s.dropna()) else pd.NA),
).reset_index()

amb_gender = int((agg['gender_method_n'] > 1).sum())
amb_state = int((agg['state_method_n'] > 1).sum())
print('ambiguous keys (gender_method >1 unique):', amb_gender)
print('ambiguous keys (state_method >1 unique):', amb_state)

# For ambiguous keys, set method as NA to avoid injecting wrong labels
agg.loc[agg['gender_method_n'] > 1, 'gender_method'] = pd.NA
agg.loc[agg['state_method_n'] > 1, 'state_method'] = pd.NA

# Merge back into completed
completed2 = completed.merge(
    agg[key_cols + method_cols],
    on=key_cols,
    how='left',
    suffixes=('', '_from_mentions')
)

print('\nmerge result:')
print('matched gender_method:', int(completed2['gender_method'].notna().sum()), '/', len(completed2))
print('matched state_method:', int(completed2['state_method'].notna().sum()), '/', len(completed2))

missing_methods = completed2[completed2['gender_method'].isna() | completed2['state_method'].isna()].copy()
if len(missing_methods):
    print('\nrows with missing method after merge:', len(missing_methods))
    print(missing_methods[key_cols + ['human_gender', 'human_state']].head(10).to_string(index=False))

completed2.to_csv(output_path, index=False, encoding='utf-8-sig')
print(f'\nSaved: {output_path}')

# keep in-memory object for downstream analysis cells
completed = completed2

In [ ]:
# Part 11A.y: Evaluate cascade performance by routed method (rule / LLM)
from sklearn.metrics import classification_report, accuracy_score, f1_score

MERGED_COMPLETED_PATH = OUT_DIR / 'annotation_task_completed_with_methods.csv'
if MERGED_COMPLETED_PATH.exists():
    completed_eval = pd.read_csv(MERGED_COMPLETED_PATH)
    print(f'Loaded merged file: {MERGED_COMPLETED_PATH}')
else:
    completed_eval = completed.copy()
    print('Merged file not found; falling back to in-memory completed DataFrame.')

def eval_by_method(df, method_col, true_col, pred_col, labels, title):
    print('\n' + '=' * 72)
    print(title)
    print('=' * 72)

    required = [method_col, true_col, pred_col]
    missing = [c for c in required if c not in df.columns]
    if missing:
        print(f'Missing columns: {missing}')
        return

    base = df[df[true_col].isin(labels)].copy()
    print(f'Total labeled rows in scope: {len(base)}')

    for method in ['rule', 'haiku', 'sonnet_fallback']:
        sub = base[base[method_col] == method].copy()
        n = len(sub)
        print(f'\n[{method}] n={n}')
        if n == 0:
            print('No samples.')
            continue

        y_true = sub[true_col]
        y_pred = sub[pred_col].fillna('Unknown')

        acc = accuracy_score(y_true, y_pred)
        mf1 = f1_score(y_true, y_pred, labels=labels, average='macro', zero_division=0)
        print(f'Accuracy={acc:.3f}  Macro-F1={mf1:.3f}')
        print(classification_report(
            y_true, y_pred, labels=labels, digits=3, zero_division=0
        ))

eval_by_method(
    completed_eval,
    method_col='gender_method',
    true_col='human_gender',
    pred_col='cascade_gender',
    labels=['Female', 'Male'],
    title='GENDER by routed method (evaluating cascade output)'
)

eval_by_method(
    completed_eval,
    method_col='state_method',
    true_col='human_state',
    pred_col='cascade_state',
    labels=['Alive', 'Deceased', 'Supernatural'],
    title='STATE by routed method (evaluating cascade output)'
)